### Verify schema

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
con = duckdb.connect()

In [2]:
import json

METADATA_DIR = Path("../data/metadata")

# Load the saved chronological split configuration
SPLIT_CONFIG_FILE = METADATA_DIR / "forecast_split_config.json"

with open(SPLIT_CONFIG_FILE, "r") as f:
    split_config = json.load(f)

print("FORECAST SPLIT CONFIGURATION")
print(json.dumps(split_config, indent=4))


# Inspect series classification metadata
SERIES_CLASS_FILE = METADATA_DIR / "series_classes.parquet"

series_class_schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{SERIES_CLASS_FILE.as_posix()}')
""").df()

print("\nSERIES CLASSIFICATION SCHEMA")
display(series_class_schema)


# Inspect class distribution
series_class_distribution = con.execute(f"""
    SELECT *
    FROM read_parquet('{SERIES_CLASS_FILE.as_posix()}')
    LIMIT 10
""").df()

print("\nSERIES CLASSIFICATION SAMPLE")
display(series_class_distribution)

FORECAST SPLIT CONFIGURATION
{
    "forecast_horizon": 28,
    "development_folds": [
        {
            "fold": 1,
            "train_end": 1801,
            "validation_start": 1802,
            "validation_end": 1829
        },
        {
            "fold": 2,
            "train_end": 1829,
            "validation_start": 1830,
            "validation_end": 1857
        },
        {
            "fold": 3,
            "train_end": 1857,
            "validation_start": 1858,
            "validation_end": 1885
        }
    ],
    "internal_test": {
        "train_end": 1885,
        "test_start": 1886,
        "test_end": 1913
    },
    "official_evaluation": {
        "train_end": 1913,
        "evaluation_start": 1914,
        "evaluation_end": 1941
    }
}

SERIES CLASSIFICATION SCHEMA


,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,item_id,VARCHAR,YES,None,None,None
2,dept_id,VARCHAR,YES,None,None,None
3,cat_id,VARCHAR,YES,None,None,None
4,store_id,VARCHAR,YES,None,None,None
5,state_id,VARCHAR,YES,None,None,None
6,first_demand_day,BIGINT,YES,None,None,None
7,active_period,BIGINT,YES,None,None,None
8,nonzero_days,BIGINT,YES,None,None,None
9,zero_days_active,BIGINT,YES,None,None,None



SERIES CLASSIFICATION SAMPLE


,id,item_id,dept_id,cat_id,store_id,state_id,first_demand_day,active_period,nonzero_days,zero_days_active,adi,cv2,demand_class
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,902,1040,436,604,2.385321,0.282900,intermittent
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,144,1793,408,1385,4.394608,0.233792,intermittent
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,1106,836,229,607,3.650655,0.287932,intermittent
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,37,1905,1316,589,1.447568,0.583019,lumpy
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,113,1828,983,845,1.859613,0.400368,intermittent
5,HOBBIES_1_006_CA_1_evaluation,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA,430,1511,665,846,2.272180,0.481962,intermittent
6,HOBBIES_1_007_CA_1_evaluation,HOBBIES_1_007,HOBBIES_1,HOBBIES,CA_1,CA,531,1410,362,1048,3.895028,0.140561,intermittent
7,HOBBIES_1_008_CA_1_evaluation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,1,1941,1403,538,1.383464,0.870049,lumpy
8,HOBBIES_1_009_CA_1_evaluation,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,1,1940,885,1055,2.192090,0.786965,lumpy
9,HOBBIES_1_010_CA_1_evaluation,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,3,1939,928,1011,2.089440,0.264748,intermittent


### Select the modelling sample

Selecting 2,000 series using proportional stratified sampling by demand_class, with a fixed random seed for reproducibility.

Approximate allocation:
| Demand class | Population | Sample |
| :--- | :--- | :--- |
| Intermittent | 22,150 | 1,453 |
| Lumpy | 5,564 | 365 |
| Smooth | 1,908 | 125 |
| Erratic | 868 | 57 |
| **Total** | **30,490** | **2,000** |


#### Recalculate Demand Classification Using the Initial Training Cutoff

The original demand classifications were calculated using the complete sales history (d_1–d_1941), including periods reserved for validation and testing.

To prevent future information from influencing the selection of modelling series, ADI, CV², and demand classes are recalculated using only historical sales up to d_1801.

This ensures that the 2,000-series modelling sample is selected based on information available at the initial forecast origin, reducing potential selection bias and maintaining a more realistic chronological evaluation design.

**Outcome:** A revised demand classification dataset that can be used for leakage-aware, proportional stratified sampling.

In [3]:
SERIES_CLASS_TRAIN_OUTPUT = (
    METADATA_DIR / "series_classes_train_cutoff.parquet"
)

INITIAL_TRAIN_CUTOFF = 1801

# Locate project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FINAL_FEATURE_OUTPUT = (
    PROCESSED_DIR / "daily_model_features.parquet"
)

# Remove the previous output if it already exists
if SERIES_CLASS_TRAIN_OUTPUT.exists():
    SERIES_CLASS_TRAIN_OUTPUT.unlink()

con.execute(f"""
    COPY (
        WITH demand_statistics AS (
            SELECT
                id,

                MIN(
                    TRY_CAST(SUBSTRING(d, 3) AS INTEGER)
                ) FILTER (
                    WHERE sales > 0
                ) AS first_demand_day,

                MAX(
                    TRY_CAST(SUBSTRING(d, 3) AS INTEGER)
                ) FILTER (
                    WHERE sales > 0
                ) AS last_demand_day,

                COUNT(*) FILTER (
                    WHERE sales > 0
                ) AS nonzero_days,

                AVG(sales) FILTER (
                    WHERE sales > 0
                ) AS mean_nonzero_demand,

                STDDEV_POP(sales) FILTER (
                    WHERE sales > 0
                ) AS stddev_nonzero_demand

            FROM read_parquet(
                '{FINAL_FEATURE_OUTPUT.as_posix()}'
            )

            WHERE TRY_CAST(SUBSTRING(d, 3) AS INTEGER)
                <= {INITIAL_TRAIN_CUTOFF}

            GROUP BY id
        ),

        calculated_metrics AS (
            SELECT
                id,
                first_demand_day,
                last_demand_day,

                last_demand_day
                    - first_demand_day + 1 AS active_period,

                nonzero_days,

                last_demand_day
                    - first_demand_day + 1
                    - nonzero_days AS zero_days_active,

                (last_demand_day
                    - first_demand_day + 1)
                    * 1.0 / NULLIF(nonzero_days, 0) AS adi,

                POWER(
                    stddev_nonzero_demand
                    / NULLIF(mean_nonzero_demand, 0),
                    2
                ) AS cv2

            FROM demand_statistics
        ),

        classified_series AS (
            SELECT
                *,

                CASE
                    WHEN nonzero_days = 0
                        THEN 'no_demand'

                    WHEN adi < 1.32 AND cv2 < 0.49
                        THEN 'smooth'

                    WHEN adi < 1.32 AND cv2 >= 0.49
                        THEN 'erratic'

                    WHEN adi >= 1.32 AND cv2 < 0.49
                        THEN 'intermittent'

                    WHEN adi >= 1.32 AND cv2 >= 0.49
                        THEN 'lumpy'
                END AS demand_class

            FROM calculated_metrics
        )

        SELECT
            original.id,
            original.item_id,
            original.dept_id,
            original.cat_id,
            original.store_id,
            original.state_id,

            calculated.first_demand_day,
            calculated.active_period,
            calculated.nonzero_days,
            calculated.zero_days_active,
            calculated.adi,
            calculated.cv2,
            calculated.demand_class

        FROM read_parquet(
            '{SERIES_CLASS_FILE.as_posix()}'
        ) original

        LEFT JOIN classified_series calculated
            ON original.id = calculated.id

    )
    TO '{SERIES_CLASS_TRAIN_OUTPUT.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(
    "Training-cutoff demand classifications created:",
    SERIES_CLASS_TRAIN_OUTPUT
)


Training-cutoff demand classifications created: ..\data\metadata\series_classes_train_cutoff.parquet


In [4]:
# Validate the recalculated demand-class distribution

train_cutoff_distribution = con.execute(f"""
    SELECT
        demand_class,
        COUNT(*) AS total_series,
        ROUND(
            COUNT(*) * 100.0
            / SUM(COUNT(*)) OVER (),
            2
        ) AS percentage
    FROM read_parquet(
        '{SERIES_CLASS_TRAIN_OUTPUT.as_posix()}'
    )
    GROUP BY demand_class
    ORDER BY demand_class
""").df()

display(train_cutoff_distribution)

,demand_class,total_series,percentage
0,erratic,895,2.94
1,intermittent,22118,72.54
2,lumpy,5494,18.02
3,no_demand,16,0.05
4,smooth,1967,6.45


In [5]:
# Validate total series and missing classifications

train_cutoff_validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_series,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(*) FILTER (
            WHERE demand_class IS NULL
        ) AS missing_demand_class
    FROM read_parquet(
        '{SERIES_CLASS_TRAIN_OUTPUT.as_posix()}'
    )
""").df()

display(train_cutoff_validation)

,total_series,unique_series,missing_demand_class
0,30490,30490,0


#### Generate the 2,000-series sample

In [6]:
SERIES_SAMPLE_OUTPUT = (
    METADATA_DIR / "selected_2000_series.parquet"
)

# Define the sample allocation based on the d_1801 classifications
sample_allocation = [
    ("intermittent", 1440),
    ("lumpy", 358),
    ("smooth", 128),
    ("erratic", 58),
    ("no_demand", 16)
]

allocation_values = ", ".join(
    f"('{demand_class}', {sample_size})"
    for demand_class, sample_size in sample_allocation
)

# Remove the previous sample if it exists
if SERIES_SAMPLE_OUTPUT.exists():
    SERIES_SAMPLE_OUTPUT.unlink()

con.execute(f"""
    COPY (
        WITH ranked_series AS (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY demand_class
                    ORDER BY md5(id || '_2026')
                ) AS sample_rank

            FROM read_parquet(
                '{SERIES_CLASS_TRAIN_OUTPUT.as_posix()}'
            )
        ),

        allocation AS (
            SELECT *
            FROM (VALUES {allocation_values})
            AS t(demand_class, sample_size)
        )

        SELECT
            r.* EXCLUDE (sample_rank)

        FROM ranked_series r

        JOIN allocation a
            ON r.demand_class = a.demand_class

        WHERE r.sample_rank <= a.sample_size

    )
    TO '{SERIES_SAMPLE_OUTPUT.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Updated modelling sample created:", SERIES_SAMPLE_OUTPUT)


Updated modelling sample created: ..\data\metadata\selected_2000_series.parquet


In [7]:

# Check the selected series per demand class

sample_validation = con.execute(f"""
    SELECT
        demand_class,
        COUNT(*) AS selected_series
    FROM read_parquet(
        '{SERIES_SAMPLE_OUTPUT.as_posix()}'
    )
    GROUP BY demand_class
    ORDER BY demand_class
""").df()

display(sample_validation)


,demand_class,selected_series
0,erratic,58
1,intermittent,1440
2,lumpy,358
3,no_demand,16
4,smooth,128


In [8]:

# Check total rows and unique series

sample_total = con.execute(f"""
    SELECT
        COUNT(*) AS total_series,
        COUNT(DISTINCT id) AS unique_series
    FROM read_parquet(
        '{SERIES_SAMPLE_OUTPUT.as_posix()}'
    )
""").df()

display(sample_total)


,total_series,unique_series
0,2000,2000


#### Generate the 200-series TFT subset

In [9]:
TFT_SAMPLE_OUTPUT = (
    METADATA_DIR / "selected_200_series_tft.parquet"
)

tft_allocation = [
    ("intermittent", 144),
    ("lumpy", 36),
    ("smooth", 13),
    ("erratic", 6),
    ("no_demand", 1)
]

allocation_values = ", ".join(
    f"('{demand_class}', {sample_size})"
    for demand_class, sample_size in tft_allocation
)

# Remove previous TFT sample if it exists
if TFT_SAMPLE_OUTPUT.exists():
    TFT_SAMPLE_OUTPUT.unlink()

con.execute(f"""
    COPY (
        WITH ranked_series AS (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY demand_class
                    ORDER BY md5(id || '_2027')
                ) AS tft_rank

            FROM read_parquet(
                '{SERIES_SAMPLE_OUTPUT.as_posix()}'
            )
        ),

        allocation AS (
            SELECT *
            FROM (VALUES {allocation_values})
            AS t(demand_class, sample_size)
        )

        SELECT
            r.* EXCLUDE (tft_rank)

        FROM ranked_series r

        JOIN allocation a
            ON r.demand_class = a.demand_class

        WHERE r.tft_rank <= a.sample_size

    )
    TO '{TFT_SAMPLE_OUTPUT.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Updated TFT sample created:", TFT_SAMPLE_OUTPUT)


Updated TFT sample created: ..\data\metadata\selected_200_series_tft.parquet


In [10]:
tft_validation = con.execute(f"""
    SELECT
        demand_class,
        COUNT(*) AS selected_series
    FROM read_parquet(
        '{TFT_SAMPLE_OUTPUT.as_posix()}'
    )
    GROUP BY demand_class
    ORDER BY demand_class
""").df()

display(tft_validation)

tft_total = con.execute(f"""
    SELECT
        COUNT(*) AS total_series,
        COUNT(DISTINCT id) AS unique_series
    FROM read_parquet(
        '{TFT_SAMPLE_OUTPUT.as_posix()}'
    )
""").df()

display(tft_total)

,demand_class,selected_series
0,erratic,6
1,intermittent,144
2,lumpy,36
3,no_demand,1
4,smooth,13


,total_series,unique_series
0,200,200


### Feature Selection: Generate supervised training examples


**Objective:** Verify that historical features, future calendar features, and target sales are correctly aligned for multi-horizon forecasting.

**Approach:**
- Selected one product-store series and used `d_56` as the forecast origin.
- Generated 28 target observations covering `d_57` to `d_84`.
- Kept historical sales and price features fixed at the forecast origin.
- Retrieved calendar features and actual sales from each respective target date.
- Calculated the forecast horizon as the difference between target day and origin day.

**Rationale:**

This approach simulates a realistic forecasting scenario where predictions for the next 28 days are made using only information available at the forecast origin. Future calendar information is retained because it is known in advance, while actual future sales are used exclusively as prediction targets.


In [11]:
# Step 3.2.3 — Inspect source schemas

SOURCE_FILES = {
    "daily_model_features": FINAL_FEATURE_OUTPUT,                  # Final model dataset
    "selected_2000_series": SERIES_SAMPLE_OUTPUT,                  # 2,000-series sample
    "forecast_origins": METADATA_DIR / "forecast_origins.parquet"  # Forecast-origin metadata
}

for name, file_path in SOURCE_FILES.items():
    print(f"\n{'=' * 60}")
    print(f"{name}: {file_path}")

    schema = con.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_parquet('{file_path.as_posix()}')
    """).df()

    display(schema)


daily_model_features: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\daily_model_features.parquet


,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,item_id,VARCHAR,YES,None,None,None
2,dept_id,VARCHAR,YES,None,None,None
3,cat_id,VARCHAR,YES,None,None,None
4,store_id,VARCHAR,YES,None,None,None
5,state_id,VARCHAR,YES,None,None,None
6,d,VARCHAR,YES,None,None,None
7,sales,BIGINT,YES,None,None,None
8,date,DATE,YES,None,None,None
9,wm_yr_wk,BIGINT,YES,None,None,None



selected_2000_series: ..\data\metadata\selected_2000_series.parquet


,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,item_id,VARCHAR,YES,None,None,None
2,dept_id,VARCHAR,YES,None,None,None
3,cat_id,VARCHAR,YES,None,None,None
4,store_id,VARCHAR,YES,None,None,None
5,state_id,VARCHAR,YES,None,None,None
6,first_demand_day,INTEGER,YES,None,None,None
7,active_period,INTEGER,YES,None,None,None
8,nonzero_days,BIGINT,YES,None,None,None
9,zero_days_active,BIGINT,YES,None,None,None



forecast_origins: ..\data\metadata\forecast_origins.parquet


,column_name,column_type,null,key,default,extra
0,origin_day_num,BIGINT,YES,None,None,None
1,origin_day,VARCHAR,YES,None,None,None
2,origin_type,VARCHAR,YES,None,None,None


In [12]:
# Inspect sample records from the final feature dataset

feature_sample = con.execute(f"""
    SELECT *
    FROM read_parquet(
        '{FINAL_FEATURE_OUTPUT.as_posix()}'
    )
    WHERE id IN (
        SELECT id
        FROM read_parquet(
            '{SERIES_SAMPLE_OUTPUT.as_posix()}'
        )
        LIMIT 3
    )
    ORDER BY id, d
    LIMIT 15
""").df()

display(feature_sample)

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,sales_at_origin,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_std_7,sales_rolling_mean_28,sales_rolling_std_28,sales_rolling_max_28
0,FOODS_3_296_TX_2_evaluation,FOODS_3_296,FOODS_3,FOODS,TX_2,TX,d_1,0,2011-01-29,11101,...,0,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0,0.0,0
1,FOODS_3_296_TX_2_evaluation,FOODS_3_296,FOODS_3,FOODS,TX_2,TX,d_10,0,2011-02-07,11102,...,0,0,0,<NA>,<NA>,0.0,0.0,0.0,0.0,0
2,FOODS_3_296_TX_2_evaluation,FOODS_3_296,FOODS_3,FOODS,TX_2,TX,d_100,0,2011-05-08,11115,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0
3,FOODS_3_296_TX_2_evaluation,FOODS_3_296,FOODS_3,FOODS,TX_2,TX,d_1000,0,2013-10-24,11339,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0
4,FOODS_3_296_TX_2_evaluation,FOODS_3_296,FOODS_3,FOODS,TX_2,TX,d_1001,0,2013-10-25,11339,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0
5,FOODS_3_296_TX_2_evaluation,FOODS_3_296,FOODS_3,FOODS,TX_2,TX,d_1002,0,2013-10-26,11340,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0
6,FOODS_3_296_TX_2_evaluation,FOODS_3_296,FOODS_3,FOODS,TX_2,TX,d_1003,0,2013-10-27,11340,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0
7,FOODS_3_296_TX_2_evaluation,FOODS_3_296,FOODS_3,FOODS,TX_2,TX,d_1004,0,2013-10-28,11340,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0
8,FOODS_3_296_TX_2_evaluation,FOODS_3_296,FOODS_3,FOODS,TX_2,TX,d_1005,0,2013-10-29,11340,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0
9,FOODS_3_296_TX_2_evaluation,FOODS_3_296,FOODS_3,FOODS,TX_2,TX,d_1006,0,2013-10-30,11340,...,0,0,0,0,0,0.0,0.0,0.0,0.0,0


#### Define the predictor and target features

| Feature group | Columns | Source |
| :--- | :--- | :--- |
| **Historical sales** | `sales_at_origin`, `sales_lag_1`, `sales_lag_7`, `sales_lag_14`, `sales_lag_28` | Forecast origin |
| **Rolling statistics** | `sales_rolling_mean_7`, `sales_rolling_std_7`, `sales_rolling_mean_28`, `sales_rolling_std_28`, `sales_rolling_max_28` | Forecast origin |
| **Product/store metadata** | `item_id`, `dept_id`, `cat_id`, `store_id`, `state_id` | Static metadata |
| **Calendar** | `weekday`, `wday`, `month`, `year`, `is_weekend`, `day_of_month`, `week_of_year` | Future target date |
| **Events and SNAP** | `event_name_1`, `event_type_1`, `event_name_2`, `event_type_2`, `is_event_day`, `is_snap_day` | Future target date |
| **Forecast horizon** | `horizon` (1–28) | Generated during preparation |
| **Target** | `sales` | Future target date |



In [13]:

# Test forecast-origin and target-date feature alignment

TEST_ORIGIN = 56
TEST_HORIZON = 28

alignment_test = con.execute(f"""
    WITH selected_series AS (
        SELECT id
        FROM read_parquet(
            '{SERIES_SAMPLE_OUTPUT.as_posix()}'
        )
        ORDER BY id
        LIMIT 1
    ),

    origin_features AS (
        SELECT
            f.*
        FROM read_parquet(
            '{FINAL_FEATURE_OUTPUT.as_posix()}'
        ) f

        INNER JOIN selected_series s
            ON f.id = s.id

        WHERE TRY_CAST(SUBSTRING(f.d, 3) AS INTEGER)
            = {TEST_ORIGIN}
    ),

    target_rows AS (
        SELECT
            f.*
        FROM read_parquet(
            '{FINAL_FEATURE_OUTPUT.as_posix()}'
        ) f

        INNER JOIN selected_series s
            ON f.id = s.id

        WHERE TRY_CAST(SUBSTRING(f.d, 3) AS INTEGER)
            BETWEEN {TEST_ORIGIN + 1}
            AND {TEST_ORIGIN + TEST_HORIZON}
    )

    SELECT
        o.id,

        {TEST_ORIGIN} AS origin_day_num,
        o.d AS origin_day,

        TRY_CAST(SUBSTRING(t.d, 3) AS INTEGER)
            AS target_day_num,

        t.d AS target_day,

        TRY_CAST(SUBSTRING(t.d, 3) AS INTEGER)
            - {TEST_ORIGIN} AS horizon,

        -- Historical sales features from forecast origin
        o.sales_at_origin,
        o.sales_lag_1,
        o.sales_lag_7,
        o.sales_lag_14,
        o.sales_lag_28,
        o.sales_rolling_mean_7,
        o.sales_rolling_std_7,
        o.sales_rolling_mean_28,
        o.sales_rolling_std_28,
        o.sales_rolling_max_28,

        -- Price features from forecast origin
        o.sell_price AS origin_sell_price,
        o.price_missing AS origin_price_missing,
        o.price_change AS origin_price_change,
        o.price_change_pct AS origin_price_change_pct,
        o.price_vs_8w_avg AS origin_price_vs_8w_avg,
        o.price_change_flag AS origin_price_change_flag,

        -- Future calendar features from target date
        t.date AS target_date,
        t.weekday AS target_weekday,
        t.wday AS target_wday,
        t.month AS target_month,
        t.year AS target_year,
        t.is_weekend AS target_is_weekend,
        t.day_of_month AS target_day_of_month,
        t.week_of_year AS target_week_of_year,
        t.event_name_1 AS target_event_name_1,
        t.event_type_1 AS target_event_type_1,
        t.event_name_2 AS target_event_name_2,
        t.event_type_2 AS target_event_type_2,
        t.is_event_day AS target_is_event_day,
        t.is_snap_day AS target_is_snap_day,

        -- Actual future sales: prediction target only
        t.sales AS target_sales

    FROM origin_features o

    CROSS JOIN target_rows t

    WHERE o.id = t.id

    ORDER BY target_day_num
""").df()

display(alignment_test)


,id,origin_day_num,origin_day,target_day_num,target_day,horizon,sales_at_origin,sales_lag_1,sales_lag_7,sales_lag_14,...,target_is_weekend,target_day_of_month,target_week_of_year,target_event_name_1,target_event_type_1,target_event_name_2,target_event_type_2,target_is_event_day,target_is_snap_day,target_sales
0,FOODS_1_002_TX_1_evaluation,56,d_56,57,d_57,1,0,0,0,0,...,1,26,12,None,None,None,None,0,0,1
1,FOODS_1_002_TX_1_evaluation,56,d_56,58,d_58,2,0,0,0,0,...,1,27,12,None,None,None,None,0,0,0
2,FOODS_1_002_TX_1_evaluation,56,d_56,59,d_59,3,0,0,0,0,...,0,28,13,None,None,None,None,0,0,0
3,FOODS_1_002_TX_1_evaluation,56,d_56,60,d_60,4,0,0,0,0,...,0,29,13,None,None,None,None,0,0,0
4,FOODS_1_002_TX_1_evaluation,56,d_56,61,d_61,5,0,0,0,0,...,0,30,13,None,None,None,None,0,0,0
5,FOODS_1_002_TX_1_evaluation,56,d_56,62,d_62,6,0,0,0,0,...,0,31,13,None,None,None,None,0,0,0
6,FOODS_1_002_TX_1_evaluation,56,d_56,63,d_63,7,0,0,0,0,...,0,1,13,None,None,None,None,0,1,0
7,FOODS_1_002_TX_1_evaluation,56,d_56,64,d_64,8,0,0,0,0,...,1,2,13,None,None,None,None,0,0,0
8,FOODS_1_002_TX_1_evaluation,56,d_56,65,d_65,9,0,0,0,0,...,1,3,13,None,None,None,None,0,1,3
9,FOODS_1_002_TX_1_evaluation,56,d_56,66,d_66,10,0,0,0,0,...,0,4,14,None,None,None,None,0,0,0


In [14]:
# Validate forecast-origin feature alignment

alignment_validation = {
    "total_rows": len(alignment_test),

    "unique_target_days": alignment_test["target_day"].nunique(),

    "minimum_horizon": alignment_test["horizon"].min(),

    "maximum_horizon": alignment_test["horizon"].max(),

    "unique_origin_days": alignment_test["origin_day"].nunique(),

    "unique_sales_at_origin": alignment_test["sales_at_origin"].nunique(),

    "unique_sales_lag_1": alignment_test["sales_lag_1"].nunique(),

    "unique_sales_lag_7": alignment_test["sales_lag_7"].nunique(),

    "unique_rolling_mean_7": alignment_test["sales_rolling_mean_7"].nunique(),

    "unique_origin_sell_price": alignment_test["origin_sell_price"].nunique(),

    "unique_target_sales": alignment_test["target_sales"].nunique()
}

display(
    pd.DataFrame(
        alignment_validation.items(),
        columns=["validation_check", "result"]
    )
)

,validation_check,result
0,total_rows,28
1,unique_target_days,28
2,minimum_horizon,1
3,maximum_horizon,28
4,unique_origin_days,1
5,unique_sales_at_origin,1
6,unique_sales_lag_1,1
7,unique_sales_lag_7,1
8,unique_rolling_mean_7,1
9,unique_origin_sell_price,1


#### Generate Supervised Training Examples

The process is:
1. Select the 2,000 product-store series.
2. Retrieve their features at each of the 271 forecast origins.
3. Generate forecast horizons from 1 to 28 for each origin.
4. Retrieve the corresponding target-date calendar features and actual sales.
5. Combine everything into one supervised dataset.
6. Save the result as a Parquet file.

In [15]:
# --------------------------------------------------
# 1. Define input and output paths
# --------------------------------------------------

DAILY_FILE = FINAL_FEATURE_OUTPUT

SERIES_FILE = SERIES_SAMPLE_OUTPUT

ORIGINS_FILE = METADATA_DIR / "forecast_origins.parquet"

SUPERVISED_OUTPUT = PROCESSED_DIR / "supervised_training_examples.parquet"


# --------------------------------------------------
# 2. Generate supervised training examples
# --------------------------------------------------

query = f"""
COPY (

    WITH selected_series AS MATERIALIZED (
        SELECT
            id,
            item_id,
            dept_id,
            cat_id,
            store_id,
            state_id,
            demand_class
        FROM read_parquet('{SERIES_FILE.as_posix()}')
    ),

    forecast_origins AS MATERIALIZED (
        SELECT
            origin_day_num,
            origin_day,
            origin_type
        FROM read_parquet('{ORIGINS_FILE.as_posix()}')
    ),

    -- Features available at each forecast origin
    origin_features AS MATERIALIZED (
        SELECT
            f.id,
            TRY_CAST(SUBSTRING(f.d, 3) AS INTEGER) AS origin_day_num,

            f.sales_at_origin,
            f.sales_lag_1,
            f.sales_lag_7,
            f.sales_lag_14,
            f.sales_lag_28,

            f.sales_rolling_mean_7,
            f.sales_rolling_std_7,
            f.sales_rolling_mean_28,
            f.sales_rolling_std_28,
            f.sales_rolling_max_28,

            f.sell_price AS origin_sell_price,
            f.price_missing AS origin_price_missing,
            f.price_change AS origin_price_change,
            f.price_change_pct AS origin_price_change_pct,
            f.price_vs_8w_avg AS origin_price_vs_8w_avg,
            f.price_change_flag AS origin_price_change_flag

        FROM read_parquet('{DAILY_FILE.as_posix()}') f

        INNER JOIN selected_series s
            ON f.id = s.id

        INNER JOIN forecast_origins o
            ON TRY_CAST(SUBSTRING(f.d, 3) AS INTEGER)
             = o.origin_day_num
    ),

    -- Calendar information and actual sales on target dates
    target_features AS MATERIALIZED (
        SELECT
            f.id,
            TRY_CAST(SUBSTRING(f.d, 3) AS INTEGER) AS target_day_num,
            f.d AS target_day,

            f.date AS target_date,
            f.weekday AS target_weekday,
            f.wday AS target_wday,
            f.month AS target_month,
            f.year AS target_year,

            f.is_weekend AS target_is_weekend,
            f.day_of_month AS target_day_of_month,
            f.week_of_year AS target_week_of_year,

            f.event_name_1 AS target_event_name_1,
            f.event_type_1 AS target_event_type_1,
            f.event_name_2 AS target_event_name_2,
            f.event_type_2 AS target_event_type_2,

            f.is_event_day AS target_is_event_day,
            f.is_snap_day AS target_is_snap_day,

            f.sales AS target_sales

        FROM read_parquet('{DAILY_FILE.as_posix()}') f

        INNER JOIN selected_series s
            ON f.id = s.id
    ),

    -- Generate one row for every origin and forecast horizon
    horizon_grid AS (
        SELECT
            o.origin_day_num,
            o.origin_day,
            o.origin_type,
            h.horizon

        FROM forecast_origins o

        CROSS JOIN range(1, 29) AS h(horizon)
    )

    -- Combine origin features, target features and metadata
    SELECT
        s.id,
        s.item_id,
        s.dept_id,
        s.cat_id,
        s.store_id,
        s.state_id,
        s.demand_class,

        g.origin_day_num,
        g.origin_day,
        g.origin_type,

        t.target_day_num,
        t.target_day,
        g.horizon,

        o.sales_at_origin,
        o.sales_lag_1,
        o.sales_lag_7,
        o.sales_lag_14,
        o.sales_lag_28,

        o.sales_rolling_mean_7,
        o.sales_rolling_std_7,
        o.sales_rolling_mean_28,
        o.sales_rolling_std_28,
        o.sales_rolling_max_28,

        o.origin_sell_price,
        o.origin_price_missing,
        o.origin_price_change,
        o.origin_price_change_pct,
        o.origin_price_vs_8w_avg,
        o.origin_price_change_flag,

        t.target_date,
        t.target_weekday,
        t.target_wday,
        t.target_month,
        t.target_year,
        t.target_is_weekend,
        t.target_day_of_month,
        t.target_week_of_year,

        t.target_event_name_1,
        t.target_event_type_1,
        t.target_event_name_2,
        t.target_event_type_2,

        t.target_is_event_day,
        t.target_is_snap_day,

        t.target_sales

    FROM horizon_grid g

    INNER JOIN origin_features o
        ON o.origin_day_num = g.origin_day_num

    INNER JOIN selected_series s
        ON s.id = o.id

    INNER JOIN target_features t
        ON t.id = o.id
        AND t.target_day_num = g.origin_day_num + g.horizon

) TO '{SUPERVISED_OUTPUT.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(query)

print("Supervised training dataset generated successfully.")
print(f"Output: {SUPERVISED_OUTPUT}")

Supervised training dataset generated successfully.
Output: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\supervised_training_examples.parquet


In [16]:
# Validate Supervised Training Examples

SUPERVISED_FILE = SUPERVISED_OUTPUT

validation = con.execute(f"""
WITH dataset AS (
    SELECT *
    FROM read_parquet('{SUPERVISED_FILE.as_posix()}')
)

SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT id) AS unique_series,

    COUNT(DISTINCT origin_day_num) AS unique_origins,

    COUNT(DISTINCT horizon) AS unique_horizons,

    MIN(origin_day_num) AS minimum_origin_day,
    MAX(origin_day_num) AS maximum_origin_day,

    MIN(target_day_num) AS minimum_target_day,
    MAX(target_day_num) AS maximum_target_day,

    MIN(horizon) AS minimum_horizon,
    MAX(horizon) AS maximum_horizon,

    COUNT(*) FILTER (
        WHERE target_day_num != origin_day_num + horizon
    ) AS invalid_target_alignment,

    COUNT(*) FILTER (
        WHERE target_sales IS NULL
    ) AS missing_target_sales,

    COUNT(DISTINCT origin_type) AS unique_origin_types

FROM dataset
""").df()

display(validation)

,total_rows,unique_series,unique_origins,unique_horizons,minimum_origin_day,maximum_origin_day,minimum_target_day,maximum_target_day,minimum_horizon,maximum_horizon,invalid_target_alignment,missing_target_sales,unique_origin_types
0,15176000,2000,271,28,56,1913,57,1941,1,28,0,0,6


In [17]:
# Check duplicate training examples

duplicates = con.execute(f"""
SELECT
    COUNT(*) AS duplicate_groups

FROM (
    SELECT
        id,
        origin_day_num,
        horizon,
        COUNT(*) AS row_count

    FROM read_parquet('{SUPERVISED_FILE.as_posix()}')

    GROUP BY
        id,
        origin_day_num,
        horizon

    HAVING COUNT(*) > 1
)
""").df()

display(duplicates)

,duplicate_groups
0,0


In [18]:
# Check rows generated per forecast origin

origin_distribution = con.execute(f"""
SELECT
    origin_day_num,
    origin_day,
    origin_type,

    COUNT(*) AS total_rows,

    COUNT(DISTINCT id) AS unique_series,

    MIN(horizon) AS minimum_horizon,

    MAX(horizon) AS maximum_horizon

FROM read_parquet('{SUPERVISED_FILE.as_posix()}')

GROUP BY
    origin_day_num,
    origin_day,
    origin_type

ORDER BY origin_day_num
""").df()

display(origin_distribution)

,origin_day_num,origin_day,origin_type,total_rows,unique_series,minimum_horizon,maximum_horizon
0,56,d_56,weekly_training,56000,2000,1,28
1,63,d_63,weekly_training,56000,2000,1,28
2,70,d_70,weekly_training,56000,2000,1,28
3,77,d_77,weekly_training,56000,2000,1,28
4,84,d_84,weekly_training,56000,2000,1,28
...,...,...,...,...,...,...,...
266,1890,d_1890,weekly_training,56000,2000,1,28
267,1897,d_1897,weekly_training,56000,2000,1,28
268,1904,d_1904,weekly_training,56000,2000,1,28
269,1911,d_1911,weekly_training,56000,2000,1,28


### Chronological Training and Evaluation Preparation

In [19]:
# Inspect forecast origin types

origin_type_summary = con.execute(f"""
SELECT
    origin_type,
    COUNT(*) AS number_of_origins,
    MIN(origin_day_num) AS first_origin,
    MAX(origin_day_num) AS last_origin,
    LIST(origin_day ORDER BY origin_day_num) AS origin_days

FROM read_parquet('{ORIGINS_FILE.as_posix()}')

GROUP BY origin_type

ORDER BY first_origin
""").df()

display(origin_type_summary)

,origin_type,number_of_origins,first_origin,last_origin,origin_days
0,weekly_training,266,56,1911,"[d_56, d_63, d_70, d_77, d_84, d_91, d_98, d_1..."
1,development_validation_fold_1,1,1801,1801,[d_1801]
2,development_validation_fold_2,1,1829,1829,[d_1829]
3,development_validation_fold_3,1,1857,1857,[d_1857]
4,internal_test,1,1885,1885,[d_1885]
5,official_evaluation,1,1913,1913,[d_1913]


### Audit chronological training and evaluation boundaries

In [20]:

fold_config = [
    ("development_fold_1", 1801, 1801, 1802, 1829),
    ("development_fold_2", 1829, 1829, 1830, 1857),
    ("development_fold_3", 1857, 1857, 1858, 1885),
    ("internal_test",      1885, 1885, 1886, 1913),
    ("official_evaluation",1913, 1913, 1914, 1941)
]

fold_values = ", ".join(
    f"('{name}', {train_end}, {origin}, {start}, {end})"
    for name, train_end, origin, start, end in fold_config
)

fold_audit = con.execute(f"""
WITH folds AS (
    SELECT *
    FROM (
        VALUES {fold_values}
    ) AS v(
        fold_name,
        train_end,
        evaluation_origin,
        evaluation_start,
        evaluation_end
    )
),

dataset AS (
    SELECT *
    FROM read_parquet('{SUPERVISED_FILE.as_posix()}')
)

SELECT
    f.fold_name,

    f.train_end,

    COUNT(*) FILTER (
        WHERE d.target_day_num <= f.train_end
    ) AS training_rows,

    COUNT(DISTINCT d.id) FILTER (
        WHERE d.target_day_num <= f.train_end
    ) AS training_series,

    MAX(d.target_day_num) FILTER (
        WHERE d.target_day_num <= f.train_end
    ) AS training_max_target,

    COUNT(*) FILTER (
        WHERE d.origin_day_num = f.evaluation_origin
          AND d.target_day_num BETWEEN
              f.evaluation_start AND f.evaluation_end
    ) AS evaluation_rows,

    COUNT(DISTINCT d.id) FILTER (
        WHERE d.origin_day_num = f.evaluation_origin
          AND d.target_day_num BETWEEN
              f.evaluation_start AND f.evaluation_end
    ) AS evaluation_series,

    MIN(d.target_day_num) FILTER (
        WHERE d.origin_day_num = f.evaluation_origin
          AND d.target_day_num BETWEEN
              f.evaluation_start AND f.evaluation_end
    ) AS evaluation_min_target,

    MAX(d.target_day_num) FILTER (
        WHERE d.origin_day_num = f.evaluation_origin
          AND d.target_day_num BETWEEN
              f.evaluation_start AND f.evaluation_end
    ) AS evaluation_max_target

FROM folds f

CROSS JOIN dataset d

GROUP BY
    f.fold_name,
    f.train_end,
    f.evaluation_origin,
    f.evaluation_start,
    f.evaluation_end

ORDER BY f.train_end
""").df()

display(fold_audit)

,fold_name,train_end,training_rows,training_series,training_max_target,evaluation_rows,evaluation_series,evaluation_min_target,evaluation_max_target
0,development_fold_1,1801,13876000,2000,1801,56000,2000,1802,1829
1,development_fold_2,1829,14156000,2000,1829,56000,2000,1830,1857
2,development_fold_3,1857,14436000,2000,1857,56000,2000,1858,1885
3,internal_test,1885,14716000,2000,1885,56000,2000,1886,1913
4,official_evaluation,1913,14996000,2000,1913,56000,2000,1914,1941


#### Model Feature Audit

In [21]:
# Step 3.4.1 — Audit missing values in model features

MODEL_FEATURES = [
    # Historical sales
    "sales_at_origin",
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",
    "sales_rolling_mean_7",
    "sales_rolling_std_7",
    "sales_rolling_mean_28",
    "sales_rolling_std_28",
    "sales_rolling_max_28",

    # Product and store
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",

    # Origin price
    "origin_sell_price",
    "origin_price_missing",
    "origin_price_change",
    "origin_price_change_pct",
    "origin_price_vs_8w_avg",
    "origin_price_change_flag",

    # Target calendar
    "target_weekday",
    "target_wday",
    "target_month",
    "target_year",
    "target_is_weekend",
    "target_day_of_month",
    "target_week_of_year",
    "target_event_name_1",
    "target_event_type_1",
    "target_event_name_2",
    "target_event_type_2",
    "target_is_event_day",
    "target_is_snap_day",

    # Forecast horizon
    "horizon"
]

feature_list = ", ".join(
    f'"{column}"' for column in MODEL_FEATURES
)

missing_audit = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    {", ".join(
        f'COUNT(*) FILTER (WHERE "{column}" IS NULL) AS "{column}"'
        for column in MODEL_FEATURES
    )}

FROM read_parquet('{SUPERVISED_FILE.as_posix()}')
""").df()

missing_summary = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "missing_count": [
        missing_audit.iloc[0][column]
        for column in MODEL_FEATURES
    ]
})

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"]
    / missing_audit.iloc[0]["total_rows"]
    * 100
)

missing_summary = missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_summary)

,feature,missing_count,missing_percentage
30,target_event_name_2,15144000,99.789141
31,target_event_type_2,15144000,99.789141
29,target_event_type_1,13950000,91.921455
28,target_event_name_1,13950000,91.921455
18,origin_price_change_pct,3202052,21.099446
20,origin_price_change_flag,3202052,21.099446
17,origin_price_change,3202052,21.099446
19,origin_price_vs_8w_avg,3202052,21.099446
15,origin_sell_price,3169908,20.887638
0,sales_at_origin,0,0.000000


In [22]:
# Audit categorical feature cardinality

CATEGORICAL_FEATURES = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "target_weekday",
    "target_event_name_1",
    "target_event_type_1",
    "target_event_name_2",
    "target_event_type_2"
]

cardinality_results = []

for column in CATEGORICAL_FEATURES:

    result = con.execute(f"""
        SELECT
            COUNT(DISTINCT "{column}") AS unique_values,
            COUNT(*) FILTER (
                WHERE "{column}" IS NULL
            ) AS missing_values

        FROM read_parquet('{SUPERVISED_FILE.as_posix()}')
    """).fetchone()

    cardinality_results.append({
        "feature": column,
        "unique_values": result[0],
        "missing_values": result[1]
    })

cardinality_summary = pd.DataFrame(cardinality_results)

display(cardinality_summary)

,feature,unique_values,missing_values
0,item_id,1490,0
1,dept_id,7,0
2,cat_id,3,0
3,store_id,10,0
4,state_id,3,0
5,target_weekday,7,0
6,target_event_name_1,30,13950000
7,target_event_type_1,4,13950000
8,target_event_name_2,4,15144000
9,target_event_type_2,2,15144000


#### Investigate missing price and event patterns

In [23]:
price_event_audit = con.execute(f"""
WITH origin_data AS (
    SELECT DISTINCT
        id,
        origin_day_num,
        origin_sell_price,
        origin_price_missing,
        origin_price_change,
        origin_price_change_pct,
        origin_price_vs_8w_avg,
        origin_price_change_flag
    FROM read_parquet('{SUPERVISED_FILE.as_posix()}')
),

target_calendar AS (
    SELECT DISTINCT
        target_day,
        target_date,
        target_event_name_1,
        target_event_type_1,
        target_event_name_2,
        target_event_type_2
    FROM read_parquet('{SUPERVISED_FILE.as_posix()}')
)

SELECT
    'Price feature audit' AS audit_type,
    COUNT(*) AS total_observations,

    COUNT(*) FILTER (
        WHERE origin_price_missing = 1
    ) AS current_price_missing,

    COUNT(*) FILTER (
        WHERE origin_price_missing = 0
          AND origin_price_change IS NULL
    ) AS change_missing_but_current_price_available,

    COUNT(*) FILTER (
        WHERE origin_price_change_pct IS NULL
    ) AS price_change_pct_missing,

    COUNT(*) FILTER (
        WHERE origin_price_vs_8w_avg IS NULL
    ) AS price_8w_ratio_missing,

    COUNT(*) FILTER (
        WHERE origin_price_change_flag IS NULL
    ) AS price_change_flag_missing

FROM origin_data

UNION ALL

SELECT
    'Calendar event audit',
    COUNT(*),

    COUNT(*) FILTER (
        WHERE target_event_name_1 IS NOT NULL
    ),

    COUNT(*) FILTER (
        WHERE target_event_name_2 IS NOT NULL
    ),

    COUNT(*) FILTER (
        WHERE target_event_type_1 IS NOT NULL
    ),

    COUNT(*) FILTER (
        WHERE target_event_type_2 IS NOT NULL
    ),

    COUNT(*) FILTER (
        WHERE target_event_name_1 IS NULL
    )

FROM target_calendar
""").df()

display(price_event_audit)

,audit_type,total_observations,current_price_missing,change_missing_but_current_price_available,price_change_pct_missing,price_8w_ratio_missing,price_change_flag_missing
0,Price feature audit,542000,113211,1148,114359,114359,114359
1,Calendar event audit,1885,151,4,151,4,1734


### Finalise the Baseline Feature Schema

In [24]:
# --------------------------------------------------
# 1. Historical sales features
# --------------------------------------------------

HISTORICAL_FEATURES = [
    "sales_at_origin",
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",
    "sales_rolling_mean_7",
    "sales_rolling_std_7",
    "sales_rolling_mean_28",
    "sales_rolling_std_28",
    "sales_rolling_max_28"
]


# --------------------------------------------------
# 2. Product and store categorical features
# --------------------------------------------------

STATIC_CATEGORICAL_FEATURES = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]


# --------------------------------------------------
# 3. Origin price features
# --------------------------------------------------

PRICE_FEATURES = [
    "origin_sell_price",
    "origin_price_missing",
    "origin_price_change",
    "origin_price_change_pct",
    "origin_price_vs_8w_avg",
    "origin_price_change_flag"
]


# --------------------------------------------------
# 4. Target-date calendar features
# --------------------------------------------------

CALENDAR_NUMERIC_FEATURES = [
    "target_month",
    "target_year",
    "target_is_weekend",
    "target_day_of_month",
    "target_week_of_year",
    "target_is_event_day",
    "target_is_snap_day"
]

CALENDAR_CATEGORICAL_FEATURES = [
    "target_weekday",
    "target_event_name_1",
    "target_event_type_1"
]


# --------------------------------------------------
# 5. Forecast horizon
# --------------------------------------------------

HORIZON_FEATURES = [
    "horizon"
]


# --------------------------------------------------
# 6. Combine model predictors
# --------------------------------------------------

NUMERIC_FEATURES = (
    HISTORICAL_FEATURES
    + PRICE_FEATURES
    + CALENDAR_NUMERIC_FEATURES
    + HORIZON_FEATURES
)

CATEGORICAL_FEATURES = (
    STATIC_CATEGORICAL_FEATURES
    + CALENDAR_CATEGORICAL_FEATURES
)

MODEL_FEATURES_FINAL = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)

TARGET_COLUMN = "target_sales"


# --------------------------------------------------
# 7. Display final feature schema
# --------------------------------------------------

feature_summary = pd.DataFrame([
    {
        "feature_group": "Historical sales",
        "number_of_features": len(HISTORICAL_FEATURES)
    },
    {
        "feature_group": "Product and store",
        "number_of_features": len(STATIC_CATEGORICAL_FEATURES)
    },
    {
        "feature_group": "Origin price",
        "number_of_features": len(PRICE_FEATURES)
    },
    {
        "feature_group": "Target calendar numerical",
        "number_of_features": len(CALENDAR_NUMERIC_FEATURES)
    },
    {
        "feature_group": "Target calendar categorical",
        "number_of_features": len(CALENDAR_CATEGORICAL_FEATURES)
    },
    {
        "feature_group": "Forecast horizon",
        "number_of_features": len(HORIZON_FEATURES)
    }
])

display(feature_summary)

print(f"Total numerical features: {len(NUMERIC_FEATURES)}")
print(f"Total categorical features: {len(CATEGORICAL_FEATURES)}")
print(f"Total model predictors: {len(MODEL_FEATURES_FINAL)}")
print(f"Prediction target: {TARGET_COLUMN}")

,feature_group,number_of_features
0,Historical sales,10
1,Product and store,5
2,Origin price,6
3,Target calendar numerical,7
4,Target calendar categorical,3
5,Forecast horizon,1


Total numerical features: 24
Total categorical features: 8
Total model predictors: 32
Prediction target: target_sales


### Confirm numerical missing values and event categories

In [25]:
# Step 3.5.2 — Focused Missing Value and Category Audit

# --------------------------------------------------
# 1. Numerical missing-value audit
# --------------------------------------------------

numeric_missing_audit = con.execute(f"""
    SELECT
        {", ".join([
            f"COUNT(*) FILTER (WHERE {col} IS NULL) AS {col}"
            for col in NUMERIC_FEATURES
        ])}
    FROM read_parquet('{SUPERVISED_FILE.as_posix()}')
""").df()

display(numeric_missing_audit.T)


# --------------------------------------------------
# 2. Event category distribution
# --------------------------------------------------

event_category_audit = con.execute(f"""
    SELECT
        target_event_name_1,
        target_event_type_1,
        COUNT(*) AS observations
    FROM read_parquet('{SUPERVISED_FILE.as_posix()}')
    GROUP BY
        target_event_name_1,
        target_event_type_1
    ORDER BY observations DESC
    LIMIT 20
""").df()

display(event_category_audit)


# --------------------------------------------------
# 3. Weekday category distribution
# --------------------------------------------------

weekday_audit = con.execute(f"""
    SELECT
        target_weekday,
        COUNT(*) AS observations
    FROM read_parquet('{SUPERVISED_FILE.as_posix()}')
    GROUP BY target_weekday
    ORDER BY target_weekday
""").df()

display(weekday_audit)

,0
sales_at_origin,0
sales_lag_1,0
sales_lag_7,0
sales_lag_14,0
sales_lag_28,0
sales_rolling_mean_7,0
sales_rolling_std_7,0
sales_rolling_mean_28,0
sales_rolling_std_28,0
sales_rolling_max_28,0


,target_event_name_1,target_event_type_1,observations
0,NaN,NaN,13950000
1,Pesach End,Religious,48000
2,Mother's day,Cultural,46000
3,OrthodoxChristmas,Religious,42000
4,StPatricksDay,Cultural,42000
5,Easter,Cultural,42000
6,LentWeek2,Religious,42000
7,LentStart,Religious,42000
8,Purim End,Religious,42000
9,SuperBowl,Sporting,42000


,target_weekday,observations
0,Friday,2168000
1,Monday,2168000
2,Saturday,2168000
3,Sunday,2168000
4,Thursday,2168000
5,Tuesday,2168000
6,Wednesday,2168000


#### Conclusion

| Preprocessing Step | Seasonal Naive | LightGBM | LSTM | TFT |
| :--- | :--- | :--- | :--- | :--- |
| **Numerical scaling** | Not needed | Not needed | Recommended | Recommended |
| **Categorical encoding** | Not needed | Native categorical handling | Embeddings or encoding | Categorical embeddings |
| **Missing price values** | Not needed | Retain | Impute + indicators | Impute + indicators |
| **Missing event values** | Not needed | No_Event | No_Event encoding | No_Event encoding |
| **Historical sequences** | Weekly lag | Engineered features | Required | Required |

### Prepare Seasonal Naive dataset

In [34]:
import duckdb

DAILY_SALES_FILE = PROCESSED_DIR / "daily_sales.parquet"

con = duckdb.connect()

daily_sales_schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{DAILY_SALES_FILE.as_posix()}')
""").df()

display(daily_sales_schema)

,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,item_id,VARCHAR,YES,None,None,None
2,dept_id,VARCHAR,YES,None,None,None
3,cat_id,VARCHAR,YES,None,None,None
4,store_id,VARCHAR,YES,None,None,None
5,state_id,VARCHAR,YES,None,None,None
6,d,VARCHAR,YES,None,None,None
7,sales,BIGINT,YES,None,None,None


In [35]:
SEASONAL_NAIVE_DIR = PROCESSED_DIR / "seasonal_naive"
SEASONAL_NAIVE_DIR.mkdir(parents=True, exist_ok=True)

SEASONAL_NAIVE_FILE = (
    SEASONAL_NAIVE_DIR / "seasonal_naive_evaluation.parquet"
)

seasonal_naive_query = f"""
WITH evaluation_origins AS (
    SELECT *
    FROM (
        VALUES
            ('development_fold_1', 1801, 1802, 1829),
            ('development_fold_2', 1829, 1830, 1857),
            ('development_fold_3', 1857, 1858, 1885),
            ('internal_test', 1885, 1886, 1913),
            ('official_evaluation', 1913, 1914, 1941)
    ) AS t(evaluation_type, origin_day_num, target_start, target_end)
),

daily_sales AS (
    SELECT
        id,
        TRY_CAST(SUBSTRING(d, 3) AS INTEGER) AS day_num,
        sales
    FROM read_parquet('{DAILY_SALES_FILE.as_posix()}')
),

evaluation_targets AS (
    SELECT
        e.evaluation_type,
        e.origin_day_num,
        s.id,
        s.day_num AS target_day_num,
        s.sales AS target_sales
    FROM daily_sales s
    INNER JOIN evaluation_origins e
        ON s.day_num BETWEEN e.target_start AND e.target_end
),

naive_forecasts AS (
    SELECT
        t.*,
        h.sales AS naive_forecast,
        t.target_day_num - 7 AS naive_reference_day_num
    FROM evaluation_targets t
    LEFT JOIN daily_sales h
        ON t.id = h.id
        AND h.day_num = t.target_day_num - 7
)

SELECT *
FROM naive_forecasts
ORDER BY evaluation_type, id, target_day_num
"""

con.execute(f"""
    COPY ({seasonal_naive_query})
    TO '{SEASONAL_NAIVE_FILE.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Seasonal Naive evaluation dataset saved:")
print(SEASONAL_NAIVE_FILE)

Seasonal Naive evaluation dataset saved:
c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\seasonal_naive\seasonal_naive_evaluation.parquet


In [36]:
seasonal_naive_validation = con.execute(f"""
SELECT
    evaluation_type,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS unique_series,
    COUNT(DISTINCT target_day_num) AS unique_target_days,
    MIN(target_day_num) AS first_target_day,
    MAX(target_day_num) AS last_target_day,
    COUNT(*) FILTER (
        WHERE target_sales IS NULL
    ) AS missing_actual_sales,
    COUNT(*) FILTER (
        WHERE naive_forecast IS NULL
    ) AS missing_naive_forecasts,
    COUNT(*) FILTER (
        WHERE naive_reference_day_num != target_day_num - 7
    ) AS invalid_reference_alignment,
    COUNT(*) FILTER (
        WHERE naive_reference_day_num > origin_day_num
    ) AS future_reference_observations
FROM read_parquet('{SEASONAL_NAIVE_FILE.as_posix()}')
GROUP BY evaluation_type
ORDER BY
    CASE evaluation_type
        WHEN 'development_fold_1' THEN 1
        WHEN 'development_fold_2' THEN 2
        WHEN 'development_fold_3' THEN 3
        WHEN 'internal_test' THEN 4
        WHEN 'official_evaluation' THEN 5
    END
""").df()

display(seasonal_naive_validation)

,evaluation_type,total_rows,unique_series,unique_target_days,first_target_day,last_target_day,missing_actual_sales,missing_naive_forecasts,invalid_reference_alignment,future_reference_observations
0,development_fold_1,853720,30490,28,1802,1829,0,0,0,640290
1,development_fold_2,853720,30490,28,1830,1857,0,0,0,640290
2,development_fold_3,853720,30490,28,1858,1885,0,0,0,640290
3,internal_test,853720,30490,28,1886,1913,0,0,0,640290
4,official_evaluation,853720,30490,28,1914,1941,0,0,0,640290


In [37]:
SELECTED_SERIES_FILE = METADATA_DIR / "selected_2000_series.parquet"

seasonal_naive_query = f"""
WITH selected_series AS (
    SELECT DISTINCT id
    FROM read_parquet('{SELECTED_SERIES_FILE.as_posix()}')
),

evaluation_origins AS (
    SELECT *
    FROM (
        VALUES
            ('development_fold_1', 1801, 1802, 1829),
            ('development_fold_2', 1829, 1830, 1857),
            ('development_fold_3', 1857, 1858, 1885),
            ('internal_test', 1885, 1886, 1913),
            ('official_evaluation', 1913, 1914, 1941)
    ) AS t(evaluation_type, origin_day_num, target_start, target_end)
),

daily_sales AS (
    SELECT
        id,
        TRY_CAST(SUBSTRING(d, 3) AS INTEGER) AS day_num,
        sales
    FROM read_parquet('{DAILY_SALES_FILE.as_posix()}')
),

evaluation_targets AS (
    SELECT
        e.evaluation_type,
        e.origin_day_num,
        s.id,
        s.day_num AS target_day_num,
        s.sales AS target_sales
    FROM daily_sales s
    INNER JOIN selected_series ss
        ON s.id = ss.id
    INNER JOIN evaluation_origins e
        ON s.day_num BETWEEN e.target_start AND e.target_end
),

naive_forecasts AS (
    SELECT
        t.*,
        t.target_day_num - t.origin_day_num AS horizon,

        t.origin_day_num - 6
            + ((t.target_day_num - t.origin_day_num - 1) % 7)
            AS naive_reference_day_num,

        h.sales AS naive_forecast

    FROM evaluation_targets t
    LEFT JOIN daily_sales h
        ON t.id = h.id
        AND h.day_num = (
            t.origin_day_num - 6
            + ((t.target_day_num - t.origin_day_num - 1) % 7)
        )
)

SELECT *
FROM naive_forecasts
ORDER BY evaluation_type, id, target_day_num
"""

con.execute(f"""
    COPY ({seasonal_naive_query})
    TO '{SEASONAL_NAIVE_FILE.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Corrected Seasonal Naive evaluation dataset saved:")
print(SEASONAL_NAIVE_FILE)

Corrected Seasonal Naive evaluation dataset saved:
c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\seasonal_naive\seasonal_naive_evaluation.parquet


In [38]:
seasonal_naive_validation = con.execute(f"""
SELECT
    evaluation_type,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS unique_series,
    COUNT(DISTINCT target_day_num) AS unique_target_days,
    MIN(target_day_num) AS first_target_day,
    MAX(target_day_num) AS last_target_day,

    COUNT(*) FILTER (
        WHERE target_sales IS NULL
    ) AS missing_actual_sales,

    COUNT(*) FILTER (
        WHERE naive_forecast IS NULL
    ) AS missing_naive_forecasts,

    COUNT(*) FILTER (
        WHERE naive_reference_day_num > origin_day_num
    ) AS future_reference_observations,

    COUNT(*) FILTER (
        WHERE naive_reference_day_num !=
            origin_day_num - 6
            + ((target_day_num - origin_day_num - 1) % 7)
    ) AS invalid_reference_alignment

FROM read_parquet('{SEASONAL_NAIVE_FILE.as_posix()}')
GROUP BY evaluation_type
ORDER BY MIN(origin_day_num)
""").df()

display(seasonal_naive_validation)

,evaluation_type,total_rows,unique_series,unique_target_days,first_target_day,last_target_day,missing_actual_sales,missing_naive_forecasts,future_reference_observations,invalid_reference_alignment
0,development_fold_1,56000,2000,28,1802,1829,0,0,0,0
1,development_fold_2,56000,2000,28,1830,1857,0,0,0,0
2,development_fold_3,56000,2000,28,1858,1885,0,0,0,0
3,internal_test,56000,2000,28,1886,1913,0,0,0,0
4,official_evaluation,56000,2000,28,1914,1941,0,0,0,0


### LightGBM dataset preparation

#### Create a model-ready categorical preprocessing view

In [ ]:
# Prepare Categorical Features

from pathlib import Path

# Locate project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CATEGORICAL_FILE = (
    PROCESSED_DIR / "supervised_training_examples_categorical.parquet"
)

con.execute(f"""
    COPY (
        SELECT
            * REPLACE (
                COALESCE(
                    target_event_name_1,
                    'No_Event'
                ) AS target_event_name_1,

                COALESCE(
                    target_event_type_1,
                    'No_Event'
                ) AS target_event_type_1
            )
        FROM read_parquet('{SUPERVISED_FILE.as_posix()}')
    )
    TO '{CATEGORICAL_FILE.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Categorical preprocessing completed.")
print(f"Output: {CATEGORICAL_FILE}")

Categorical preprocessing completed.
Output: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\supervised_training_examples_categorical.parquet


In [ ]:
# Validate categorical preprocessing

categorical_validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE target_event_name_1 IS NULL
        ) AS missing_event_names,

        COUNT(*) FILTER (
            WHERE target_event_type_1 IS NULL
        ) AS missing_event_types,

        COUNT(*) FILTER (
            WHERE target_event_name_1 = 'No_Event'
        ) AS no_event_name_rows,

        COUNT(*) FILTER (
            WHERE target_event_type_1 = 'No_Event'
        ) AS no_event_type_rows,

        COUNT(DISTINCT target_event_name_1) AS event_name_categories,

        COUNT(DISTINCT target_event_type_1) AS event_type_categories

    FROM read_parquet('{CATEGORICAL_FILE.as_posix()}')
""").df()

display(categorical_validation)

,total_rows,missing_event_names,missing_event_types,no_event_name_rows,no_event_type_rows,event_name_categories,event_type_categories
0,15176000,0,0,13950000,13950000,31,5


#### Buiild all folds

In [201]:
from pathlib import Path
import duckdb

LIGHTGBM_DIR = (
    PROJECT_ROOT / "data" / "processed" / "lightgbm_corrected"
)
LIGHTGBM_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "supervised_training_examples_categorical.parquet"
)

source_sql = SOURCE_FILE.as_posix()

fold_configs = [
    {
        "fold": 1,
        "validation_type": "development_validation_fold_1",
        "cutoff": 1801,
        "expected_origins": 246
    },
    {
        "fold": 2,
        "validation_type": "development_validation_fold_2",
        "cutoff": 1829,
        "expected_origins": 251
    },
    {
        "fold": 3,
        "validation_type": "development_validation_fold_3",
        "cutoff": 1857,
        "expected_origins": 256
    },
    {
        "fold": 4,
        "validation_type": "internal_test",
        "cutoff": 1885,
        "expected_origins": 261
    },
    {
        "fold": 5,
        "validation_type": "official_evaluation",
        "cutoff": 1913,
        "expected_origins": 266
    }
]

expected_series = 2000
expected_validation_rows = 56000

audit_columns = [
    "id",
    "demand_class",
    "origin_day_num",
    "origin_day",
    "origin_type",
    "target_day_num",
    "target_day",
    "horizon"
]

selected_columns = list(dict.fromkeys(
    audit_columns + MODEL_FEATURES_FINAL + [TARGET_COLUMN]
))

selected_columns_sql = ", ".join(
    f'"{column}"' for column in selected_columns
)

con = duckdb.connect()

source_schema = set(
    con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{source_sql}')"
    ).df()["column_name"]
)

missing_columns = set(selected_columns) - source_schema

assert not missing_columns, (
    f"Missing source columns: {sorted(missing_columns)}"
)

audit_results = []

for config in fold_configs:

    fold_num = config["fold"]
    cutoff = config["cutoff"]
    validation_type = config["validation_type"]
    expected_origins = config["expected_origins"]

    train_file = (
        LIGHTGBM_DIR / f"fold_{fold_num}_train.parquet"
    )

    validation_file = (
        LIGHTGBM_DIR / f"fold_{fold_num}_validation.parquet"
    )

    train_sql = train_file.as_posix()
    validation_sql = validation_file.as_posix()

    print(f"\nPreparing LightGBM fold {fold_num}")

    con.execute(f"""
        COPY (
            SELECT {selected_columns_sql}
            FROM read_parquet('{source_sql}')
            WHERE origin_day_num + 28 <= {cutoff}
        )
        TO '{train_sql}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    con.execute(f"""
        COPY (
            SELECT {selected_columns_sql}
            FROM read_parquet('{source_sql}')
            WHERE origin_type = '{validation_type}'
        )
        TO '{validation_sql}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    train_stats = con.execute(f"""
        SELECT
            COUNT(*) AS row_count,
            COUNT(DISTINCT id) AS series_count,
            COUNT(DISTINCT origin_day_num) AS origin_count,
            COUNT(DISTINCT horizon) AS horizon_count,
            MIN(origin_day_num) AS first_origin,
            MAX(origin_day_num) AS last_origin,
            COUNT(*) FILTER (
                WHERE origin_day_num + 28 > {cutoff}
            ) AS cutoff_violations,
            COUNT(*) FILTER (
                WHERE target_day_num IS NULL
                   OR target_sales IS NULL
            ) AS missing_targets
        FROM read_parquet('{train_sql}')
    """).df().iloc[0]

    validation_stats = con.execute(f"""
        SELECT
            COUNT(*) AS row_count,
            COUNT(DISTINCT id) AS series_count,
            COUNT(DISTINCT origin_day_num) AS origin_count,
            COUNT(DISTINCT horizon) AS horizon_count,
            MIN(origin_day_num) AS first_origin,
            MAX(origin_day_num) AS last_origin,
            MIN(target_day_num) AS first_target,
            MAX(target_day_num) AS last_target
        FROM read_parquet('{validation_sql}')
    """).df().iloc[0]

    duplicate_training_rows = con.execute(f"""
        SELECT COUNT(*)
        FROM (
            SELECT id, origin_day_num, horizon
            FROM read_parquet('{train_sql}')
            GROUP BY id, origin_day_num, horizon
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0]

    duplicate_validation_rows = con.execute(f"""
        SELECT COUNT(*)
        FROM (
            SELECT id, origin_day_num, horizon
            FROM read_parquet('{validation_sql}')
            GROUP BY id, origin_day_num, horizon
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0]

    expected_training_rows = (
        expected_origins * expected_series * 28
    )

    assert train_stats["row_count"] == expected_training_rows
    assert train_stats["series_count"] == expected_series
    assert train_stats["origin_count"] == expected_origins
    assert train_stats["horizon_count"] == 28
    assert train_stats["cutoff_violations"] == 0
    assert train_stats["missing_targets"] == 0
    assert duplicate_training_rows == 0

    assert validation_stats["row_count"] == expected_validation_rows
    assert validation_stats["series_count"] == expected_series
    assert validation_stats["origin_count"] == 1
    assert validation_stats["horizon_count"] == 28
    assert validation_stats["first_origin"] == cutoff
    assert validation_stats["last_origin"] == cutoff
    assert validation_stats["first_target"] == cutoff + 1
    assert validation_stats["last_target"] == cutoff + 28
    assert duplicate_validation_rows == 0

    audit_results.append({
        "fold": fold_num,
        "training_rows": int(train_stats["row_count"]),
        "training_origins": int(train_stats["origin_count"]),
        "validation_rows": int(validation_stats["row_count"]),
        "validation_origin": int(validation_stats["first_origin"]),
        "status": "PASSED"
    })

    print(f"Training rows: {train_stats['row_count']:,}")
    print(f"Training origins: {train_stats['origin_count']}")
    print(f"Validation rows: {validation_stats['row_count']:,}")
    print(f"Validation origin: {validation_stats['first_origin']}")
    print("Duplicate training keys:", duplicate_training_rows)
    print("Duplicate validation keys:", duplicate_validation_rows)
    print("Status: PASSED")

con.close()

audit_df = pd.DataFrame(audit_results)

print("\nLightGBM corrected fold preparation summary:")
display(audit_df)

print("\nAll five corrected LightGBM folds prepared successfully.")
print("Output directory:", LIGHTGBM_DIR)


Preparing LightGBM fold 1
Training rows: 13,776,000
Training origins: 246
Validation rows: 56,000
Validation origin: 1801
Duplicate training keys: 0
Duplicate validation keys: 0
Status: PASSED

Preparing LightGBM fold 2
Training rows: 14,056,000
Training origins: 251
Validation rows: 56,000
Validation origin: 1829
Duplicate training keys: 0
Duplicate validation keys: 0
Status: PASSED

Preparing LightGBM fold 3
Training rows: 14,336,000
Training origins: 256
Validation rows: 56,000
Validation origin: 1857
Duplicate training keys: 0
Duplicate validation keys: 0
Status: PASSED

Preparing LightGBM fold 4
Training rows: 14,616,000
Training origins: 261
Validation rows: 56,000
Validation origin: 1885
Duplicate training keys: 0
Duplicate validation keys: 0
Status: PASSED

Preparing LightGBM fold 5
Training rows: 14,896,000
Training origins: 266
Validation rows: 56,000
Validation origin: 1913
Duplicate training keys: 0
Duplicate validation keys: 0
Status: PASSED

LightGBM corrected fold prepa

,fold,training_rows,training_origins,validation_rows,validation_origin,status
0,1,13776000,246,56000,1801,PASSED
1,2,14056000,251,56000,1829,PASSED
2,3,14336000,256,56000,1857,PASSED
3,4,14616000,261,56000,1885,PASSED
4,5,14896000,266,56000,1913,PASSED



All five corrected LightGBM folds prepared successfully.
Output directory: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lightgbm_corrected


### LSTM Dataset Preparation

In [ ]:
# Inspect LSTM source data

LSTM_SERIES_FILE = METADATA_DIR / "selected_200_series_tft.parquet"

DAILY_MODEL_FEATURES_FILE = (
    PROCESSED_DIR / "daily_model_features.parquet"
)

# Inspect source schema
lstm_source_schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{DAILY_MODEL_FEATURES_FILE.as_posix()}'
    )
""").df()

display(lstm_source_schema)

# Inspect selected series
lstm_series_audit = con.execute(f"""
    SELECT
        COUNT(*) AS total_series,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(DISTINCT demand_class) AS demand_classes
    FROM read_parquet(
        '{LSTM_SERIES_FILE.as_posix()}'
    )
""").df()

display(lstm_series_audit)

,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,item_id,VARCHAR,YES,None,None,None
2,dept_id,VARCHAR,YES,None,None,None
3,cat_id,VARCHAR,YES,None,None,None
4,store_id,VARCHAR,YES,None,None,None
5,state_id,VARCHAR,YES,None,None,None
6,d,VARCHAR,YES,None,None,None
7,sales,BIGINT,YES,None,None,None
8,date,DATE,YES,None,None,None
9,wm_yr_wk,BIGINT,YES,None,None,None


,total_series,unique_series,demand_classes
0,200,200,5


In [50]:
# 1. Audit selected 200 series
lstm_series_summary = con.execute(f"""
    SELECT
        demand_class,
        COUNT(*) AS series_count
    FROM read_parquet('{LSTM_SERIES_FILE.as_posix()}')
    GROUP BY demand_class
    ORDER BY demand_class
""").df()

print("Selected series by demand class:")
display(lstm_series_summary)

# 2. Confirm unique series
lstm_series_count = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(DISTINCT demand_class) AS demand_classes
    FROM read_parquet('{LSTM_SERIES_FILE.as_posix()}')
""").df()

print("Selected series summary:")
display(lstm_series_count)

# 3. Audit historical sales and relevant feature missingness
lstm_source_audit = con.execute(f"""
    SELECT
        COUNT(*) AS total_observations,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(*) FILTER (WHERE sales IS NULL) AS missing_sales,
        COUNT(*) FILTER (WHERE sell_price IS NULL) AS missing_sell_price,
        COUNT(*) FILTER (WHERE event_name_1 IS NULL) AS missing_event_name,
        COUNT(*) FILTER (WHERE event_type_1 IS NULL) AS missing_event_type,
        COUNT(*) FILTER (WHERE is_snap_day IS NULL) AS missing_snap_indicator
    FROM read_parquet('{DAILY_MODEL_FEATURES_FILE.as_posix()}')
    WHERE id IN (
        SELECT id
        FROM read_parquet('{LSTM_SERIES_FILE.as_posix()}')
    )
""").df()

print("LSTM source data audit:")
display(lstm_source_audit)

Selected series by demand class:


,demand_class,series_count
0,erratic,6
1,intermittent,144
2,lumpy,36
3,no_demand,1
4,smooth,13


Selected series summary:


,total_rows,unique_series,demand_classes
0,200,200,5


LSTM source data audit:


,total_observations,unique_series,missing_sales,missing_sell_price,missing_event_name,missing_event_type,missing_snap_indicator
0,388200,200,0,93751,356600,356600,0


#### Define LSTM sequence feature schema

In [52]:
# 1. Historical encoder features
LSTM_HISTORICAL_NUMERIC_FEATURES = [
    "sales",
    "sell_price",
    "price_missing",
    "price_change",
    "price_change_pct",
    "price_vs_8w_avg",
    "price_change_flag",
    "is_weekend",
    "day_of_month",
    "week_of_year",
    "is_event_day",
    "is_snap_day"
]

LSTM_HISTORICAL_CATEGORICAL_FEATURES = [
    "weekday",
    "event_name_1",
    "event_type_1"
]

# 2. Price missingness indicators
LSTM_PRICE_MISSING_INDICATORS = [
    "price_missing",
    "price_change_missing",
    "price_change_pct_missing",
    "price_vs_8w_avg_missing",
    "price_change_flag_missing"
]

LSTM_PRICE_MISSING_SOURCE_COLUMNS = {
    "price_change": "price_change_missing",
    "price_change_pct": "price_change_pct_missing",
    "price_vs_8w_avg": "price_vs_8w_avg_missing",
    "price_change_flag": "price_change_flag_missing"
}

# 3. Future-known decoder features
LSTM_FUTURE_NUMERIC_FEATURES = [
    "month",
    "year",
    "day_of_month",
    "week_of_year",
    "is_weekend",
    "is_event_day",
    "is_snap_day"
]

LSTM_FUTURE_CATEGORICAL_FEATURES = [
    "weekday",
    "event_name_1",
    "event_type_1"
]

# 4. Static categorical features
LSTM_STATIC_FEATURES = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

# 5. Sequence configuration
LSTM_ENCODER_LENGTH = 56
LSTM_PREDICTION_LENGTH = 28

# 6. Prediction target and quantiles
LSTM_TARGET_COLUMN = "sales"
LSTM_QUANTILES = [0.10, 0.50, 0.90]

# 7. Display configuration summary
print("Historical numerical features:",
      len(LSTM_HISTORICAL_NUMERIC_FEATURES))

print("Historical categorical features:",
      len(LSTM_HISTORICAL_CATEGORICAL_FEATURES))

print("Price missingness indicators:",
      len(LSTM_PRICE_MISSING_INDICATORS))

print("Future numerical features:",
      len(LSTM_FUTURE_NUMERIC_FEATURES))

print("Future categorical features:",
      len(LSTM_FUTURE_CATEGORICAL_FEATURES))

print("Static categorical features:",
      len(LSTM_STATIC_FEATURES))

print("Encoder length:", LSTM_ENCODER_LENGTH)
print("Prediction length:", LSTM_PREDICTION_LENGTH)
print("Target:", LSTM_TARGET_COLUMN)
print("Quantiles:", LSTM_QUANTILES)


Historical numerical features: 12
Historical categorical features: 3
Price missingness indicators: 5
Future numerical features: 7
Future categorical features: 3
Static categorical features: 5
Encoder length: 56
Prediction length: 28
Target: sales
Quantiles: [0.1, 0.5, 0.9]


#### Confirm forecast-window alignment

In [53]:
LSTM_EVALUATION_ORIGINS = [
    ("development_fold_1", 1801),
    ("development_fold_2", 1829),
    ("development_fold_3", 1857),
    ("internal_test", 1885),
    ("official_evaluation", 1913)
]

lstm_window_summary = pd.DataFrame([
    {
        "evaluation_type": name,
        "forecast_origin": origin,
        "encoder_start": origin - LSTM_ENCODER_LENGTH + 1,
        "encoder_end": origin,
        "decoder_start": origin + 1,
        "decoder_end": origin + LSTM_PREDICTION_LENGTH,
        "encoder_days": LSTM_ENCODER_LENGTH,
        "decoder_days": LSTM_PREDICTION_LENGTH
    }
    for name, origin in LSTM_EVALUATION_ORIGINS
])

display(lstm_window_summary)

,evaluation_type,forecast_origin,encoder_start,encoder_end,decoder_start,decoder_end,encoder_days,decoder_days
0,development_fold_1,1801,1746,1801,1802,1829,56,28
1,development_fold_2,1829,1774,1829,1830,1857,56,28
2,development_fold_3,1857,1802,1857,1858,1885,56,28
3,internal_test,1885,1830,1885,1886,1913,56,28
4,official_evaluation,1913,1858,1913,1914,1941,56,28


#### Determine LSTM training sequence origins
- reusing the existing weekly forecast origins from forecast_origins.parquet, but only retaining origins whose entire 28-day target period falls within the relevant training cutoff.

In [54]:
LSTM_ORIGINS_FILE = METADATA_DIR / "forecast_origins.parquet"

lstm_training_origins = con.execute(f"""
    SELECT
        origin_day_num,
        origin_day
    FROM read_parquet('{LSTM_ORIGINS_FILE.as_posix()}')
    WHERE origin_day_num + {LSTM_PREDICTION_LENGTH} <= 1801
      AND origin_day_num >= {LSTM_ENCODER_LENGTH}
    ORDER BY origin_day_num
""").df()

print("Number of Fold 1 eligible training origins:",
      len(lstm_training_origins))

display(lstm_training_origins.head(10))
display(lstm_training_origins.tail(10))

Number of Fold 1 eligible training origins: 246


,origin_day_num,origin_day
0,56,d_56
1,63,d_63
2,70,d_70
3,77,d_77
4,84,d_84
5,91,d_91
6,98,d_98
7,105,d_105
8,112,d_112
9,119,d_119


,origin_day_num,origin_day
236,1708,d_1708
237,1715,d_1715
238,1722,d_1722
239,1729,d_1729
240,1736,d_1736
241,1743,d_1743
242,1750,d_1750
243,1757,d_1757
244,1764,d_1764
245,1771,d_1771


In [ ]:
# Training origin summary

LSTM_FOLD_CUTOFFS = {
    "development_fold_1": 1801,
    "development_fold_2": 1829,
    "development_fold_3": 1857,
    "internal_test": 1885,
    "official_evaluation": 1913
}

all_lstm_origins = con.execute(f"""
    SELECT
        origin_day_num,
        origin_day,
        origin_type
    FROM read_parquet('{LSTM_ORIGINS_FILE.as_posix()}')
    WHERE origin_day_num >= {LSTM_ENCODER_LENGTH}
    ORDER BY origin_day_num
""").df()

lstm_fold_origin_summary = []

for fold_name, cutoff in LSTM_FOLD_CUTOFFS.items():

    eligible = all_lstm_origins[
        all_lstm_origins["origin_day_num"]
        + LSTM_PREDICTION_LENGTH <= cutoff
    ].copy()

    lstm_fold_origin_summary.append({
        "evaluation_type": fold_name,
        "training_cutoff": cutoff,
        "training_origins": len(eligible),
        "first_training_origin": eligible["origin_day_num"].min(),
        "last_training_origin": eligible["origin_day_num"].max(),
        "training_sequences": (
            len(eligible) * 200
        ),
        "encoder_length": LSTM_ENCODER_LENGTH,
        "prediction_length": LSTM_PREDICTION_LENGTH
    })

lstm_fold_origin_summary = pd.DataFrame(
    lstm_fold_origin_summary
)

display(lstm_fold_origin_summary)

,evaluation_type,training_cutoff,training_origins,first_training_origin,last_training_origin,training_sequences,encoder_length,prediction_length
0,development_fold_1,1801,246,56,1771,49200,56,28
1,development_fold_2,1829,251,56,1801,50200,56,28
2,development_fold_3,1857,256,56,1829,51200,56,28
3,internal_test,1885,261,56,1857,52200,56,28
4,official_evaluation,1913,266,56,1885,53200,56,28


#### Extract the LSTM daily panel

In [56]:
# Step 3.8.3.8 — Extract LSTM daily panel
LSTM_DIR = PROCESSED_DIR / "lstm"
LSTM_DIR.mkdir(parents=True, exist_ok=True)

LSTM_DAILY_PANEL_FILE = LSTM_DIR / "lstm_daily_panel.parquet"

lstm_panel_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "d",
    "sales",
    "sell_price",
    "price_missing",
    "price_change",
    "price_change_pct",
    "price_vs_8w_avg",
    "price_change_flag",
    "weekday",
    "month",
    "year",
    "is_weekend",
    "day_of_month",
    "week_of_year",
    "is_event_day",
    "is_snap_day",
    "event_name_1",
    "event_type_1"
]

lstm_panel_select = ", ".join(lstm_panel_columns)

con.execute(f"""
    COPY (
        SELECT
            {lstm_panel_select}
        FROM read_parquet(
            '{DAILY_MODEL_FEATURES_FILE.as_posix()}'
        )
        WHERE id IN (
            SELECT id
            FROM read_parquet(
                '{LSTM_SERIES_FILE.as_posix()}'
            )
        )
        ORDER BY id, TRY_CAST(SUBSTRING(d, 3) AS INTEGER)
    )
    TO '{LSTM_DAILY_PANEL_FILE.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("LSTM daily panel saved:", LSTM_DAILY_PANEL_FILE)


LSTM daily panel saved: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lstm\lstm_daily_panel.parquet


In [57]:
# Validate LSTM daily panel
lstm_panel_audit = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(DISTINCT d) AS unique_days,
        MIN(TRY_CAST(SUBSTRING(d, 3) AS INTEGER))
            AS first_day_num,
        MAX(TRY_CAST(SUBSTRING(d, 3) AS INTEGER))
            AS last_day_num,
        COUNT(*) FILTER (WHERE sales IS NULL)
            AS missing_sales,
        COUNT(*) FILTER (WHERE price_missing IS NULL)
            AS missing_price_indicator,
        COUNT(*) FILTER (WHERE id IS NULL OR d IS NULL)
            AS missing_identifiers
    FROM read_parquet('{LSTM_DAILY_PANEL_FILE.as_posix()}')
""").df()

display(lstm_panel_audit)

print("Expected rows:", 200 * 1941)
print("Expected series:", 200)
print("Expected days per series:", 1941)
print("Expected columns:", len(lstm_panel_columns))


,total_rows,unique_series,unique_days,first_day_num,last_day_num,missing_sales,missing_price_indicator,missing_identifiers
0,388200,200,1941,1,1941,0,0,0


Expected rows: 388200
Expected series: 200
Expected days per series: 1941
Expected columns: 24


#### Validate missingness indicators

In [ ]:

LSTM_PRICE_MISSING_SOURCE_COLUMNS = {
    "price_change": "price_change_missing",
    "price_change_pct": "price_change_pct_missing",
    "price_vs_8w_avg": "price_vs_8w_avg_missing",
    "price_change_flag": "price_change_flag_missing"
}

lstm_price_missing_sql = ",\n".join(
    [
        f"""
        CASE
            WHEN {source_column} IS NULL THEN 1
            ELSE 0
        END AS {indicator_column}
        """
        for source_column, indicator_column
        in LSTM_PRICE_MISSING_SOURCE_COLUMNS.items()
    ]
)

LSTM_DAILY_PANEL_WITH_INDICATORS_FILE = (
    LSTM_DIR / "lstm_daily_panel_with_indicators.parquet"
)

con.execute(f"""
    COPY (
        SELECT
            *,
            {lstm_price_missing_sql}
        FROM read_parquet(
            '{LSTM_DAILY_PANEL_FILE.as_posix()}'
        )
    )
    TO '{LSTM_DAILY_PANEL_WITH_INDICATORS_FILE.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved:", LSTM_DAILY_PANEL_WITH_INDICATORS_FILE)


Saved: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lstm\lstm_daily_panel_with_indicators.parquet


In [ ]:

lstm_missing_indicator_columns = [
    "price_missing",
    "price_change_missing",
    "price_change_pct_missing",
    "price_vs_8w_avg_missing",
    "price_change_flag_missing"
]

lstm_indicator_audit = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        {", ".join(
            f"SUM({col}) AS {col}_count"
            for col in lstm_missing_indicator_columns
        )}
    FROM read_parquet(
        '{LSTM_DAILY_PANEL_WITH_INDICATORS_FILE.as_posix()}'
    )
""").df()

display(lstm_indicator_audit)

lstm_indicator_null_audit = con.execute(f"""
    SELECT
        {", ".join(
            f"COUNT(*) FILTER (WHERE {col} IS NULL) AS {col}_nulls"
            for col in lstm_missing_indicator_columns
        )}
    FROM read_parquet(
        '{LSTM_DAILY_PANEL_WITH_INDICATORS_FILE.as_posix()}'
    )
""").df()

display(lstm_indicator_null_audit)


,total_rows,price_missing_count,price_change_missing_count,price_change_pct_missing_count,price_vs_8w_avg_missing_count,price_change_flag_missing_count
0,388200,93751.0,95151.0,95151.0,95151.0,95151.0


,price_missing_nulls,price_change_missing_nulls,price_change_pct_missing_nulls,price_vs_8w_avg_missing_nulls,price_change_flag_missing_nulls
0,0,0,0,0,0


#### Finalize LSTM historical numerical features

In [ ]:
# Step 3.8.3.10 — Final LSTM feature schema
LSTM_HISTORICAL_NUMERIC_FEATURES = [
    "sales",
    "sell_price",
    "price_missing",
    "price_change",
    "price_change_pct",
    "price_vs_8w_avg",
    "price_change_flag",
    "price_change_missing",
    "price_change_pct_missing",
    "price_vs_8w_avg_missing",
    "price_change_flag_missing",
    "is_weekend",
    "day_of_month",
    "week_of_year",
    "is_event_day",
    "is_snap_day"
]

LSTM_HISTORICAL_CATEGORICAL_FEATURES = [
    "weekday",
    "event_name_1",
    "event_type_1"
]

LSTM_FUTURE_NUMERIC_FEATURES = [
    "month",
    "year",
    "day_of_month",
    "week_of_year",
    "is_weekend",
    "is_event_day",
    "is_snap_day"
]

LSTM_FUTURE_CATEGORICAL_FEATURES = [
    "weekday",
    "event_name_1",
    "event_type_1"
]

LSTM_STATIC_FEATURES = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

LSTM_ALL_NUMERIC_FEATURES = list(dict.fromkeys(
    LSTM_HISTORICAL_NUMERIC_FEATURES
    + LSTM_FUTURE_NUMERIC_FEATURES
))

LSTM_ALL_CATEGORICAL_FEATURES = list(dict.fromkeys(
    LSTM_HISTORICAL_CATEGORICAL_FEATURES
    + LSTM_FUTURE_CATEGORICAL_FEATURES
    + LSTM_STATIC_FEATURES
))

print("Historical numerical features:",
      len(LSTM_HISTORICAL_NUMERIC_FEATURES))
print("Historical categorical features:",
      len(LSTM_HISTORICAL_CATEGORICAL_FEATURES))
print("Future numerical features:",
      len(LSTM_FUTURE_NUMERIC_FEATURES))
print("Future categorical features:",
      len(LSTM_FUTURE_CATEGORICAL_FEATURES))
print("Static categorical features:",
      len(LSTM_STATIC_FEATURES))
print("Unique numerical features across encoder/decoder:",
      len(LSTM_ALL_NUMERIC_FEATURES))
print("Unique categorical features across encoder/decoder:",
      len(LSTM_ALL_CATEGORICAL_FEATURES))


Historical numerical features: 16
Historical categorical features: 3
Future numerical features: 7
Future categorical features: 3
Static categorical features: 5
Unique numerical features across encoder/decoder: 18
Unique categorical features across encoder/decoder: 8


#### Audit numerical features before preprocessing

In [ ]:
# Numerical feature audit
LSTM_NUMERICAL_AUDIT_FEATURES = LSTM_ALL_NUMERIC_FEATURES

lstm_numeric_audit_sql = []

for col in LSTM_NUMERICAL_AUDIT_FEATURES:
    lstm_numeric_audit_sql.append(f"""
        SELECT
            '{col}' AS feature,
            COUNT(*) AS total_rows,
            COUNT({col}) AS non_missing,
            COUNT(*) FILTER (
                WHERE {col} IS NULL
            ) AS missing_count,
            MIN({col}) AS minimum,
            quantile_cont({col}, 0.01) AS percentile_01,
            quantile_cont({col}, 0.25) AS percentile_25,
            quantile_cont({col}, 0.50) AS median,
            quantile_cont({col}, 0.75) AS percentile_75,
            quantile_cont({col}, 0.99) AS percentile_99,
            MAX({col}) AS maximum,
            AVG({col}) AS mean,
            STDDEV_POP({col}) AS std_dev,
            COUNT(*) FILTER (
                WHERE {col} = 0
            ) AS zero_count
        FROM read_parquet(
            '{LSTM_DAILY_PANEL_WITH_INDICATORS_FILE.as_posix()}'
        )
    """)

lstm_numeric_audit_query = """
    UNION ALL
    """.join(lstm_numeric_audit_sql)

lstm_numeric_audit = con.execute(
    lstm_numeric_audit_query
).df()

display(
    lstm_numeric_audit.sort_values("feature").reset_index(drop=True)
)


,feature,total_rows,non_missing,missing_count,minimum,percentile_01,percentile_25,median,percentile_75,percentile_99,maximum,mean,std_dev,zero_count
0,day_of_month,388200,388200,0,1.000000,1.000000,8.00,16.00,23.00,31.000000,31.000000,15.691911,8.792348,0
1,is_event_day,388200,388200,0,0.000000,0.000000,0.00,0.00,0.00,1.000000,1.000000,0.081401,0.273450,356600
2,is_snap_day,388200,388200,0,0.000000,0.000000,0.00,0.00,1.00,1.000000,1.000000,0.329727,0.470114,260200
3,is_weekend,388200,388200,0,0.000000,0.000000,0.00,0.00,1.00,1.000000,1.000000,0.286450,0.452102,277000
4,month,388200,388200,0,1.000000,1.000000,3.00,6.00,9.00,12.000000,12.000000,6.334879,3.439199,0
5,price_change,388200,293049,95151,-10.380000,0.000000,0.00,0.00,0.00,0.000000,10.380000,0.000456,0.106049,289682
6,price_change_flag,388200,293049,95151,0.000000,0.000000,0.00,0.00,0.00,1.000000,1.000000,0.011490,0.106572,289682
7,price_change_flag_missing,388200,388200,0,0.000000,0.000000,0.00,0.00,0.00,1.000000,1.000000,0.245108,0.430151,293049
8,price_change_missing,388200,388200,0,0.000000,0.000000,0.00,0.00,0.00,1.000000,1.000000,0.245108,0.430151,293049
9,price_change_pct,388200,293049,95151,-0.936027,0.000000,0.00,0.00,0.00,0.000000,14.631579,0.001104,0.085342,289682


In [ ]:
# Binary feature audit
LSTM_BINARY_FEATURES = [
    "price_missing",
    "price_change_missing",
    "price_change_pct_missing",
    "price_vs_8w_avg_missing",
    "price_change_flag_missing",
    "price_change_flag",
    "is_weekend",
    "is_event_day",
    "is_snap_day"
]

binary_audit_queries = []

for col in LSTM_BINARY_FEATURES:
    binary_audit_queries.append(f"""
        SELECT
            '{col}' AS feature,
            {col} AS observed_value,
            COUNT(*) AS row_count
        FROM read_parquet(
            '{LSTM_DAILY_PANEL_WITH_INDICATORS_FILE.as_posix()}'
        )
        GROUP BY {col}
    """)

lstm_binary_audit = con.execute(
    " UNION ALL ".join(binary_audit_queries)
).df()

display(
    lstm_binary_audit.sort_values(
        ["feature", "observed_value"],
        na_position="first"
    ).reset_index(drop=True)
)


,feature,observed_value,row_count
0,is_event_day,0,356600
1,is_event_day,1,31600
2,is_snap_day,0,260200
3,is_snap_day,1,128000
4,is_weekend,0,277000
5,is_weekend,1,111200
6,price_change_flag,<NA>,95151
7,price_change_flag,0,289682
8,price_change_flag,1,3367
9,price_change_flag_missing,0,293049


In [64]:

# ==========================================
# Step 3.8.3.11.2 — Check missingness indicator redundancy
# ==========================================

indicator_columns = [
    "price_change_missing",
    "price_change_pct_missing",
    "price_vs_8w_avg_missing",
    "price_change_flag_missing"
]

reference_indicator = indicator_columns[0]

redundancy_checks = []

for col in indicator_columns[1:]:
    result = con.execute(f"""
        SELECT
            COUNT(*) FILTER (
                WHERE {reference_indicator} != {col}
            ) AS mismatched_rows
        FROM read_parquet(
            '{LSTM_DAILY_PANEL_WITH_INDICATORS_FILE.as_posix()}'
        )
    """).fetchone()

    redundancy_checks.append({
        "reference_indicator": reference_indicator,
        "compared_indicator": col,
        "mismatched_rows": result[0]
    })

lstm_indicator_redundancy_audit = pd.DataFrame(
    redundancy_checks
)

display(lstm_indicator_redundancy_audit)


,reference_indicator,compared_indicator,mismatched_rows
0,price_change_missing,price_change_pct_missing,0
1,price_change_missing,price_vs_8w_avg_missing,0
2,price_change_missing,price_change_flag_missing,0


#### Finalize the LSTM numerical feature schema

In [ ]:
# Final LSTM daily panel
LSTM_FINAL_DAILY_PANEL_FILE = (
    LSTM_DIR / "lstm_daily_panel_final.parquet"
)

con.execute(f"""
    COPY (
        SELECT
            * EXCLUDE (
                price_change_missing,
                price_change_pct_missing,
                price_vs_8w_avg_missing,
                price_change_flag_missing
            ),

            price_change_missing
                AS price_history_features_missing

        FROM read_parquet(
            '{LSTM_DAILY_PANEL_WITH_INDICATORS_FILE.as_posix()}'
        )
        ORDER BY
            id,
            TRY_CAST(SUBSTRING(d, 3) AS INTEGER)
    )
    TO '{LSTM_FINAL_DAILY_PANEL_FILE.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Final LSTM panel:", LSTM_FINAL_DAILY_PANEL_FILE)


Final LSTM panel: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lstm\lstm_daily_panel_final.parquet


In [66]:
# Final LSTM feature schema
LSTM_HISTORICAL_NUMERIC_FEATURES = [
    "sales",
    "sell_price",
    "price_missing",
    "price_change",
    "price_change_pct",
    "price_vs_8w_avg",
    "price_change_flag",
    "price_history_features_missing",
    "is_weekend",
    "day_of_month",
    "week_of_year",
    "is_event_day",
    "is_snap_day"
]

LSTM_HISTORICAL_CATEGORICAL_FEATURES = [
    "weekday",
    "event_name_1",
    "event_type_1"
]

LSTM_FUTURE_NUMERIC_FEATURES = [
    "month",
    "year",
    "day_of_month",
    "week_of_year",
    "is_weekend",
    "is_event_day",
    "is_snap_day"
]

LSTM_FUTURE_CATEGORICAL_FEATURES = [
    "weekday",
    "event_name_1",
    "event_type_1"
]

LSTM_STATIC_FEATURES = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

LSTM_ALL_NUMERIC_FEATURES = list(dict.fromkeys(
    LSTM_HISTORICAL_NUMERIC_FEATURES
    + LSTM_FUTURE_NUMERIC_FEATURES
))

LSTM_ALL_CATEGORICAL_FEATURES = list(dict.fromkeys(
    LSTM_HISTORICAL_CATEGORICAL_FEATURES
    + LSTM_FUTURE_CATEGORICAL_FEATURES
    + LSTM_STATIC_FEATURES
))

print("Historical numerical:", len(LSTM_HISTORICAL_NUMERIC_FEATURES))
print("Historical categorical:", len(LSTM_HISTORICAL_CATEGORICAL_FEATURES))
print("Future numerical:", len(LSTM_FUTURE_NUMERIC_FEATURES))
print("Future categorical:", len(LSTM_FUTURE_CATEGORICAL_FEATURES))
print("Static categorical:", len(LSTM_STATIC_FEATURES))
print("Unique numerical:", len(LSTM_ALL_NUMERIC_FEATURES))
print("Unique categorical:", len(LSTM_ALL_CATEGORICAL_FEATURES))


Historical numerical: 13
Historical categorical: 3
Future numerical: 7
Future categorical: 3
Static categorical: 5
Unique numerical: 15
Unique categorical: 8


In [ ]:
# Validate final LSTM panel
lstm_final_audit = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(DISTINCT d) AS unique_days,

        COUNT(*) FILTER (
            WHERE price_history_features_missing IS NULL
        ) AS missing_history_indicator,

        COUNT(*) FILTER (
            WHERE price_history_features_missing = 1
        ) AS missing_history_observations,

        COUNT(*) FILTER (
            WHERE price_missing IS NULL
        ) AS missing_price_indicator,

        COUNT(*) FILTER (
            WHERE sales IS NULL
        ) AS missing_sales

    FROM read_parquet(
        '{LSTM_FINAL_DAILY_PANEL_FILE.as_posix()}'
    )
""").df()

display(lstm_final_audit)


,total_rows,unique_series,unique_days,missing_history_indicator,missing_history_observations,missing_price_indicator,missing_sales
0,388200,200,1941,0,95151,0,0


#### Define LSTM preprocessing strategy

| Feature group | Features | Transformation |
| :--- | :--- | :--- |
| **Historical sales** | `sales` | `log1p(sales)` |
| **Current price** | `sell_price` | Training-median imputation, then RobustScaler |
| **Price changes** | `price_change`, `price_change_pct` | Training-median imputation, then RobustScaler |
| **Price ratio** | `price_vs_8w_avg` | Training-median imputation, then RobustScaler |
| **Price flag** | `price_change_flag` | Missing values → 0; retain as binary |
| **Missingness indicators** | `price_missing`, `price_history_features_missing` | Keep binary 0/1 |
| **Calendar binary** | `is_weekend`, `is_event_day`, `is_snap_day` | Keep binary 0/1 |
| **Calendar continuous** | `year`, `day_of_month` | `StandardScaler` |
| **Calendar cyclical** | `month`, `week_of_year` | Cyclical sin/cos encoding |
| **Categorical** | `weekday`, `event_name_1`, `event_type_1` | Integer category IDs |
| **Static categorical** | `item_id`, `dept_id`, `cat_id`, `store_id`, `state_id` | Integer category IDs |

In [68]:
# Continuous price features requiring imputation and scaling
LSTM_PRICE_CONTINUOUS_FEATURES = [
    "sell_price",
    "price_change",
    "price_change_pct",
    "price_vs_8w_avg"
]

# Binary price features
LSTM_PRICE_BINARY_FEATURES = [
    "price_missing",
    "price_history_features_missing",
    "price_change_flag"
]

# Continuous calendar features
LSTM_CALENDAR_CONTINUOUS_FEATURES = [
    "year",
    "day_of_month"
]

# Cyclical calendar features
LSTM_CYCLICAL_FEATURES = {
    "month": 12,
    "week_of_year": 52.1775
}

# Binary calendar features
LSTM_CALENDAR_BINARY_FEATURES = [
    "is_weekend",
    "is_event_day",
    "is_snap_day"
]

# Historical sales transformation
LSTM_SALES_TRANSFORM = "log1p"

# Target remains in original sales units
LSTM_TARGET_TRANSFORM = "none"

# Preprocessing methods
LSTM_PRICE_IMPUTATION = "training_median"
LSTM_PRICE_SCALER = "RobustScaler"
LSTM_CALENDAR_SCALER = "StandardScaler"

# Categorical handling
LSTM_CATEGORICAL_ENCODING = "integer_category_ids"
LSTM_UNKNOWN_CATEGORY = "UNK"

print("LSTM preprocessing configuration defined.")

LSTM preprocessing configuration defined.


In [69]:
import numpy as np

def add_lstm_cyclical_features(df):
    """
    Convert month and week_of_year into cyclical sin/cos features.
    """
    df = df.copy()

    df["month_sin"] = np.sin(
        2 * np.pi * (df["month"] - 1) / 12
    )
    df["month_cos"] = np.cos(
        2 * np.pi * (df["month"] - 1) / 12
    )

    df["week_of_year_sin"] = np.sin(
        2 * np.pi * (df["week_of_year"] - 1) / 52.1775
    )
    df["week_of_year_cos"] = np.cos(
        2 * np.pi * (df["week_of_year"] - 1) / 52.1775
    )

    return df

In [70]:
# Replace raw month and week_of_year with cyclical representations.

LSTM_HISTORICAL_NUMERIC_FEATURES = [
    "sales",
    "sell_price",
    "price_missing",
    "price_change",
    "price_change_pct",
    "price_vs_8w_avg",
    "price_change_flag",
    "price_history_features_missing",
    "is_weekend",
    "day_of_month",
    "week_of_year_sin",
    "week_of_year_cos",
    "is_event_day",
    "is_snap_day"
]

LSTM_FUTURE_NUMERIC_FEATURES = [
    "month_sin",
    "month_cos",
    "year",
    "day_of_month",
    "week_of_year_sin",
    "week_of_year_cos",
    "is_weekend",
    "is_event_day",
    "is_snap_day"
]

LSTM_ALL_NUMERIC_FEATURES = list(dict.fromkeys(
    LSTM_HISTORICAL_NUMERIC_FEATURES +
    LSTM_FUTURE_NUMERIC_FEATURES
))

LSTM_ALL_CATEGORICAL_FEATURES = list(dict.fromkeys(
    LSTM_HISTORICAL_CATEGORICAL_FEATURES +
    LSTM_FUTURE_CATEGORICAL_FEATURES +
    LSTM_STATIC_FEATURES
))

print("Historical numerical:", len(LSTM_HISTORICAL_NUMERIC_FEATURES))
print("Future numerical:", len(LSTM_FUTURE_NUMERIC_FEATURES))
print("Unique numerical:", len(LSTM_ALL_NUMERIC_FEATURES))
print("Unique categorical:", len(LSTM_ALL_CATEGORICAL_FEATURES))
print("Numerical features:", LSTM_ALL_NUMERIC_FEATURES)

Historical numerical: 14
Future numerical: 9
Unique numerical: 17
Unique categorical: 8
Numerical features: ['sales', 'sell_price', 'price_missing', 'price_change', 'price_change_pct', 'price_vs_8w_avg', 'price_change_flag', 'price_history_features_missing', 'is_weekend', 'day_of_month', 'week_of_year_sin', 'week_of_year_cos', 'is_event_day', 'is_snap_day', 'month_sin', 'month_cos', 'year']


#### Prepare fold-specific LSTM sequences
Expected:
| Component | Shape per sequence |
| :--- | :--- |
| **Historical numerical** | `(56, 14)` |
| **Historical categorical** | `(56, 3)` |
| **Future numerical** | `(28, 9)` |
| **Future categorical** | `(28, 3)` |
| **Static categorical** | `(5,)` |
| **Target sales** | `(28,)` |

In [71]:
LSTM_SEQUENCE_COMPONENTS = {
    "historical_numeric": {
        "features": LSTM_HISTORICAL_NUMERIC_FEATURES,
        "length": LSTM_ENCODER_LENGTH
    },
    "historical_categorical": {
        "features": LSTM_HISTORICAL_CATEGORICAL_FEATURES,
        "length": LSTM_ENCODER_LENGTH
    },
    "future_numeric": {
        "features": LSTM_FUTURE_NUMERIC_FEATURES,
        "length": LSTM_PREDICTION_LENGTH
    },
    "future_categorical": {
        "features": LSTM_FUTURE_CATEGORICAL_FEATURES,
        "length": LSTM_PREDICTION_LENGTH
    },
    "static_categorical": {
        "features": LSTM_STATIC_FEATURES
    },
    "target": {
        "column": LSTM_TARGET_COLUMN,
        "length": LSTM_PREDICTION_LENGTH
    }
}

for component, config in LSTM_SEQUENCE_COMPONENTS.items():
    print(f"{component}: {config}")

historical_numeric: {'features': ['sales', 'sell_price', 'price_missing', 'price_change', 'price_change_pct', 'price_vs_8w_avg', 'price_change_flag', 'price_history_features_missing', 'is_weekend', 'day_of_month', 'week_of_year_sin', 'week_of_year_cos', 'is_event_day', 'is_snap_day'], 'length': 56}
historical_categorical: {'features': ['weekday', 'event_name_1', 'event_type_1'], 'length': 56}
future_numeric: {'features': ['month_sin', 'month_cos', 'year', 'day_of_month', 'week_of_year_sin', 'week_of_year_cos', 'is_weekend', 'is_event_day', 'is_snap_day'], 'length': 28}
future_categorical: {'features': ['weekday', 'event_name_1', 'event_type_1'], 'length': 28}
static_categorical: {'features': ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']}
target: {'column': 'sales', 'length': 28}


In [72]:
lstm_fold_sequence_origins = {}

for fold_name, cutoff in LSTM_FOLD_CUTOFFS.items():

    eligible_origins = all_lstm_origins.loc[
        (all_lstm_origins["origin_day_num"] >= LSTM_ENCODER_LENGTH)
        & (
            all_lstm_origins["origin_day_num"]
            + LSTM_PREDICTION_LENGTH
            <= cutoff
        ),
        "origin_day_num"
    ].astype(int).tolist()

    lstm_fold_sequence_origins[fold_name] = eligible_origins

    print(
        fold_name,
        "| Training origins:", len(eligible_origins),
        "| First:", min(eligible_origins),
        "| Last:", max(eligible_origins),
        "| Training sequences:", len(eligible_origins) * 200
    )

development_fold_1 | Training origins: 246 | First: 56 | Last: 1771 | Training sequences: 49200
development_fold_2 | Training origins: 251 | First: 56 | Last: 1801 | Training sequences: 50200
development_fold_3 | Training origins: 256 | First: 56 | Last: 1829 | Training sequences: 51200
internal_test | Training origins: 261 | First: 56 | Last: 1857 | Training sequences: 52200
official_evaluation | Training origins: 266 | First: 56 | Last: 1885 | Training sequences: 53200


#### Define fold-specific sequence origins
- To identify which forecast origins are eligible for training in each fold.
- A training sequence is valid only if its entire 28-day target period falls within the training cutoff

In [73]:
lstm_fold_sequence_origins = {}

for fold_name, cutoff in LSTM_FOLD_CUTOFFS.items():

    eligible_origins = all_lstm_origins.loc[
        (all_lstm_origins["origin_day_num"] >= LSTM_ENCODER_LENGTH)
        & (
            all_lstm_origins["origin_day_num"]
            + LSTM_PREDICTION_LENGTH
            <= cutoff
        ),
        "origin_day_num"
    ].astype(int).tolist()

    lstm_fold_sequence_origins[fold_name] = eligible_origins

    print(
        fold_name,
        "| Training origins:", len(eligible_origins),
        "| First:", min(eligible_origins),
        "| Last:", max(eligible_origins),
        "| Training sequences:", len(eligible_origins) * 200
    )

development_fold_1 | Training origins: 246 | First: 56 | Last: 1771 | Training sequences: 49200
development_fold_2 | Training origins: 251 | First: 56 | Last: 1801 | Training sequences: 50200
development_fold_3 | Training origins: 256 | First: 56 | Last: 1829 | Training sequences: 51200
internal_test | Training origins: 261 | First: 56 | Last: 1857 | Training sequences: 52200
official_evaluation | Training origins: 266 | First: 56 | Last: 1885 | Training sequences: 53200


#### Define evaluation sequence origins
Unlike training origins, evaluation origins must remain fixed. Each evaluation sequence uses:
- 56 days of historical observations.
- 28 days of future-known calendar information.
- 28 days of actual sales as the evaluation target.

In [74]:
LSTM_EVALUATION_SEQUENCE_CONFIG = {
    "development_fold_1": {
        "origin": 1801,
        "encoder_start": 1746,
        "decoder_start": 1802,
        "decoder_end": 1829
    },
    "development_fold_2": {
        "origin": 1829,
        "encoder_start": 1774,
        "decoder_start": 1830,
        "decoder_end": 1857
    },
    "development_fold_3": {
        "origin": 1857,
        "encoder_start": 1802,
        "decoder_start": 1858,
        "decoder_end": 1885
    },
    "internal_test": {
        "origin": 1885,
        "encoder_start": 1830,
        "decoder_start": 1886,
        "decoder_end": 1913
    },
    "official_evaluation": {
        "origin": 1913,
        "encoder_start": 1858,
        "decoder_start": 1914,
        "decoder_end": 1941
    }
}

for fold_name, config in LSTM_EVALUATION_SEQUENCE_CONFIG.items():
    assert config["origin"] - config["encoder_start"] + 1 == 56
    assert config["decoder_start"] == config["origin"] + 1
    assert config["decoder_end"] - config["decoder_start"] + 1 == 28

    print(
        fold_name,
        "| Encoder:", f'd_{config["encoder_start"]}-d_{config["origin"]}',
        "| Decoder:", f'd_{config["decoder_start"]}-d_{config["decoder_end"]}'
    )

print("\nAll evaluation sequence boundaries validated.")

development_fold_1 | Encoder: d_1746-d_1801 | Decoder: d_1802-d_1829
development_fold_2 | Encoder: d_1774-d_1829 | Decoder: d_1830-d_1857
development_fold_3 | Encoder: d_1802-d_1857 | Decoder: d_1858-d_1885
internal_test | Encoder: d_1830-d_1885 | Decoder: d_1886-d_1913
official_evaluation | Encoder: d_1858-d_1913 | Decoder: d_1914-d_1941

All evaluation sequence boundaries validated.


#### Extract raw LSTM sequence data

In [80]:
# Load final LSTM daily panel
lstm_panel_df = pd.read_parquet(LSTM_FINAL_DAILY_PANEL_FILE)

# Convert day identifier into an integer for sequence slicing
lstm_panel_df["day_num"] = (
    lstm_panel_df["d"]
    .str.replace("d_", "", regex=False)
    .astype(int)
)

# Sort chronologically
lstm_panel_df = lstm_panel_df.sort_values(
    ["id", "day_num"]
).reset_index(drop=True)

# Validate panel structure
lstm_panel_audit = {
    "total_rows": len(lstm_panel_df),
    "unique_series": lstm_panel_df["id"].nunique(),
    "unique_days": lstm_panel_df["day_num"].nunique(),
    "minimum_day": lstm_panel_df["day_num"].min(),
    "maximum_day": lstm_panel_df["day_num"].max(),
    "duplicate_series_days": lstm_panel_df.duplicated(
        ["id", "day_num"]
    ).sum(),
    "missing_sales": lstm_panel_df["sales"].isna().sum()
}

for key, value in lstm_panel_audit.items():
    print(f"{key}: {value}")

assert lstm_panel_audit["total_rows"] == 388200
assert lstm_panel_audit["unique_series"] == 200
assert lstm_panel_audit["unique_days"] == 1941
assert lstm_panel_audit["minimum_day"] == 1
assert lstm_panel_audit["maximum_day"] == 1941
assert lstm_panel_audit["duplicate_series_days"] == 0
assert lstm_panel_audit["missing_sales"] == 0


total_rows: 388200
unique_series: 200
unique_days: 1941
minimum_day: 1
maximum_day: 1941
duplicate_series_days: 0
missing_sales: 0


In [81]:
# Generate cyclical calendar features
lstm_panel_df = add_lstm_cyclical_features(lstm_panel_df)

# Verify the new columns
cyclical_columns = [
    "month_sin",
    "month_cos",
    "week_of_year_sin",
    "week_of_year_cos"
]

print(lstm_panel_df[cyclical_columns].isna().sum())
print(lstm_panel_df[cyclical_columns].describe())

month_sin           0
month_cos           0
week_of_year_sin    0
week_of_year_cos    0
dtype: int64
          month_sin     month_cos  week_of_year_sin  week_of_year_cos
count  3.882000e+05  3.882000e+05     388200.000000     388200.000000
mean   4.356497e-02  1.278748e-02          0.048877          0.010474
std    7.127559e-01  6.999430e-01          0.714227          0.698129
min   -1.000000e+00 -1.000000e+00         -0.999872         -0.999943
25%   -5.000000e-01 -5.000000e-01         -0.677156         -0.673215
50%    1.224647e-16  6.123234e-17          0.120129          0.005344
75%    8.660254e-01  5.000000e-01          0.753667          0.665274
max    1.000000e+00  1.000000e+00          0.999986          1.000000


In [88]:
def extract_lstm_sequence(series_df, origin_day):
    """
    Extract one 56-day encoder and 28-day decoder sequence.
    Returns raw, unprocessed features and target values.
    """

    series_df = series_df.sort_values("day_num").reset_index(drop=True)

    encoder_start = origin_day - LSTM_ENCODER_LENGTH + 1
    decoder_start = origin_day + 1
    decoder_end = origin_day + LSTM_PREDICTION_LENGTH

    encoder_df = series_df[
        series_df["day_num"].between(encoder_start, origin_day)
    ]

    decoder_df = series_df[
        series_df["day_num"].between(decoder_start, decoder_end)
    ]

    # Validate sequence lengths
    assert len(encoder_df) == LSTM_ENCODER_LENGTH
    assert len(decoder_df) == LSTM_PREDICTION_LENGTH

    # Extract historical numerical features
    historical_numeric = encoder_df[
        LSTM_HISTORICAL_NUMERIC_FEATURES
    ].to_numpy(dtype=np.float32)

    # Extract historical categorical features
    historical_categorical = encoder_df[
        LSTM_HISTORICAL_CATEGORICAL_FEATURES
    ].to_numpy(dtype=object)

    # Extract future-known numerical features
    future_numeric = decoder_df[
        LSTM_FUTURE_NUMERIC_FEATURES
    ].to_numpy(dtype=np.float32)

    # Extract future-known categorical features
    future_categorical = decoder_df[
        LSTM_FUTURE_CATEGORICAL_FEATURES
    ].to_numpy(dtype=object)

    # Extract static categorical features
    static_categorical = (
        encoder_df[LSTM_STATIC_FEATURES]
        .iloc[-1]
        .to_numpy(dtype=object)
    )

    # Extract actual future sales as target
    target_sales = decoder_df[
        LSTM_TARGET_COLUMN
    ].to_numpy(dtype=np.float32)

    return {
        "historical_numeric": historical_numeric,
        "historical_categorical": historical_categorical,
        "future_numeric": future_numeric,
        "future_categorical": future_categorical,
        "static_categorical": static_categorical,
        "target_sales": target_sales
    }




#### Generate all LSTM training sequences

In [90]:
LSTM_TRAINING_SEQUENCES = {}

for fold_name, origins in lstm_fold_sequence_origins.items():

    fold_sequences = {
        "historical_numeric": [],
        "historical_categorical": [],
        "future_numeric": [],
        "future_categorical": [],
        "static_categorical": [],
        "target_sales": [],
        "series_ids": [],
        "origins": []
    }

    for origin_day in origins:

        for series_id, series_df in lstm_panel_df.groupby(
            "id", sort=False
        ):

            sequence = extract_lstm_sequence(
                series_df,
                origin_day
            )

            for component in [
                "historical_numeric",
                "historical_categorical",
                "future_numeric",
                "future_categorical",
                "static_categorical",
                "target_sales"
            ]:
                fold_sequences[component].append(
                    sequence[component]
                )

            fold_sequences["series_ids"].append(series_id)
            fold_sequences["origins"].append(origin_day)

    # Convert numerical components to NumPy arrays.
    for component in [
        "historical_numeric",
        "future_numeric",
        "static_categorical",
        "target_sales"
    ]:
        fold_sequences[component] = np.stack(
            fold_sequences[component]
        )

    # Keep categorical components as object arrays for now.
    for component in [
        "historical_categorical",
        "future_categorical"
    ]:
        fold_sequences[component] = np.stack(
            fold_sequences[component]
        )

    fold_sequences["series_ids"] = np.array(
        fold_sequences["series_ids"]
    )

    fold_sequences["origins"] = np.array(
        fold_sequences["origins"]
    )

    LSTM_TRAINING_SEQUENCES[fold_name] = fold_sequences

    print(
        f"{fold_name}: "
        f"{len(fold_sequences['series_ids']):,} sequences generated"
    )

development_fold_1: 49,200 sequences generated
development_fold_2: 50,200 sequences generated
development_fold_3: 51,200 sequences generated
internal_test: 52,200 sequences generated
official_evaluation: 53,200 sequences generated


In [92]:
# Validate training sequence dimensions and alignment
for fold_name, sequences in LSTM_TRAINING_SEQUENCES.items():

    print(f"\n{fold_name}")

    expected_shapes = {
        "historical_numeric": (56, 14),
        "historical_categorical": (56, 3),
        "future_numeric": (28, 9),
        "future_categorical": (28, 3),
        "static_categorical": (5,),
        "target_sales": (28,)
    }

    for component, expected_shape in expected_shapes.items():

        actual_shape = sequences[component].shape[1:]

        print(
            f"{component}: {sequences[component].shape}"
        )

        assert actual_shape == expected_shape

    # Validate sequence metadata
    assert len(sequences["series_ids"]) == len(sequences["origins"])

    assert len(set(sequences["series_ids"])) == 200

    # Every origin should contain exactly 200 series
    origin_counts = pd.Series(
        sequences["origins"]
    ).value_counts()

    assert origin_counts.nunique() == 1
    assert origin_counts.iloc[0] == 200

    # Verify target values are nonnegative
    assert np.all(sequences["target_sales"] >= 0)




development_fold_1
historical_numeric: (49200, 56, 14)
historical_categorical: (49200, 56, 3)
future_numeric: (49200, 28, 9)
future_categorical: (49200, 28, 3)
static_categorical: (49200, 5)
target_sales: (49200, 28)

development_fold_2
historical_numeric: (50200, 56, 14)
historical_categorical: (50200, 56, 3)
future_numeric: (50200, 28, 9)
future_categorical: (50200, 28, 3)
static_categorical: (50200, 5)
target_sales: (50200, 28)

development_fold_3
historical_numeric: (51200, 56, 14)
historical_categorical: (51200, 56, 3)
future_numeric: (51200, 28, 9)
future_categorical: (51200, 28, 3)
static_categorical: (51200, 5)
target_sales: (51200, 28)

internal_test
historical_numeric: (52200, 56, 14)
historical_categorical: (52200, 56, 3)
future_numeric: (52200, 28, 9)
future_categorical: (52200, 28, 3)
static_categorical: (52200, 5)
target_sales: (52200, 28)

official_evaluation
historical_numeric: (53200, 56, 14)
historical_categorical: (53200, 56, 3)
future_numeric: (53200, 28, 9)
future

#### Generate evaluation sequences

In [93]:
LSTM_EVALUATION_SEQUENCES = {}

for fold_name, config in LSTM_EVALUATION_SEQUENCE_CONFIG.items():

    origin_day = config["origin"]

    fold_sequences = {
        "historical_numeric": [],
        "historical_categorical": [],
        "future_numeric": [],
        "future_categorical": [],
        "static_categorical": [],
        "target_sales": [],
        "series_ids": []
    }

    for series_id, series_df in lstm_panel_df.groupby(
        "id", sort=False
    ):

        sequence = extract_lstm_sequence(
            series_df,
            origin_day
        )

        for component in [
            "historical_numeric",
            "historical_categorical",
            "future_numeric",
            "future_categorical",
            "static_categorical",
            "target_sales"
        ]:
            fold_sequences[component].append(
                sequence[component]
            )

        fold_sequences["series_ids"].append(series_id)

    for component in [
        "historical_numeric",
        "historical_categorical",
        "future_numeric",
        "future_categorical",
        "static_categorical",
        "target_sales"
    ]:
        fold_sequences[component] = np.stack(
            fold_sequences[component]
        )

    fold_sequences["series_ids"] = np.array(
        fold_sequences["series_ids"]
    )

    LSTM_EVALUATION_SEQUENCES[fold_name] = fold_sequences

    print(
        f"{fold_name}: "
        f"{len(fold_sequences['series_ids'])} evaluation sequences"
    )

development_fold_1: 200 evaluation sequences
development_fold_2: 200 evaluation sequences
development_fold_3: 200 evaluation sequences
internal_test: 200 evaluation sequences
official_evaluation: 200 evaluation sequences


In [95]:
# Validate evaluation sequence dimensions and alignment
for fold_name, sequences in LSTM_EVALUATION_SEQUENCES.items():

    print(f"\n{fold_name}")

    expected_shapes = {
        "historical_numeric": (56, 14),
        "historical_categorical": (56, 3),
        "future_numeric": (28, 9),
        "future_categorical": (28, 3),
        "static_categorical": (5,),
        "target_sales": (28,)
    }

    for component, expected_shape in expected_shapes.items():

        actual_shape = sequences[component].shape[1:]

        print(f"{component}: {sequences[component].shape}")

        assert actual_shape == expected_shape

    assert len(sequences["series_ids"]) == 200
    assert len(np.unique(sequences["series_ids"])) == 200

    assert np.all(sequences["target_sales"] >= 0)

    # Confirm evaluation origin matches configuration
    assert (
        len(sequences["series_ids"])
        == len(LSTM_SERIES_FILE and LSTM_EVALUATION_SEQUENCE_CONFIG[fold_name])
        or len(sequences["series_ids"]) == 200
    )



development_fold_1
historical_numeric: (200, 56, 14)
historical_categorical: (200, 56, 3)
future_numeric: (200, 28, 9)
future_categorical: (200, 28, 3)
static_categorical: (200, 5)
target_sales: (200, 28)

development_fold_2
historical_numeric: (200, 56, 14)
historical_categorical: (200, 56, 3)
future_numeric: (200, 28, 9)
future_categorical: (200, 28, 3)
static_categorical: (200, 5)
target_sales: (200, 28)

development_fold_3
historical_numeric: (200, 56, 14)
historical_categorical: (200, 56, 3)
future_numeric: (200, 28, 9)
future_categorical: (200, 28, 3)
static_categorical: (200, 5)
target_sales: (200, 28)

internal_test
historical_numeric: (200, 56, 14)
historical_categorical: (200, 56, 3)
future_numeric: (200, 28, 9)
future_categorical: (200, 28, 3)
static_categorical: (200, 5)
target_sales: (200, 28)

official_evaluation
historical_numeric: (200, 56, 14)
historical_categorical: (200, 56, 3)
future_numeric: (200, 28, 9)
future_categorical: (200, 28, 3)
static_categorical: (200, 5

#### Fold-specific preprocessing

In [96]:
# Continuous price features: median imputation + RobustScaler
LSTM_PRICE_CONTINUOUS_FEATURES = [
    "sell_price",
    "price_change",
    "price_change_pct",
    "price_vs_8w_avg"
]

# Continuous calendar features: StandardScaler
LSTM_CALENDAR_CONTINUOUS_FEATURES = [
    "year",
    "day_of_month"
]

# Binary price features: retain 0/1
LSTM_PRICE_BINARY_FEATURES = [
    "price_missing",
    "price_history_features_missing",
    "price_change_flag"
]

# Binary calendar features: retain 0/1
LSTM_CALENDAR_BINARY_FEATURES = [
    "is_weekend",
    "is_event_day",
    "is_snap_day"
]

# Cyclical features: retain sin/cos values in [-1, 1]
LSTM_CYCLICAL_FEATURES_FINAL = [
    "month_sin",
    "month_cos",
    "week_of_year_sin",
    "week_of_year_cos"
]

# Historical sales: log1p transformation
LSTM_SALES_FEATURE = ["sales"]

print("Price continuous:", len(LSTM_PRICE_CONTINUOUS_FEATURES))
print("Calendar continuous:", len(LSTM_CALENDAR_CONTINUOUS_FEATURES))
print("Price binary:", len(LSTM_PRICE_BINARY_FEATURES))
print("Calendar binary:", len(LSTM_CALENDAR_BINARY_FEATURES))
print("Cyclical:", len(LSTM_CYCLICAL_FEATURES_FINAL))
print("Sales:", len(LSTM_SALES_FEATURE))

Price continuous: 4
Calendar continuous: 2
Price binary: 3
Calendar binary: 3
Cyclical: 4
Sales: 1


In [97]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler

LSTM_NUMERICAL_PREPROCESSORS = {}

for fold_name, cutoff in LSTM_FOLD_CUTOFFS.items():

    # Use only observations available by this fold's training cutoff.
    fold_fit_df = lstm_panel_df.loc[
        lstm_panel_df["day_num"] <= cutoff
    ].copy()

    # Fit price imputation and scaling.
    price_imputer = SimpleImputer(strategy="median")
    price_scaler = RobustScaler()

    price_values = price_imputer.fit_transform(
        fold_fit_df[LSTM_PRICE_CONTINUOUS_FEATURES]
    )
    price_scaler.fit(price_values)

    # Fit calendar scaling.
    calendar_scaler = StandardScaler()
    calendar_scaler.fit(
        fold_fit_df[LSTM_CALENDAR_CONTINUOUS_FEATURES]
    )

    LSTM_NUMERICAL_PREPROCESSORS[fold_name] = {
        "price_imputer": price_imputer,
        "price_scaler": price_scaler,
        "calendar_scaler": calendar_scaler,
        "fit_cutoff": cutoff,
        "fit_rows": len(fold_fit_df)
    }

    print(
        f"{fold_name}: "
        f"cutoff=d_{cutoff}, "
        f"fit_rows={len(fold_fit_df):,}"
    )

development_fold_1: cutoff=d_1801, fit_rows=360,200
development_fold_2: cutoff=d_1829, fit_rows=365,800
development_fold_3: cutoff=d_1857, fit_rows=371,400
internal_test: cutoff=d_1885, fit_rows=377,000
official_evaluation: cutoff=d_1913, fit_rows=382,600


In [99]:
# Validate fitted preprocessors
for fold_name, preprocessors in LSTM_NUMERICAL_PREPROCESSORS.items():

    price_imputer = preprocessors["price_imputer"]
    price_scaler = preprocessors["price_scaler"]
    calendar_scaler = preprocessors["calendar_scaler"]

    assert np.isfinite(price_imputer.statistics_).all()
    assert np.isfinite(price_scaler.center_).all()
    assert np.isfinite(price_scaler.scale_).all()
    assert np.isfinite(calendar_scaler.mean_).all()
    assert np.isfinite(calendar_scaler.scale_).all()

    assert preprocessors["fit_cutoff"] == LSTM_FOLD_CUTOFFS[fold_name]

    print(
        f"{fold_name}: "
        f"price medians={np.round(price_imputer.statistics_, 4)}, "
        f"price scaler centres={np.round(price_scaler.center_, 4)}, "
        f"calendar means={np.round(calendar_scaler.mean_, 4)}"
    )


development_fold_1: price medians=[3.37 0.   0.   1.  ], price scaler centres=[3.37 0.   0.   1.  ], calendar means=[2013.0355   15.724 ]
development_fold_2: price medians=[3.37 0.   0.   1.  ], price scaler centres=[3.37 0.   0.   1.  ], calendar means=[2013.0809   15.7512]
development_fold_3: price medians=[3.37 0.   0.   1.  ], price scaler centres=[3.37 0.   0.   1.  ], calendar means=[2013.1249   15.7324]
internal_test: price medians=[3.37 0.   0.   1.  ], price scaler centres=[3.37 0.   0.   1.  ], calendar means=[2013.1676   15.7146]
official_evaluation: price medians=[3.37 0.   0.   1.  ], price scaler centres=[3.37 0.   0.   1.  ], calendar means=[2013.2091   15.7031]


#### Prepare categorical vocabularies

In [100]:
LSTM_CATEGORICAL_COLUMNS = (
    LSTM_HISTORICAL_CATEGORICAL_FEATURES
    + LSTM_STATIC_FEATURES
)

# Include future categorical columns not already listed.
LSTM_CATEGORICAL_COLUMNS = list(dict.fromkeys(
    LSTM_CATEGORICAL_COLUMNS
    + LSTM_FUTURE_CATEGORICAL_FEATURES
))

LSTM_CATEGORY_VOCABS = {}

for fold_name, cutoff in LSTM_FOLD_CUTOFFS.items():

    fold_fit_df = lstm_panel_df.loc[
        lstm_panel_df["day_num"] <= cutoff
    ].copy()

    vocabularies = {}

    for column in LSTM_CATEGORICAL_COLUMNS:

        values = fold_fit_df[column]

        if column in ["event_name_1", "event_type_1"]:
            values = values.fillna("No_Event")
        else:
            values = values.fillna("UNK")

        categories = sorted(values.astype(str).unique())

        # ID 0 is reserved for unseen categories.
        vocabularies[column] = {
            "UNK": 0,
            **{
                category: index
                for index, category in enumerate(categories, start=1)
                if category != "UNK"
            }
        }

    LSTM_CATEGORY_VOCABS[fold_name] = vocabularies

    print(f"\n{fold_name}")
    for column, vocab in vocabularies.items():
        print(f"  {column}: {len(vocab)} categories")


development_fold_1
  weekday: 8 categories
  event_name_1: 32 categories
  event_type_1: 6 categories
  item_id: 193 categories
  dept_id: 8 categories
  cat_id: 4 categories
  store_id: 11 categories
  state_id: 4 categories

development_fold_2
  weekday: 8 categories
  event_name_1: 32 categories
  event_type_1: 6 categories
  item_id: 193 categories
  dept_id: 8 categories
  cat_id: 4 categories
  store_id: 11 categories
  state_id: 4 categories

development_fold_3
  weekday: 8 categories
  event_name_1: 32 categories
  event_type_1: 6 categories
  item_id: 193 categories
  dept_id: 8 categories
  cat_id: 4 categories
  store_id: 11 categories
  state_id: 4 categories

internal_test
  weekday: 8 categories
  event_name_1: 32 categories
  event_type_1: 6 categories
  item_id: 193 categories
  dept_id: 8 categories
  cat_id: 4 categories
  store_id: 11 categories
  state_id: 4 categories

official_evaluation
  weekday: 8 categories
  event_name_1: 32 categories
  event_type_1: 6 cate

In [102]:
# Validate vocabulary coverage
def normalize_lstm_category(value, column):
    if pd.isna(value):
        return "No_Event" if column in [
            "event_name_1", "event_type_1"
        ] else "UNK"
    return str(value)


for fold_name in LSTM_FOLD_CUTOFFS:

    vocabularies = LSTM_CATEGORY_VOCABS[fold_name]

    print(f"\n{fold_name}")

    for column, vocab in vocabularies.items():

        train_values = lstm_panel_df.loc[
            lstm_panel_df["day_num"] <= LSTM_FOLD_CUTOFFS[fold_name],
            column
        ]

        train_values = {
            normalize_lstm_category(value, column)
            for value in train_values
        }

        unseen_train = train_values - set(vocab)

        assert len(unseen_train) == 0, (
            f"{fold_name}, {column}: unseen training categories "
            f"{unseen_train}"
        )

        print(
            f"  {column}: "
            f"vocabulary={len(vocab)}, "
            f"training_unseen={len(unseen_train)}"
        )



development_fold_1
  weekday: vocabulary=8, training_unseen=0
  event_name_1: vocabulary=32, training_unseen=0
  event_type_1: vocabulary=6, training_unseen=0
  item_id: vocabulary=193, training_unseen=0
  dept_id: vocabulary=8, training_unseen=0
  cat_id: vocabulary=4, training_unseen=0
  store_id: vocabulary=11, training_unseen=0
  state_id: vocabulary=4, training_unseen=0

development_fold_2
  weekday: vocabulary=8, training_unseen=0
  event_name_1: vocabulary=32, training_unseen=0
  event_type_1: vocabulary=6, training_unseen=0
  item_id: vocabulary=193, training_unseen=0
  dept_id: vocabulary=8, training_unseen=0
  cat_id: vocabulary=4, training_unseen=0
  store_id: vocabulary=11, training_unseen=0
  state_id: vocabulary=4, training_unseen=0

development_fold_3
  weekday: vocabulary=8, training_unseen=0
  event_name_1: vocabulary=32, training_unseen=0
  event_type_1: vocabulary=6, training_unseen=0
  item_id: vocabulary=193, training_unseen=0
  dept_id: vocabulary=8, training_uns

#### Encode and preprocess LSTM sequences

In [ ]:

# 1. REFIT FOLD-SPECIFIC NUMERICAL PREPROCESSORS

LSTM_NUMERICAL_PREPROCESSORS = {}
LSTM_CALENDAR_SCALERS = {}

for fold_name, cutoff in LSTM_FOLD_CUTOFFS.items():

    # Use only observations available up to this fold's cutoff.
    fit_df = lstm_panel_df.loc[
        lstm_panel_df["day_num"] <= cutoff
    ].copy()

    # Price: median imputation + RobustScaler
    #     price_values = fit_df[
        LSTM_PRICE_CONTINUOUS_FEATURES
    ].to_numpy(dtype=np.float64)

    price_imputer = SimpleImputer(strategy="median")

    price_values_imputed = price_imputer.fit_transform(
        price_values
    )

    price_scaler = RobustScaler()

    price_scaler.fit(price_values_imputed)

    LSTM_NUMERICAL_PREPROCESSORS[fold_name] = {
        "price_imputer": price_imputer,
        "price_scaler": price_scaler,
        "fit_cutoff": cutoff,
        "fit_rows": len(fit_df)
    }

    # Calendar: fit each feature using NumPy arrays

    day_of_month_scaler = StandardScaler()

    day_of_month_scaler.fit(
        fit_df[["day_of_month"]].to_numpy(dtype=np.float64)
    )

    year_scaler = StandardScaler()

    year_scaler.fit(
        fit_df[["year"]].to_numpy(dtype=np.float64)
    )

    LSTM_CALENDAR_SCALERS[fold_name] = {
        "day_of_month": day_of_month_scaler,
        "year": year_scaler
    }

    print(
        f"{fold_name}: preprocessors fitted | "
        f"cutoff=d_{cutoff} | "
        f"fit_rows={len(fit_df):,}"
    )


# 2. CATEGORICAL ENCODING

def encode_lstm_categories(values, column, vocab):

    normalized = [
        normalize_lstm_category(value, column)
        for value in values.reshape(-1)
    ]

    encoded = [
        vocab.get(value, 0)
        for value in normalized
    ]

    return np.asarray(
        encoded,
        dtype=np.int64
    ).reshape(values.shape)


# 3. SEQUENCE PREPROCESSING
def preprocess_lstm_sequences(arrays, fold_name):

    preprocessors = LSTM_NUMERICAL_PREPROCESSORS[fold_name]
    calendar_scalers = LSTM_CALENDAR_SCALERS[fold_name]
    vocabularies = LSTM_CATEGORY_VOCABS[fold_name]

    # Copy raw batched arrays.
    hist_num = arrays["historical_numeric"].copy()
    hist_cat = arrays["historical_categorical"].copy()

    future_num = arrays["future_numeric"].copy()
    future_cat = arrays["future_categorical"].copy()

    static_cat = arrays["static_categorical"].copy()

    # 1. Historical sales: log1p transformation
    sales_idx = LSTM_HISTORICAL_NUMERIC_FEATURES.index("sales")

    hist_num[:, :, sales_idx] = np.log1p(
        hist_num[:, :, sales_idx]
    )

    # 2. Historical price: median imputation + RobustScaler
    price_indices = [
        LSTM_HISTORICAL_NUMERIC_FEATURES.index(col)
        for col in LSTM_PRICE_CONTINUOUS_FEATURES
    ]

    price_values = hist_num[:, :, price_indices]
    original_shape = price_values.shape

    price_values = price_values.reshape(
        -1, len(price_indices)
    )

    price_values = preprocessors["price_imputer"].transform(
        price_values
    )

    price_values = preprocessors["price_scaler"].transform(
        price_values
    )

    hist_num[:, :, price_indices] = price_values.reshape(
        original_shape
    )

    # 3. Historical day_of_month: StandardScaler
    hist_day_idx = LSTM_HISTORICAL_NUMERIC_FEATURES.index(
        "day_of_month"
    )

    hist_day_values = hist_num[:, :, hist_day_idx].reshape(
        -1, 1
    )

    hist_num[:, :, hist_day_idx] = (
        calendar_scalers["day_of_month"]
        .transform(hist_day_values)
        .reshape(hist_num.shape[0], hist_num.shape[1])
    )

    # 4. Future calendar: StandardScaler
    future_year_idx = LSTM_FUTURE_NUMERIC_FEATURES.index("year")

    future_day_idx = LSTM_FUTURE_NUMERIC_FEATURES.index(
        "day_of_month"
    )

    future_year_values = future_num[
        :, :, future_year_idx
    ].reshape(-1, 1)

    future_num[:, :, future_year_idx] = (
        calendar_scalers["year"]
        .transform(future_year_values)
        .reshape(future_num.shape[0], future_num.shape[1])
    )

    future_day_values = future_num[
        :, :, future_day_idx
    ].reshape(-1, 1)

    future_num[:, :, future_day_idx] = (
        calendar_scalers["day_of_month"]
        .transform(future_day_values)
        .reshape(future_num.shape[0], future_num.shape[1])
    )

    # 5. Missing binary price-change flag -> 0
    flag_idx = LSTM_HISTORICAL_NUMERIC_FEATURES.index(
        "price_change_flag"
    )

    hist_num[:, :, flag_idx] = np.nan_to_num(
        hist_num[:, :, flag_idx],
        nan=0.0
    )

    # 6. Historical categorical encoding
    for j, column in enumerate(
        LSTM_HISTORICAL_CATEGORICAL_FEATURES
    ):

        hist_cat[:, :, j] = encode_lstm_categories(
            hist_cat[:, :, j],
            column,
            vocabularies[column]
        )

    # 7. Future categorical encoding
    for j, column in enumerate(
        LSTM_FUTURE_CATEGORICAL_FEATURES
    ):

        future_cat[:, :, j] = encode_lstm_categories(
            future_cat[:, :, j],
            column,
            vocabularies[column]
        )

    # 8. Static categorical encoding
    for j, column in enumerate(LSTM_STATIC_FEATURES):

        static_cat[:, j] = encode_lstm_categories(
            static_cat[:, j],
            column,
            vocabularies[column]
        )

    # 9. Final numerical integrity checks
    assert np.isfinite(hist_num).all(), (
        f"{fold_name}: historical numerical inputs contain "
        "NaN or infinity."
    )

    assert np.isfinite(future_num).all(), (
        f"{fold_name}: future numerical inputs contain "
        "NaN or infinity."
    )

    # Categorical IDs must be non-negative.
    assert (hist_cat >= 0).all()
    assert (future_cat >= 0).all()
    assert (static_cat >= 0).all()

    return {
        "historical_numeric": hist_num.astype(np.float32),
        "historical_categorical": hist_cat.astype(np.int64),
        "future_numeric": future_num.astype(np.float32),
        "future_categorical": future_cat.astype(np.int64),
        "static_categorical": static_cat.astype(np.int64),

        # Keep target sales in original units.
        "target_sales": arrays["target_sales"].copy(),

        "series_ids": arrays["series_ids"].copy(),

        "origins": (
            arrays["origins"].copy()
            if "origins" in arrays else None
        )
    }


# 4. PREPROCESS ALL FIVE FOLDS
LSTM_PROCESSED_TRAINING_SEQUENCES = {}
LSTM_PROCESSED_EVALUATION_SEQUENCES = {}

for fold_name in LSTM_FOLD_CUTOFFS:

    LSTM_PROCESSED_TRAINING_SEQUENCES[fold_name] = (
        preprocess_lstm_sequences(
            LSTM_TRAINING_SEQUENCES[fold_name],
            fold_name
        )
    )

    LSTM_PROCESSED_EVALUATION_SEQUENCES[fold_name] = (
        preprocess_lstm_sequences(
            LSTM_EVALUATION_SEQUENCES[fold_name],
            fold_name
        )
    )

    train_shape = (
        LSTM_PROCESSED_TRAINING_SEQUENCES[fold_name]
        ["historical_numeric"].shape
    )

    valid_shape = (
        LSTM_PROCESSED_EVALUATION_SEQUENCES[fold_name]
        ["historical_numeric"].shape
    )

    print(
        f"{fold_name}: "
        f"training={train_shape}, "
        f"evaluation={valid_shape}"
    )

print("\nAll five folds processed.")

development_fold_1: preprocessors fitted | cutoff=d_1801 | fit_rows=360,200
development_fold_2: preprocessors fitted | cutoff=d_1829 | fit_rows=365,800
development_fold_3: preprocessors fitted | cutoff=d_1857 | fit_rows=371,400
internal_test: preprocessors fitted | cutoff=d_1885 | fit_rows=377,000
official_evaluation: preprocessors fitted | cutoff=d_1913 | fit_rows=382,600
development_fold_1: training=(49200, 56, 14), evaluation=(200, 56, 14)
development_fold_2: training=(50200, 56, 14), evaluation=(200, 56, 14)
development_fold_3: training=(51200, 56, 14), evaluation=(200, 56, 14)
internal_test: training=(52200, 56, 14), evaluation=(200, 56, 14)
official_evaluation: training=(53200, 56, 14), evaluation=(200, 56, 14)

All five folds processed.


### Final checking for LSTM dataset

In [ ]:
# CATEGORICAL UNKNOWN AUDIT

for fold_name in LSTM_FOLD_CUTOFFS:

    eval_data = LSTM_PROCESSED_EVALUATION_SEQUENCES[fold_name]

    print(f"\n{fold_name}")

    # Historical categorical features
    for j, column in enumerate(LSTM_HISTORICAL_CATEGORICAL_FEATURES):

        values = eval_data["historical_categorical"][:, :, j]

        unknown_count = np.sum(values == 0)
        total_count = values.size

        print(
            f"Historical {column}: "
            f"{unknown_count:,}/{total_count:,} "
            f"({unknown_count / total_count:.4%}) mapped to UNK"
        )

    # Future categorical features
    for j, column in enumerate(LSTM_FUTURE_CATEGORICAL_FEATURES):

        values = eval_data["future_categorical"][:, :, j]

        unknown_count = np.sum(values == 0)
        total_count = values.size

        print(
            f"Future {column}: "
            f"{unknown_count:,}/{total_count:,} "
            f"({unknown_count / total_count:.4%}) mapped to UNK"
        )

    # Static categorical features
    for j, column in enumerate(LSTM_STATIC_FEATURES):

        values = eval_data["static_categorical"][:, j]

        unknown_count = np.sum(values == 0)
        total_count = values.size

        print(
            f"Static {column}: "
            f"{unknown_count:,}/{total_count:,} "
            f"({unknown_count / total_count:.4%}) mapped to UNK"
        )


development_fold_1
Historical weekday: 0/11,200 (0.0000%) mapped to UNK
Historical event_name_1: 0/11,200 (0.0000%) mapped to UNK
Historical event_type_1: 0/11,200 (0.0000%) mapped to UNK
Future weekday: 0/5,600 (0.0000%) mapped to UNK
Future event_name_1: 0/5,600 (0.0000%) mapped to UNK
Future event_type_1: 0/5,600 (0.0000%) mapped to UNK
Static item_id: 0/200 (0.0000%) mapped to UNK
Static dept_id: 0/200 (0.0000%) mapped to UNK
Static cat_id: 0/200 (0.0000%) mapped to UNK
Static store_id: 0/200 (0.0000%) mapped to UNK
Static state_id: 0/200 (0.0000%) mapped to UNK

development_fold_2
Historical weekday: 0/11,200 (0.0000%) mapped to UNK
Historical event_name_1: 0/11,200 (0.0000%) mapped to UNK
Historical event_type_1: 0/11,200 (0.0000%) mapped to UNK
Future weekday: 0/5,600 (0.0000%) mapped to UNK
Future event_name_1: 0/5,600 (0.0000%) mapped to UNK
Future event_type_1: 0/5,600 (0.0000%) mapped to UNK
Static item_id: 0/200 (0.0000%) mapped to UNK
Static dept_id: 0/200 (0.0000%) mappe

In [ ]:
# TRANSFORMED NUMERICAL FEATURE AUDIT

fold_name = "development_fold_1"

train_data = LSTM_PROCESSED_TRAINING_SEQUENCES[fold_name]

historical_features = LSTM_HISTORICAL_NUMERIC_FEATURES
future_features = LSTM_FUTURE_NUMERIC_FEATURES
# 1. Historical numerical feature distributions
historical_values = train_data["historical_numeric"].reshape(
    -1,
    len(historical_features)
)

historical_audit = []

for j, feature in enumerate(historical_features):

    values = historical_values[:, j]

    historical_audit.append({
        "feature": feature,
        "min": np.min(values),
        "median": np.median(values),
        "mean": np.mean(values),
        "max": np.max(values),
        "std": np.std(values),
        "unique_values": len(np.unique(values))
    })

historical_audit_df = pd.DataFrame(historical_audit)

print("HISTORICAL NUMERICAL FEATURES")
display(historical_audit_df.round(4))

# 2. Future numerical feature distributions
future_values = train_data["future_numeric"].reshape(
    -1,
    len(future_features)
)

future_audit = []

for j, feature in enumerate(future_features):

    values = future_values[:, j]

    future_audit.append({
        "feature": feature,
        "min": np.min(values),
        "median": np.median(values),
        "mean": np.mean(values),
        "max": np.max(values),
        "std": np.std(values),
        "unique_values": len(np.unique(values))
    })

future_audit_df = pd.DataFrame(future_audit)

print("\nFUTURE NUMERICAL FEATURES")
display(future_audit_df.round(4))

# 3. Binary feature integrity
binary_features = (
    LSTM_PRICE_BINARY_FEATURES
    + LSTM_CALENDAR_BINARY_FEATURES
)

binary_results = []

for feature in binary_features:

    feature_idx = historical_features.index(feature)

    values = train_data["historical_numeric"][:, :, feature_idx]

    binary_results.append({
        "feature": feature,
        "unique_values": sorted(np.unique(values).tolist()),
        "is_binary": set(np.unique(values)).issubset({0.0, 1.0})
    })

binary_audit_df = pd.DataFrame(binary_results)

print("\nBINARY FEATURE AUDIT")
display(binary_audit_df)

# 4. Cyclical feature bounds
cyclical_results = []

for feature in LSTM_CYCLICAL_FEATURES_FINAL:

    # Check historical encoder if the feature exists there.
    if feature in historical_features:

        feature_idx = historical_features.index(feature)

        values = train_data["historical_numeric"][:, :, feature_idx]

        cyclical_results.append({
            "feature": feature,
            "input": "historical",
            "min": np.min(values),
            "max": np.max(values),
            "within_expected_range": (
                np.min(values) >= -1.000001
                and np.max(values) <= 1.000001
            )
        })

    # Check future decoder if the feature exists there.
    if feature in future_features:

        feature_idx = future_features.index(feature)

        values = train_data["future_numeric"][:, :, feature_idx]

        cyclical_results.append({
            "feature": feature,
            "input": "future",
            "min": np.min(values),
            "max": np.max(values),
            "within_expected_range": (
                np.min(values) >= -1.000001
                and np.max(values) <= 1.000001
            )
        })

cyclical_audit_df = pd.DataFrame(cyclical_results)

print("\nCYCLICAL FEATURE AUDIT")
display(cyclical_audit_df.round(4))

assert cyclical_audit_df["within_expected_range"].all()

HISTORICAL NUMERICAL FEATURES


,feature,min,median,mean,max,std,unique_values
0,sales,0.0000,0.0000,0.3900,4.9836,0.6832,79
1,sell_price,-1.7026,-0.0000,0.4109,13.1231,1.6719,234
2,price_missing,0.0000,0.0000,0.2625,1.0000,0.4400,2
3,price_change,-10.3800,0.0000,0.0004,10.3800,0.0942,155
4,price_change_pct,-0.9360,0.0000,0.0009,14.6316,0.0778,348
5,price_vs_8w_avg,-0.9360,0.0000,0.0005,0.6000,0.0230,2065
6,price_change_flag,0.0000,0.0000,0.0089,1.0000,0.0938,2
7,price_history_features_missing,0.0000,0.0000,0.2653,1.0000,0.4415,2
8,is_weekend,0.0000,0.0000,0.2857,1.0000,0.4518,2
9,day_of_month,-1.6695,0.0313,0.0008,1.7321,0.9975,31



FUTURE NUMERICAL FEATURES


,feature,min,median,mean,max,std,unique_values
0,month_sin,-1.0000,0.0000,-0.0270,1.0000,0.7124,8
1,month_cos,-1.0000,-0.0000,-0.0499,1.0000,0.6995,8
2,year,-1.4470,-0.0253,0.0446,2.1074,0.9721,6
3,day_of_month,-1.6695,0.0313,0.0011,1.7321,0.9978,31
4,week_of_year_sin,-0.9999,-0.1095,-0.0314,1.0000,0.7098,53
5,week_of_year_cos,-0.9999,-0.1148,-0.0421,1.0000,0.7024,53
6,is_weekend,0.0000,0.0000,0.2857,1.0000,0.4518,2
7,is_event_day,0.0000,0.0000,0.0787,1.0000,0.2693,2
8,is_snap_day,0.0000,0.0000,0.3284,1.0000,0.4696,2



BINARY FEATURE AUDIT


,feature,unique_values,is_binary
0,price_missing,"[0.0, 1.0]",True
1,price_history_features_missing,"[0.0, 1.0]",True
2,price_change_flag,"[0.0, 1.0]",True
3,is_weekend,"[0.0, 1.0]",True
4,is_event_day,"[0.0, 1.0]",True
5,is_snap_day,"[0.0, 1.0]",True



CYCLICAL FEATURE AUDIT


,feature,input,min,max,within_expected_range
0,month_sin,future,-1.0000,1.0,True
1,month_cos,future,-1.0000,1.0,True
2,week_of_year_sin,historical,-0.9999,1.0,True
3,week_of_year_sin,future,-0.9999,1.0,True
4,week_of_year_cos,historical,-0.9999,1.0,True
5,week_of_year_cos,future,-0.9999,1.0,True



CYCLICAL FEATURE AUDIT


,feature,input,min,max,within_expected_range
0,month_sin,future,-1.0000,1.0,True
1,month_cos,future,-1.0000,1.0,True
2,week_of_year_sin,historical,-0.9999,1.0,True
3,week_of_year_sin,future,-0.9999,1.0,True
4,week_of_year_cos,historical,-0.9999,1.0,True
5,week_of_year_cos,future,-0.9999,1.0,True


In [ ]:
# CROSS-FOLD PREPROCESSING VALIDATION

cross_fold_results = []

for fold_name in LSTM_FOLD_CUTOFFS:

    train = LSTM_PROCESSED_TRAINING_SEQUENCES[fold_name]
    evaluation = LSTM_PROCESSED_EVALUATION_SEQUENCES[fold_name]

    raw_train = LSTM_TRAINING_SEQUENCES[fold_name]
    raw_eval = LSTM_EVALUATION_SEQUENCES[fold_name]

    cross_fold_results.append({
        "fold": fold_name,

        "train_target_match": np.array_equal(
            train["target_sales"],
            raw_train["target_sales"]
        ),

        "eval_target_match": np.array_equal(
            evaluation["target_sales"],
            raw_eval["target_sales"]
        ),

        "train_numeric_finite": (
            np.isfinite(train["historical_numeric"]).all()
            and np.isfinite(train["future_numeric"]).all()
        ),

        "eval_numeric_finite": (
            np.isfinite(evaluation["historical_numeric"]).all()
            and np.isfinite(evaluation["future_numeric"]).all()
        ),

        "train_binary_valid": all(
            set(np.unique(train["historical_numeric"][:, :, 
                LSTM_HISTORICAL_NUMERIC_FEATURES.index(feature)
            ])).issubset({0.0, 1.0})
            for feature in LSTM_PRICE_BINARY_FEATURES
            + LSTM_CALENDAR_BINARY_FEATURES
        ),

        "eval_binary_valid": all(
            set(np.unique(evaluation["historical_numeric"][:, :, 
                LSTM_HISTORICAL_NUMERIC_FEATURES.index(feature)
            ])).issubset({0.0, 1.0})
            for feature in LSTM_PRICE_BINARY_FEATURES
            + LSTM_CALENDAR_BINARY_FEATURES
        )
    })

cross_fold_audit_df = pd.DataFrame(cross_fold_results)

display(cross_fold_audit_df)

assert cross_fold_audit_df.drop(columns="fold").all().all()


,fold,train_target_match,eval_target_match,train_numeric_finite,eval_numeric_finite,train_binary_valid,eval_binary_valid
0,development_fold_1,True,True,True,True,True,True
1,development_fold_2,True,True,True,True,True,True
2,development_fold_3,True,True,True,True,True,True
3,internal_test,True,True,True,True,True,True
4,official_evaluation,True,True,True,True,True,True


In [ ]:
evaluation_origin_map = dict(LSTM_EVALUATION_ORIGINS)

final_audit_results = []

for fold_name in LSTM_FOLD_CUTOFFS:

    raw_train = LSTM_TRAINING_SEQUENCES[fold_name]
    train = LSTM_PROCESSED_TRAINING_SEQUENCES[fold_name]

    raw_eval = LSTM_EVALUATION_SEQUENCES[fold_name]
    evaluation = LSTM_PROCESSED_EVALUATION_SEQUENCES[fold_name]

    # 1. Verify expected array dimensions

    assert train["historical_numeric"].shape[1:] == (56, 14)
    assert train["historical_categorical"].shape[1:] == (56, 3)
    assert train["future_numeric"].shape[1:] == (28, 9)
    assert train["future_categorical"].shape[1:] == (28, 3)
    assert train["static_categorical"].shape[1:] == (5,)
    assert train["target_sales"].shape[1:] == (28,)

    assert evaluation["historical_numeric"].shape == (200, 56, 14)
    assert evaluation["historical_categorical"].shape == (200, 56, 3)
    assert evaluation["future_numeric"].shape == (200, 28, 9)
    assert evaluation["future_categorical"].shape == (200, 28, 3)
    assert evaluation["static_categorical"].shape == (200, 5)
    assert evaluation["target_sales"].shape == (200, 28)

    # 2. Verify data types

    assert train["historical_numeric"].dtype == np.float32
    assert train["future_numeric"].dtype == np.float32

    assert train["historical_categorical"].dtype == np.int64
    assert train["future_categorical"].dtype == np.int64
    assert train["static_categorical"].dtype == np.int64

    assert evaluation["historical_numeric"].dtype == np.float32
    assert evaluation["future_numeric"].dtype == np.float32

    assert evaluation["historical_categorical"].dtype == np.int64
    assert evaluation["future_categorical"].dtype == np.int64
    assert evaluation["static_categorical"].dtype == np.int64

    # 3. Verify series ID alignment

    assert np.array_equal(
        raw_train["series_ids"],
        train["series_ids"]
    )

    assert np.array_equal(
        raw_eval["series_ids"],
        evaluation["series_ids"]
    )

    train_series = set(train["series_ids"])
    eval_series = set(evaluation["series_ids"])

    assert len(train_series) == 200
    assert len(eval_series) == 200
    assert train_series == eval_series

    # 4. Verify training series-origin alignment

    assert "origins" in train
    assert len(train["origins"]) == len(train["series_ids"])

    train_alignment_df = pd.DataFrame({
        "series_id": train["series_ids"],
        "origin": train["origins"]
    })

    # No duplicate series-origin combinations.
    assert not train_alignment_df.duplicated(
        ["series_id", "origin"]
    ).any()

    # Every training origin must contain exactly 200 series.
    origin_counts = (
        train_alignment_df.groupby("origin")["series_id"]
        .nunique()
    )

    assert (origin_counts == 200).all()

    # 5. Verify evaluation origin and series alignment

    expected_eval_origin = evaluation_origin_map[fold_name]

    if evaluation.get("origins") is not None:

        assert np.all(
            evaluation["origins"] == expected_eval_origin
        )

    # Each evaluation series must appear exactly once.
    assert len(np.unique(evaluation["series_ids"])) == 200

    # 6. Verify target sequence dimensions and preservation

    assert np.array_equal(
        raw_train["target_sales"],
        train["target_sales"]
    )

    assert np.array_equal(
        raw_eval["target_sales"],
        evaluation["target_sales"]
    )

    assert train["target_sales"].shape[0] == len(train["series_ids"])
    assert evaluation["target_sales"].shape[0] == 200

    # 7. Record audit results

    final_audit_results.append({
        "fold": fold_name,
        "training_sequences": len(train["series_ids"]),
        "training_origins": len(origin_counts),
        "evaluation_sequences": len(evaluation["series_ids"]),
        "evaluation_origin": expected_eval_origin,
        "unique_training_series": len(train_series),
        "unique_evaluation_series": len(eval_series),
        "numeric_dtype": str(train["historical_numeric"].dtype),
        "categorical_dtype": str(train["historical_categorical"].dtype),
        "alignment_valid": True,
        "target_preserved": True
    })

# 8. Display final audit
lstm_final_structure_audit = pd.DataFrame(final_audit_results)

display(lstm_final_structure_audit)


,fold,training_sequences,training_origins,evaluation_sequences,evaluation_origin,unique_training_series,unique_evaluation_series,numeric_dtype,categorical_dtype,alignment_valid,target_preserved
0,development_fold_1,49200,246,200,1801,200,200,float32,int64,True,True
1,development_fold_2,50200,251,200,1829,200,200,float32,int64,True,True
2,development_fold_3,51200,256,200,1857,200,200,float32,int64,True,True
3,internal_test,52200,261,200,1885,200,200,float32,int64,True,True
4,official_evaluation,53200,266,200,1913,200,200,float32,int64,True,True


#### Save processed dataset

In [ ]:
import json
import joblib
# 1. Create output directories
LSTM_DIR.mkdir(parents=True, exist_ok=True)

LSTM_DATASET_DIR = LSTM_DIR / "processed_datasets"
LSTM_PREPROCESSOR_DIR = LSTM_DIR / "preprocessors"

LSTM_DATASET_DIR.mkdir(parents=True, exist_ok=True)
LSTM_PREPROCESSOR_DIR.mkdir(parents=True, exist_ok=True)

# 2. Save training and evaluation arrays for every fold
for fold_name in LSTM_FOLD_CUTOFFS:

    train = LSTM_PROCESSED_TRAINING_SEQUENCES[fold_name]
    evaluation = LSTM_PROCESSED_EVALUATION_SEQUENCES[fold_name]

    evaluation_origin = evaluation_origin_map[fold_name]

    # Training dataset
    train_file = LSTM_DATASET_DIR / f"{fold_name}_train.npz"

    np.savez_compressed(
        train_file,
        historical_numeric=train["historical_numeric"],
        historical_categorical=train["historical_categorical"],
        future_numeric=train["future_numeric"],
        future_categorical=train["future_categorical"],
        static_categorical=train["static_categorical"],
        target_sales=train["target_sales"],
        series_ids=train["series_ids"],
        origins=train["origins"]
    )

    # Evaluation dataset
    evaluation_file = (
        LSTM_DATASET_DIR / f"{fold_name}_evaluation.npz"
    )

    np.savez_compressed(
        evaluation_file,
        historical_numeric=evaluation["historical_numeric"],
        historical_categorical=evaluation["historical_categorical"],
        future_numeric=evaluation["future_numeric"],
        future_categorical=evaluation["future_categorical"],
        static_categorical=evaluation["static_categorical"],
        target_sales=evaluation["target_sales"],
        series_ids=evaluation["series_ids"],
        evaluation_origin=np.asarray(evaluation_origin)
    )

    print(
        f"{fold_name}: "
        f"training and evaluation datasets saved."
    )

# 3. Save fitted preprocessing objects and vocabularies
LSTM_PREPROCESSORS_FILE = (
    LSTM_PREPROCESSOR_DIR / "lstm_preprocessors.joblib"
)

lstm_preprocessor_bundle = {
    "numerical_preprocessors": LSTM_NUMERICAL_PREPROCESSORS,
    "calendar_scalers": LSTM_CALENDAR_SCALERS,
    "category_vocabularies": LSTM_CATEGORY_VOCABS
}

joblib.dump(
    lstm_preprocessor_bundle,
    LSTM_PREPROCESSORS_FILE,
    compress=3
)

# 4. Save feature schema and preprocessing configuration
lstm_config = {
    "project": "AI-Powered Retail Demand Intelligence Platform",
    "model": "LSTM",
    "encoder_length": LSTM_ENCODER_LENGTH,
    "prediction_length": LSTM_PREDICTION_LENGTH,
    "quantiles": LSTM_QUANTILES,

    "fold_cutoffs": LSTM_FOLD_CUTOFFS,
    "evaluation_origins": {
        fold: origin
        for fold, origin in LSTM_EVALUATION_ORIGINS
    },

    "historical_numeric_features": LSTM_HISTORICAL_NUMERIC_FEATURES,
    "historical_categorical_features": LSTM_HISTORICAL_CATEGORICAL_FEATURES,
    "future_numeric_features": LSTM_FUTURE_NUMERIC_FEATURES,
    "future_categorical_features": LSTM_FUTURE_CATEGORICAL_FEATURES,
    "static_features": LSTM_STATIC_FEATURES,

    "preprocessing": {
        "sales": "log1p",
        "price_continuous": "Median imputation followed by RobustScaler",
        "calendar_continuous": "StandardScaler",
        "binary_features": "Retain 0/1; missing price_change_flag replaced with 0",
        "cyclical_features": "Sinusoidal encoding; unchanged",
        "categorical_features": "Fold-specific integer encoding; UNK=0",
        "target": "Original sales units; no transformation"
    },

    "dataset": {
        "selected_series": 200,
        "training_origins": "Expanding-window weekly origins",
        "forecast_horizon": 28,
        "encoder_length": 56
    }
}

LSTM_CONFIG_FILE = LSTM_DIR / "lstm_dataset_config.json"

with open(LSTM_CONFIG_FILE, "w", encoding="utf-8") as f:
    json.dump(lstm_config, f, indent=4)

# 5. Report saved files
print("\nLSTM dataset preparation artifacts saved.")

print(f"Dataset directory: {LSTM_DATASET_DIR}")
print(f"Preprocessor file: {LSTM_PREPROCESSORS_FILE}")
print(f"Configuration file: {LSTM_CONFIG_FILE}")

development_fold_1: training and evaluation datasets saved.
development_fold_2: training and evaluation datasets saved.
development_fold_3: training and evaluation datasets saved.
internal_test: training and evaluation datasets saved.
official_evaluation: training and evaluation datasets saved.

LSTM dataset preparation artifacts saved.
Dataset directory: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lstm\processed_datasets
Preprocessor file: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lstm\preprocessors\lstm_preprocessors.joblib
Configuration file: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\lstm\lstm_dataset_config.json


### TFT Dataset Preparation

#### Define TFT feature schema
| Feature group | Features | Input period |
| :--- | :--- | :--- |
| **Observed numerical** | `sales`, `sell_price`, `price_change`, `price_change_pct`, `price_vs_8w_avg` | Historical only |
| **Observed indicators** | `price_missing`, `price_history_features_missing`, `price_change_flag` | Historical only |
| **Known numerical** | Month/year/day, cyclical week features, weekend, event and SNAP indicators | Historical and future |
| **Known categorical** | `weekday`, `event_name_1`, `event_type_1` | Historical and future |
| **Static categorical** | `item_id`, `dept_id`, `cat_id`, `store_id`, `state_id` | Static |
| **Target** | `sales` | Historical encoder and future labels |

In [124]:
TFT_ENCODER_LENGTH = 56
TFT_PREDICTION_LENGTH = 28
TFT_QUANTILES = [0.10, 0.50, 0.90]

TFT_STATIC_CATEGORICAL_FEATURES = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

TFT_TIME_VARYING_KNOWN_NUMERIC_FEATURES = [
    "month_sin",
    "month_cos",
    "year",
    "day_of_month",
    "week_of_year_sin",
    "week_of_year_cos",
    "is_weekend",
    "is_event_day",
    "is_snap_day"
]

TFT_TIME_VARYING_KNOWN_CATEGORICAL_FEATURES = [
    "weekday",
    "event_name_1",
    "event_type_1"
]

TFT_TIME_VARYING_UNKNOWN_NUMERIC_FEATURES = [
    "sales",
    "sell_price",
    "price_missing",
    "price_change",
    "price_change_pct",
    "price_vs_8w_avg",
    "price_change_flag",
    "price_history_features_missing"
]

TFT_TARGET_COLUMN = "sales"

In [125]:
TFT_REQUIRED_FEATURES = (
    TFT_STATIC_CATEGORICAL_FEATURES
    + TFT_TIME_VARYING_KNOWN_NUMERIC_FEATURES
    + TFT_TIME_VARYING_KNOWN_CATEGORICAL_FEATURES
    + TFT_TIME_VARYING_UNKNOWN_NUMERIC_FEATURES
)

missing_features = [
    col for col in TFT_REQUIRED_FEATURES
    if col not in lstm_panel_df.columns
]

assert not missing_features, (
    f"Missing TFT features: {missing_features}"
)

assert TFT_TARGET_COLUMN in lstm_panel_df.columns
assert "id" in lstm_panel_df.columns
assert "d" in lstm_panel_df.columns

assert lstm_panel_df["id"].nunique() == 200

assert not lstm_panel_df.duplicated(
    subset=["id", "d"]
).any()

print("TFT feature schema validation passed.")
print(f"Panel rows: {len(lstm_panel_df):,}")
print(f"Unique series: {lstm_panel_df['id'].nunique():,}")
print(f"Encoder length: {TFT_ENCODER_LENGTH}")
print(f"Prediction length: {TFT_PREDICTION_LENGTH}")
print(f"Quantiles: {TFT_QUANTILES}")

TFT feature schema validation passed.
Panel rows: 388,200
Unique series: 200
Encoder length: 56
Prediction length: 28
Quantiles: [0.1, 0.5, 0.9]


#### Audit data

In [126]:
import pandas as pd
import numpy as np

TFT_HISTORICAL_NUMERIC_FEATURES = [
    "sales",
    "sell_price",
    "price_missing",
    "price_change",
    "price_change_pct",
    "price_vs_8w_avg",
    "price_change_flag",
    "price_history_features_missing",
    "is_weekend",
    "day_of_month",
    "week_of_year_sin",
    "week_of_year_cos",
    "is_event_day",
    "is_snap_day"
]

TFT_HISTORICAL_CATEGORICAL_FEATURES = [
    "weekday",
    "event_name_1",
    "event_type_1"
]

TFT_FUTURE_NUMERIC_FEATURES = [
    "month_sin",
    "month_cos",
    "year",
    "day_of_month",
    "week_of_year_sin",
    "week_of_year_cos",
    "is_weekend",
    "is_event_day",
    "is_snap_day"
]

TFT_FUTURE_CATEGORICAL_FEATURES = [
    "weekday",
    "event_name_1",
    "event_type_1"
]

TFT_STATIC_CATEGORICAL_FEATURES = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

TFT_TARGET_COLUMN = "sales"

TFT_FEATURE_GROUPS = {
    "historical_numeric": TFT_HISTORICAL_NUMERIC_FEATURES,
    "historical_categorical": TFT_HISTORICAL_CATEGORICAL_FEATURES,
    "future_numeric": TFT_FUTURE_NUMERIC_FEATURES,
    "future_categorical": TFT_FUTURE_CATEGORICAL_FEATURES,
    "static_categorical": TFT_STATIC_CATEGORICAL_FEATURES
}

# Verify that all required features exist
required_features = set(
    feature
    for features in TFT_FEATURE_GROUPS.values()
    for feature in features
)

required_features.add(TFT_TARGET_COLUMN)

missing_columns = sorted(
    required_features - set(lstm_panel_df.columns)
)

assert not missing_columns, (
    f"Missing TFT source columns: {missing_columns}"
)

# Audit numerical feature distributions and missingness
numeric_features = list(dict.fromkeys(
    TFT_HISTORICAL_NUMERIC_FEATURES
    + TFT_FUTURE_NUMERIC_FEATURES
))

numeric_audit = []

for feature in numeric_features:
    values = pd.to_numeric(
        lstm_panel_df[feature],
        errors="coerce"
    )

    numeric_audit.append({
        "feature": feature,
        "missing_count": int(values.isna().sum()),
        "missing_pct": round(values.isna().mean() * 100, 4),
        "min": values.min(),
        "median": values.median(),
        "mean": values.mean(),
        "max": values.max(),
        "infinite_count": int(np.isinf(values).sum())
    })

tft_numeric_audit = pd.DataFrame(numeric_audit)

display(
    tft_numeric_audit.sort_values(
        "missing_pct",
        ascending=False
    )
)

# Audit categorical feature missingness and cardinality
categorical_features = list(dict.fromkeys(
    TFT_HISTORICAL_CATEGORICAL_FEATURES
    + TFT_FUTURE_CATEGORICAL_FEATURES
    + TFT_STATIC_CATEGORICAL_FEATURES
))

categorical_audit = []

for feature in categorical_features:
    values = lstm_panel_df[feature]

    categorical_audit.append({
        "feature": feature,
        "missing_count": int(values.isna().sum()),
        "unique_values": int(values.nunique(dropna=True))
    })

tft_categorical_audit = pd.DataFrame(categorical_audit)

display(tft_categorical_audit)

# Verify target integrity
target = lstm_panel_df[TFT_TARGET_COLUMN]

print("Target audit")
print({
    "missing": int(target.isna().sum()),
    "negative": int((target < 0).sum()),
    "minimum": int(target.min()),
    "maximum": int(target.max()),
    "mean": float(target.mean())
})

,feature,missing_count,missing_pct,min,median,mean,max,infinite_count
4,price_change_pct,95151,24.5108,-0.936027,0.000000e+00,0.001104,14.631579,0
6,price_change_flag,95151,24.5108,0.000000,0.000000e+00,0.011490,1.000000,0
3,price_change,95151,24.5108,-10.380000,0.000000e+00,0.000456,10.380000,0
5,price_vs_8w_avg,95151,24.5108,0.063973,1.000000e+00,1.000615,1.600000,0
1,sell_price,93751,24.1502,0.050000,3.370000e+00,4.462580,28.960000,0
0,sales,0,0.0000,0.000000,0.000000e+00,1.168140,145.000000,0
2,price_missing,0,0.0000,0.000000,0.000000e+00,0.241502,1.000000,0
7,price_history_features_missing,0,0.0000,0.000000,0.000000e+00,0.245108,1.000000,0
8,is_weekend,0,0.0000,0.000000,0.000000e+00,0.286450,1.000000,0
9,day_of_month,0,0.0000,1.000000,1.600000e+01,15.691911,31.000000,0


,feature,missing_count,unique_values
0,weekday,0,7
1,event_name_1,356600,30
2,event_type_1,356600,4
3,item_id,0,192
4,dept_id,0,7
5,cat_id,0,3
6,store_id,0,10
7,state_id,0,3


Target audit
{'missing': 0, 'negative': 0, 'minimum': 0, 'maximum': 145, 'mean': 1.1681401339515713}


In [132]:
import torch
import numpy as np
import pandas as pd
import sklearn
import lightning
import pytorch_lightning
import pytorch_forecasting
import torchmetrics

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

print("Package versions")
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Lightning:", lightning.__version__)
print("PyTorch Lightning:", pytorch_lightning.__version__)
print("PyTorch Forecasting:", pytorch_forecasting.__version__)
print("TorchMetrics:", torchmetrics.__version__)

print("\nTFT dataset components")
print("TimeSeriesDataSet:", TimeSeriesDataSet)
print("GroupNormalizer:", GroupNormalizer)

print("\nCUDA available:", torch.cuda.is_available())

print("\nAll TFT dependencies imported successfully.")

c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\.venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:30: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


Package versions
PyTorch: 2.15.0+cpu
NumPy: 2.5.4
Pandas: 3.1.0
Scikit-learn: 1.9.1
Lightning: 2.6.6
PyTorch Lightning: 2.6.6
PyTorch Forecasting: 1.8.0
TorchMetrics: 1.9.0

TFT dataset components
TimeSeriesDataSet: <class 'pytorch_forecasting.data.timeseries._timeseries.TimeSeriesDataSet'>
GroupNormalizer: <class 'pytorch_forecasting.data.encoders.GroupNormalizer'>

CUDA available: False

All TFT dependencies imported successfully.


| TFT feature role | Features |
| :--- | :--- |
| **Static categoricals** | `item_id`, `dept_id`, `cat_id`, `store_id`, `state_id` |
| **Known future categoricals** | `weekday`, `event_name_1`, `event_type_1` |
| **Known future reals** | `month_sin`, `month_cos`, `year`, `day_of_month`, `week_of_year_sin`, `week_of_year_cos`, `is_weekend`, `is_event_day`, `is_snap_day` |
| **Unknown future reals** | `sales`, `sell_price`, `price_missing`, `price_change`, `price_change_pct`, `price_vs_8w_avg`, `price_change_flag`, `price_history_features_missing` |

In [ ]:
TFT_ENCODER_LENGTH = 56
TFT_PREDICTION_LENGTH = 28
TFT_QUANTILES = [0.10, 0.50, 0.90]

TFT_GROUP_IDS = ["id"]
TFT_TIME_IDX = "time_idx"

TFT_STATIC_CATEGORICAL_FEATURES = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

TFT_KNOWN_CATEGORICAL_FEATURES = [
    "weekday",
    "event_name_1",
    "event_type_1"
]

TFT_KNOWN_REAL_FEATURES = [
    "month_sin",
    "month_cos",
    "year",
    "day_of_month",
    "week_of_year_sin",
    "week_of_year_cos",
    "is_weekend",
    "is_event_day",
    "is_snap_day"
]

TFT_UNKNOWN_REAL_FEATURES = [
    "sales",
    "sell_price",
    "price_missing",
    "price_change",
    "price_change_pct",
    "price_vs_8w_avg",
    "price_change_flag",
    "price_history_features_missing"
]

TFT_TARGET_COLUMN = "sales"

tft_panel_df = lstm_panel_df.copy()

# Create integer time index
tft_panel_df["time_idx"] = (
    pd.to_numeric(
        tft_panel_df["d"].str.replace("d_", "", regex=False),
        errors="raise"
    )
    .astype("int64")
)

# Sort observations chronologically
tft_panel_df = tft_panel_df.sort_values(
    ["id", "time_idx"]
).reset_index(drop=True)

# Verify required columns
TFT_REQUIRED_COLUMNS = list(dict.fromkeys(
    TFT_GROUP_IDS
    + [TFT_TIME_IDX]
    + TFT_STATIC_CATEGORICAL_FEATURES
    + TFT_KNOWN_CATEGORICAL_FEATURES
    + TFT_KNOWN_REAL_FEATURES
    + TFT_UNKNOWN_REAL_FEATURES
))

missing_columns = sorted(
    set(TFT_REQUIRED_COLUMNS) - set(tft_panel_df.columns)
)

assert not missing_columns, (
    f"Missing TFT columns: {missing_columns}"
)

# Verify series and time index integrity
tft_integrity = {
    "total_rows": len(tft_panel_df),
    "unique_series": tft_panel_df["id"].nunique(),
    "unique_time_indices": tft_panel_df["time_idx"].nunique(),
    "minimum_time_idx": int(tft_panel_df["time_idx"].min()),
    "maximum_time_idx": int(tft_panel_df["time_idx"].max()),
    "duplicate_series_time_groups": int(
        tft_panel_df.duplicated(
            ["id", "time_idx"]
        ).sum()
    ),
    "missing_time_idx": int(
        tft_panel_df["time_idx"].isna().sum()
    )
}

print("TFT panel integrity")
print(tft_integrity)

# Verify static categorical features remain constant per series
static_audit = (
    tft_panel_df
    .groupby("id")[TFT_STATIC_CATEGORICAL_FEATURES]
    .nunique()
    .max()
)

display(
    static_audit.rename("maximum_unique_values_per_series")
)

# Verify each series has a complete daily timeline
series_length_audit = (
    tft_panel_df
    .groupby("id")["time_idx"]
    .agg(["count", "min", "max", "nunique"])
)

assert (series_length_audit["count"] == 1941).all()
assert (series_length_audit["nunique"] == 1941).all()
assert (series_length_audit["min"] == 1).all()
assert (series_length_audit["max"] == 1941).all()

assert tft_integrity["total_rows"] == 388200
assert tft_integrity["unique_series"] == 200
assert tft_integrity["duplicate_series_time_groups"] == 0
assert tft_integrity["missing_time_idx"] == 0
assert (static_audit == 1).all()

print("TFT feature-role and panel integrity checks passed.")

TFT panel integrity
{'total_rows': 388200, 'unique_series': 200, 'unique_time_indices': 1941, 'minimum_time_idx': 1, 'maximum_time_idx': 1941, 'duplicate_series_time_groups': 0, 'missing_time_idx': 0}


item_id     1
dept_id     1
cat_id      1
store_id    1
state_id    1
Name: maximum_unique_values_per_series, dtype: int64

TFT feature-role and panel integrity checks passed.


#### Configure fold-specific TFT training datasets
| Fold / Split | Training cutoff | Validation origin | Validation horizon |
| :--- | :--- | :--- | :--- |
| **Fold 1** | 1,801 | 1,801 | 1,802–1,829 |
| **Fold 2** | 1,829 | 1,829 | 1,830–1,857 |
| **Fold 3** | 1,857 | 1,857 | 1,858–1,885 |
| **Internal test** | 1,885 | 1,885 | 1,886–1,913 |
| **Official evaluation** | 1,913 | 1,913 | 1,914–1,941 |

In [135]:
TFT_FOLD_CONFIG = {
    "development_fold_1": {
        "cutoff": 1801,
        "validation_start": 1802,
        "validation_end": 1829
    },
    "development_fold_2": {
        "cutoff": 1829,
        "validation_start": 1830,
        "validation_end": 1857
    },
    "development_fold_3": {
        "cutoff": 1857,
        "validation_start": 1858,
        "validation_end": 1885
    },
    "internal_test": {
        "cutoff": 1885,
        "validation_start": 1886,
        "validation_end": 1913
    },
    "official_evaluation": {
        "cutoff": 1913,
        "validation_start": 1914,
        "validation_end": 1941
    }
}

# Verify fold boundaries against the existing split configuration
for fold_name, config in TFT_FOLD_CONFIG.items():
    assert (
        config["validation_start"]
        == config["cutoff"] + 1
    )

    assert (
        config["validation_end"]
        - config["validation_start"] + 1
        == TFT_PREDICTION_LENGTH
    )

print("TFT fold configuration")
display(
    pd.DataFrame.from_dict(
        TFT_FOLD_CONFIG,
        orient="index"
    ).rename_axis("fold")
)



TFT fold configuration


,cutoff,validation_start,validation_end
fold,,,
development_fold_1,1801,1802,1829
development_fold_2,1829,1830,1857
development_fold_3,1857,1858,1885
internal_test,1885,1886,1913
official_evaluation,1913,1914,1941


#### Create the first TFT training dataset (Fold 1)
Configuration:
- GroupNormalizer normalises demand separately for each product-store series.
- add_relative_time_idx gives TFT a relative position within each sample.
- add_target_scales supplies target normalisation information to the model.
- add_encoder_length supplies encoder-length information.
- allow_missing_timesteps=False ensures gaps in the daily timeline are not silently accepted.

To handle missing data:
1. Impute continuous price features using training-period medians.
2. Retain price_missing and price_history_features_missing as missingness indicators.
3. Replace missing price_change_flag with zero, retaining its separate missingness indicator.
4. Fit imputation values separately for each fold, using only data through that fold's training cutoff.
5. Apply the same fitted values to the corresponding validation dataset.

In [137]:
from sklearn.impute import SimpleImputer
import joblib

TFT_CONTINUOUS_PRICE_FEATURES = [
    "sell_price",
    "price_change",
    "price_change_pct",
    "price_vs_8w_avg"
]

fold_name = "development_fold_1"
cutoff = TFT_FOLD_CONFIG[fold_name]["cutoff"]

tft_fold1_train_df = tft_panel_df.loc[
    tft_panel_df["time_idx"] <= cutoff
].copy()

tft_fold1_price_imputer = SimpleImputer(
    strategy="median"
)

tft_fold1_train_df[
    TFT_CONTINUOUS_PRICE_FEATURES
] = tft_fold1_price_imputer.fit_transform(
    tft_fold1_train_df[
        TFT_CONTINUOUS_PRICE_FEATURES
    ]
)

# Replace missing binary price-change flag
tft_fold1_train_df["price_change_flag"] = (
    tft_fold1_train_df["price_change_flag"]
    .fillna(0)
)

# Verify no missing values remain in TFT numerical inputs
TFT_NUMERICAL_INPUTS = list(dict.fromkeys(
    TFT_KNOWN_REAL_FEATURES
    + TFT_UNKNOWN_REAL_FEATURES
))

tft_missing_audit = (
    tft_fold1_train_df[TFT_NUMERICAL_INPUTS]
    .isna()
    .sum()
)

display(
    tft_missing_audit[
        tft_missing_audit > 0
    ]
)

assert (
    tft_fold1_train_df[TFT_NUMERICAL_INPUTS]
    .isna()
    .sum()
    .sum()
    == 0
)

assert np.isfinite(
    tft_fold1_train_df[TFT_NUMERICAL_INPUTS]
    .to_numpy(dtype=np.float64)
).all()

print("Fold 1 TFT numerical imputation passed.")

print("Fitted price medians:")
display(
    pd.DataFrame({
        "feature": TFT_CONTINUOUS_PRICE_FEATURES,
        "training_median": (
            tft_fold1_price_imputer.statistics_
        )
    })
)

Series([], dtype: int64)

Fold 1 TFT numerical imputation passed.
Fitted price medians:


,feature,training_median
0,sell_price,3.37
1,price_change,0.00
2,price_change_pct,0.00
3,price_vs_8w_avg,1.00


In [ ]:
import sys
import subprocess

print("Python version:", sys.version)
print("Python executable:", sys.executable)

packages = [
    "torch",
    "numpy",
    "pandas",
    "scikit-learn",
    "lightning",
    "pytorch-lightning",
    "scikit-base"
]

for package in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "show", package],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        lines = [
            line for line in result.stdout.splitlines()
            if line.startswith(("Name:", "Version:", "Location:"))
        ]
        print("\n".join(lines))
    else:
        print(f"{package}: Not installed")

Python version: 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)]
Python executable: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\.venv\Scripts\python.exe
Name: torch
Version: 2.15.0
Location: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\.venv\Lib\site-packages
Name: numpy
Version: 2.5.4
Location: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\.venv\Lib\site-packages
Name: pandas
Version: 3.1.0
Location: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\.venv\Lib\site-packages
Name: scikit-learn
Version: 1.9.1
Location: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\.venv\Lib\site-packages
lightning: Not installed
pytorch-lightning: Not installed
scikit-base: Not installed


In [138]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

tft_fold1_training = TimeSeriesDataSet(
    tft_fold1_train_df,
    time_idx=TFT_TIME_IDX,
    target=TFT_TARGET_COLUMN,
    group_ids=TFT_GROUP_IDS,
    min_encoder_length=TFT_ENCODER_LENGTH,
    max_encoder_length=TFT_ENCODER_LENGTH,
    min_prediction_length=TFT_PREDICTION_LENGTH,
    max_prediction_length=TFT_PREDICTION_LENGTH,
    static_categoricals=TFT_STATIC_CATEGORICAL_FEATURES,
    time_varying_known_categoricals=TFT_KNOWN_CATEGORICAL_FEATURES,
    time_varying_known_reals=[TFT_TIME_IDX] + TFT_KNOWN_REAL_FEATURES,
    time_varying_unknown_categoricals=[],
    time_varying_unknown_reals=TFT_UNKNOWN_REAL_FEATURES,
    target_normalizer=GroupNormalizer(groups=TFT_GROUP_IDS),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=False
)

print("Fold 1 TFT training dataset created.")
print("Training rows:", len(tft_fold1_train_df))
print("Training series:", tft_fold1_train_df["id"].nunique())
print("Dataset samples:", len(tft_fold1_training))
print("Encoder length:", TFT_ENCODER_LENGTH)
print("Prediction length:", TFT_PREDICTION_LENGTH)

display(tft_fold1_training.index.head())

Fold 1 TFT training dataset created.
Training rows: 360200
Training series: 200
Dataset samples: 343600
Encoder length: 56
Prediction length: 28


,index_start,index_end,sequence_length,time,sequence_id
0,0,83,84,1,0
1,1,84,84,2,0
2,2,85,84,3,0
3,3,86,84,4,0
4,4,87,84,5,0


#### Align TFT training windows with LSTM
- Reason: To make the LSTM and TFT comparison fair. If TFT trains on 343,600 samples while LSTM trains on 49,200, they receive different amounts of training data and forecast-window exposure.

Strategy:
- Both use the same 200 selected product-store series.
- Both use the same 56-day historical encoder and 28-day forecast horizon.
- Both train on the same 246 weekly forecast origins.
- We filtered TFT's daily sliding windows to retain only those 246 weekly prediction start dates.

In [179]:
TFT_WEEKLY_PREDICTION_STARTS = np.arange(
    57,
    1775,
    7
)

print(
    "Expected weekly forecast origins:",
    len(TFT_WEEKLY_PREDICTION_STARTS)
)

print(
    "Expected TFT training samples:",
    len(TFT_WEEKLY_PREDICTION_STARTS) * 200
)

tft_fold1_training_weekly = tft_fold1_training.filter(
    lambda index: index["time_idx_first_prediction"].isin(
        TFT_WEEKLY_PREDICTION_STARTS
    )
)

print("\nFiltered TFT training dataset")
print("Training samples:", len(tft_fold1_training_weekly))
print(
    "Training series:",
    tft_fold1_training_weekly.index["sequence_id"].nunique()
)

print("\nUnique prediction timestamps:")
print(
    tft_fold1_training_weekly.decoded_index[
        "time_idx_first_prediction"
    ].nunique()
)

assert len(TFT_WEEKLY_PREDICTION_STARTS) == 246
assert len(tft_fold1_training_weekly) == 49_200
assert (
    tft_fold1_training_weekly.index["sequence_id"].nunique()
    == 200
)

assert set(
    tft_fold1_training_weekly.decoded_index[
        "time_idx_first_prediction"
    ].unique()
) == set(TFT_WEEKLY_PREDICTION_STARTS)

print("\nTFT Fold 1 training windows successfully aligned with LSTM.")

Expected weekly forecast origins: 246
Expected TFT training samples: 49200

Filtered TFT training dataset
Training samples: 49200
Training series: 200

Unique prediction timestamps:
246

TFT Fold 1 training windows successfully aligned with LSTM.


#### Prepare Fold 1 validation data

In [180]:
fold_name = "development_fold_1"

fold_config = TFT_FOLD_CONFIG[fold_name]

validation_start = fold_config["validation_start"]
validation_end = fold_config["validation_end"]

tft_fold1_validation_df = tft_panel_df.loc[
    tft_panel_df["time_idx"] <= validation_end
].copy()

tft_fold1_validation_df[
    TFT_CONTINUOUS_PRICE_FEATURES
] = tft_fold1_price_imputer.transform(
    tft_fold1_validation_df[
        TFT_CONTINUOUS_PRICE_FEATURES
    ]
)

tft_fold1_validation_df["price_change_flag"] = (
    tft_fold1_validation_df["price_change_flag"]
    .fillna(0)
)

tft_fold1_validation = TimeSeriesDataSet.from_dataset(
    tft_fold1_training,
    tft_fold1_validation_df,
    min_prediction_idx=validation_start,
    predict=True,
    stop_randomization=True
)

print("Fold 1 TFT validation dataset created.")
print("Validation samples:", len(tft_fold1_validation))
print(
    "Validation series:",
    tft_fold1_validation.index["sequence_id"].nunique()
)

display(tft_fold1_validation.decoded_index.head())

Fold 1 TFT validation dataset created.
Validation samples: 200
Validation series: sequence_id    200
sequence_id    200
dtype: int64


,id,time_idx_first,time_idx_last,time_idx_first_prediction
0,FOODS_1_020_WI_2_evaluation,1746,1829,1802
1,FOODS_1_023_WI_1_evaluation,1746,1829,1802
2,FOODS_1_078_TX_2_evaluation,1746,1829,1802
3,FOODS_1_079_CA_1_evaluation,1746,1829,1802
4,FOODS_1_106_CA_4_evaluation,1746,1829,1802


In [181]:
# Validate 
train_index = tft_fold1_training_weekly.decoded_index
validation_index = tft_fold1_validation.decoded_index

print("Training dataset audit")
print("Samples:", len(train_index))
print("Unique series:", train_index["id"].nunique())
print(
    "Prediction timestamps:",
    train_index["time_idx_first_prediction"].nunique()
)
print(
    "Minimum prediction timestamp:",
    train_index["time_idx_first_prediction"].min()
)
print(
    "Maximum prediction timestamp:",
    train_index["time_idx_first_prediction"].max()
)

print("\nValidation dataset audit")
print("Samples:", len(validation_index))
print("Unique series:", validation_index["id"].nunique())
print(
    "Prediction timestamps:",
    validation_index["time_idx_first_prediction"].unique()
)
print(
    "Minimum encoder end:",
    validation_index["time_idx_first"].add(55).min()
)
print(
    "Maximum encoder end:",
    validation_index["time_idx_first"].add(55).max()
)

assert len(train_index) == 49_200
assert train_index["id"].nunique() == 200
assert train_index["time_idx_first_prediction"].nunique() == 246

assert len(validation_index) == 200
assert validation_index["id"].nunique() == 200
assert validation_index["time_idx_first_prediction"].nunique() == 1
assert validation_index["time_idx_first_prediction"].iloc[0] == 1802
assert (validation_index["time_idx_last"] == 1829).all()
assert (validation_index["time_idx_first"] + 55 == 1801).all()

assert set(train_index["id"]) == set(validation_index["id"])


Training dataset audit
Samples: 49200
Unique series: 200
Prediction timestamps: 246
Minimum prediction timestamp: 57
Maximum prediction timestamp: 1772

Validation dataset audit
Samples: 200
Unique series: 200
Prediction timestamps: [1802]
Minimum encoder end: 1801
Maximum encoder end: 1801


#### Prepare all TFT fold datasets

In [187]:
TFT_DATASET_DIR = PROCESSED_DIR / "tft"
TFT_DATASET_DIR.mkdir(parents=True, exist_ok=True)

TFT_CONTINUOUS_PRICE_FEATURES = [
    "sell_price",
    "price_change",
    "price_change_pct",
    "price_vs_8w_avg"
]

TFT_NUMERICAL_INPUTS = list(dict.fromkeys(
    TFT_KNOWN_REAL_FEATURES
    + TFT_UNKNOWN_REAL_FEATURES
))

TFT_LSTM_DATASET_DIR = (
    PROCESSED_DIR
    / "lstm"
    / "processed_datasets"
)

tft_fold_datasets = {}
tft_fold_preprocessors = {}

for fold_name, fold_config in TFT_FOLD_CONFIG.items():

    cutoff = fold_config["cutoff"]
    validation_start = fold_config["validation_start"]
    validation_end = fold_config["validation_end"]

    print(f"\nPreparing {fold_name}")

    train_df = tft_panel_df.loc[
        tft_panel_df["time_idx"] <= cutoff
    ].copy()

    price_imputer = SimpleImputer(strategy="median")

    train_df[TFT_CONTINUOUS_PRICE_FEATURES] = (
        price_imputer.fit_transform(
            train_df[TFT_CONTINUOUS_PRICE_FEATURES]
        )
    )

    train_df["price_change_flag"] = (
        train_df["price_change_flag"].fillna(0)
    )

    training_dataset = TimeSeriesDataSet(
        train_df,
        time_idx=TFT_TIME_IDX,
        target=TFT_TARGET_COLUMN,
        group_ids=TFT_GROUP_IDS,
        min_encoder_length=TFT_ENCODER_LENGTH,
        max_encoder_length=TFT_ENCODER_LENGTH,
        min_prediction_length=TFT_PREDICTION_LENGTH,
        max_prediction_length=TFT_PREDICTION_LENGTH,
        static_categoricals=TFT_STATIC_CATEGORICAL_FEATURES,
        time_varying_known_categoricals=TFT_KNOWN_CATEGORICAL_FEATURES,
        time_varying_known_reals=[TFT_TIME_IDX] + TFT_KNOWN_REAL_FEATURES,
        time_varying_unknown_categoricals=[],
        time_varying_unknown_reals=TFT_UNKNOWN_REAL_FEATURES,
        target_normalizer=GroupNormalizer(groups=TFT_GROUP_IDS),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        allow_missing_timesteps=False
    )

    weekly_prediction_starts = np.arange(
        TFT_ENCODER_LENGTH + 1,
        cutoff - TFT_PREDICTION_LENGTH + 2,
        7
    )

    previous_evaluation_starts = [
        config["validation_start"]
        for config in TFT_FOLD_CONFIG.values()
        if (
            config["validation_start"] < validation_start
            and config["validation_start"] + TFT_PREDICTION_LENGTH - 1
            <= cutoff
        )
    ]

    training_prediction_starts = np.unique(
        np.concatenate([
            weekly_prediction_starts,
            previous_evaluation_starts
        ])
    )

    training_dataset_weekly = training_dataset.filter(
        lambda index: index["time_idx_first_prediction"].isin(
            training_prediction_starts
        )
    )

    validation_df = tft_panel_df.loc[
        tft_panel_df["time_idx"] <= validation_end
    ].copy()

    validation_df[TFT_CONTINUOUS_PRICE_FEATURES] = (
        price_imputer.transform(
            validation_df[TFT_CONTINUOUS_PRICE_FEATURES]
        )
    )

    validation_df["price_change_flag"] = (
        validation_df["price_change_flag"].fillna(0)
    )

    validation_dataset = TimeSeriesDataSet.from_dataset(
        training_dataset,
        validation_df,
        min_prediction_idx=validation_start,
        predict=True,
        stop_randomization=True
    )

    tft_fold_datasets[fold_name] = {
        "training": training_dataset_weekly,
        "validation": validation_dataset
    }

    tft_fold_preprocessors[fold_name] = {
        "price_imputer": price_imputer
    }

    train_index = training_dataset_weekly.decoded_index
    validation_index = validation_dataset.decoded_index

    lstm_path = (
        TFT_LSTM_DATASET_DIR
        / f"{fold_name}_train.npz"
    )

    with np.load(lstm_path, allow_pickle=False) as lstm_data:
        lstm_origins = np.sort(
            np.unique(lstm_data["origins"])
        )
        lstm_series_ids = np.sort(
            np.unique(lstm_data["series_ids"])
        )

    tft_origins = np.sort(
    train_index["time_idx_first_prediction"].unique() - 1
    )

    tft_series_ids = np.sort(
        train_index["id"].unique()
    )

    assert train_index["id"].nunique() == 200
    assert validation_index["id"].nunique() == 200
    assert len(validation_dataset) == 200

    assert len(train_index) == len(training_prediction_starts) * 200

    assert (
        train_index["time_idx_first_prediction"].nunique()
        == len(training_prediction_starts)
    )

    assert np.array_equal(
        np.sort(
            train_index["time_idx_first_prediction"].unique()
        ),
        training_prediction_starts
    )

    assert (
        validation_index["time_idx_first_prediction"].nunique()
        == 1
    )

    assert (
        validation_index["time_idx_first_prediction"].iloc[0]
        == validation_start
    )

    assert (validation_index["time_idx_last"] == validation_end).all()

    assert (
        validation_index["time_idx_first"] + TFT_ENCODER_LENGTH - 1
        == cutoff
    ).all()

    assert set(train_index["id"]) == set(validation_index["id"])

    assert (
        train_df[TFT_NUMERICAL_INPUTS].isna().sum().sum()
        == 0
    )

    assert (
        validation_df[TFT_NUMERICAL_INPUTS].isna().sum().sum()
        == 0
    )

    print("Training samples:", len(training_dataset_weekly))
    print("Training origins:", len(tft_origins))
    print("LSTM origins:", len(lstm_origins))
    print("Added previous evaluation origins:", previous_evaluation_starts)

    assert np.array_equal(
        tft_origins,
        lstm_origins
    ), (
        f"{fold_name}: TFT and LSTM training origins do not match. "
        f"Missing from TFT: {np.setdiff1d(lstm_origins, tft_origins)}. "
        f"Missing from LSTM: {np.setdiff1d(tft_origins, lstm_origins)}."
    )

    assert np.array_equal(
        tft_series_ids,
        lstm_series_ids
    ), f"{fold_name}: TFT and LSTM series IDs do not match."

    assert np.all(
        tft_origins + TFT_PREDICTION_LENGTH <= cutoff
    )

    print("Validation samples:", len(validation_dataset))
    print("Validation forecast origin:", validation_start - 1)
    print("LSTM and TFT training origins match.")
    print(f"{fold_name} passed all checks.")

print("\nAll five TFT folds prepared successfully.")


Preparing development_fold_1
Training samples: 49200
Training origins: 246
LSTM origins: 246
Added previous evaluation origins: []
Validation samples: 200
Validation forecast origin: 1801
LSTM and TFT training origins match.
development_fold_1 passed all checks.

Preparing development_fold_2
Training samples: 50200
Training origins: 251
LSTM origins: 251
Added previous evaluation origins: [1802]
Validation samples: 200
Validation forecast origin: 1829
LSTM and TFT training origins match.
development_fold_2 passed all checks.

Preparing development_fold_3
Training samples: 51200
Training origins: 256
LSTM origins: 256
Added previous evaluation origins: [1802, 1830]
Validation samples: 200
Validation forecast origin: 1857
LSTM and TFT training origins match.
development_fold_3 passed all checks.

Preparing internal_test
Training samples: 52200
Training origins: 261
LSTM origins: 261
Added previous evaluation origins: [1802, 1830, 1858]
Validation samples: 200
Validation forecast origin:

#### Save TFT datasets and preprocessors

In [188]:
import joblib
import json

TFT_DATASET_DIR = PROCESSED_DIR / "tft"
TFT_DATASET_DIR.mkdir(parents=True, exist_ok=True)

TFT_PREPROCESSOR_DIR = TFT_DATASET_DIR / "preprocessors"
TFT_PREPROCESSOR_DIR.mkdir(parents=True, exist_ok=True)

TFT_DATASETS = {}

for fold_name, datasets in tft_fold_datasets.items():

    training_path = (
        TFT_DATASET_DIR / f"{fold_name}_training.pt"
    )

    validation_path = (
        TFT_DATASET_DIR / f"{fold_name}_validation.pt"
    )

    datasets["training"].save(str(training_path))
    datasets["validation"].save(str(validation_path))

    preprocessor_path = (
        TFT_PREPROCESSOR_DIR / f"{fold_name}_preprocessors.joblib"
    )

    joblib.dump(
        tft_fold_preprocessors[fold_name],
        preprocessor_path
    )

    print(f"{fold_name} saved.")
    print("Training:", training_path)
    print("Validation:", validation_path)
    print("Preprocessor:", preprocessor_path)

    TFT_DATASETS[fold_name] = {
        "training_path": str(training_path),
        "validation_path": str(validation_path),
        "preprocessor_path": str(preprocessor_path)
    }

tft_dataset_config = {
    "encoder_length": TFT_ENCODER_LENGTH,
    "prediction_length": TFT_PREDICTION_LENGTH,
    "quantiles": TFT_QUANTILES,
    "training_sampling": "weekly",
    "series_count": 200,
    "training_folds": {
        fold_name: {
            "training_samples": len(datasets["training"]),
            "validation_samples": len(datasets["validation"])
        }
        for fold_name, datasets in tft_fold_datasets.items()
    }
}

config_path = TFT_DATASET_DIR / "tft_dataset_config.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(tft_dataset_config, file, indent=4)

print("\nTFT datasets and preprocessing artefacts saved.")
print("Configuration:", config_path)

development_fold_1 saved.
Training: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\tft\development_fold_1_training.pt
Validation: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\tft\development_fold_1_validation.pt
Preprocessor: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\tft\preprocessors\development_fold_1_preprocessors.joblib
development_fold_2 saved.
Training: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\tft\development_fold_2_training.pt
Validation: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\tft\development_fold_2_validation.pt
Preprocessor: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\data\processed\tft\preprocessors\development_fold_2_preprocessors.joblib
development_fold_3 saved.
Training: c:\Users\cjc72\Desktop\SalesForecasting\AI-Retail-Demand-Intelligence\

In [196]:
from pathlib import Path
import json
import joblib
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().parent

TFT_DATASET_DIR = PROJECT_ROOT / "data" / "processed" / "tft"
TFT_PREPROCESSOR_DIR = TFT_DATASET_DIR / "preprocessors"
LSTM_DATASET_DIR = (
    PROJECT_ROOT / "data" / "processed" / "lstm" / "processed_datasets"
)

expected_training_origins = {
    "development_fold_1": 246,
    "development_fold_2": 251,
    "development_fold_3": 256,
    "internal_test": 261,
    "official_evaluation": 266
}

expected_training_samples = {
    fold_name: origin_count * 200
    for fold_name, origin_count in expected_training_origins.items()
}

for fold_name in TFT_FOLD_CONFIG:

    print(f"\nVerifying {fold_name}")

    training_path = TFT_DATASET_DIR / f"{fold_name}_training.pt"
    validation_path = TFT_DATASET_DIR / f"{fold_name}_validation.pt"
    preprocessor_path = (
        TFT_PREPROCESSOR_DIR / f"{fold_name}_preprocessors.joblib"
    )
    lstm_path = LSTM_DATASET_DIR / f"{fold_name}_train.npz"

    assert training_path.exists(), f"Missing training file: {training_path}"
    assert validation_path.exists(), f"Missing validation file: {validation_path}"
    assert preprocessor_path.exists(), f"Missing preprocessor: {preprocessor_path}"
    assert lstm_path.exists(), f"Missing LSTM file: {lstm_path}"

    saved_training_dataset = torch.load(
        training_path,
        map_location="cpu",
        weights_only=False
    )

    saved_validation_dataset = torch.load(
        validation_path,
        map_location="cpu",
        weights_only=False
    )

    saved_preprocessor = joblib.load(preprocessor_path)

    lstm_data = np.load(
        lstm_path,
        allow_pickle=True
    )

    training_index = saved_training_dataset.decoded_index
    validation_index = saved_validation_dataset.decoded_index

    tft_training_origins = np.sort(
        training_index["time_idx_first_prediction"].unique() - 1
    )

    lstm_training_origins = np.sort(
        np.unique(lstm_data["origins"])
    )

    tft_series_ids = np.sort(
        training_index["id"].unique().astype(str)
    )

    lstm_series_ids = np.sort(
        np.unique(lstm_data["series_ids"]).astype(str)
    )

    validation_origins = (
        validation_index["time_idx_first_prediction"].unique() - 1
    )

    assert len(saved_training_dataset) == expected_training_samples[fold_name], (
        f"{fold_name}: incorrect training sample count"
    )

    assert len(tft_training_origins) == expected_training_origins[fold_name], (
        f"{fold_name}: incorrect training origin count"
    )

    assert np.array_equal(
        tft_training_origins,
        lstm_training_origins
    ), f"{fold_name}: saved TFT and LSTM origins do not match"

    assert np.array_equal(
        tft_series_ids,
        lstm_series_ids
    ), f"{fold_name}: saved TFT and LSTM series IDs do not match"

    assert len(saved_validation_dataset) == 200, (
        f"{fold_name}: incorrect validation sample count"
    )

    assert len(validation_origins) == 1, (
        f"{fold_name}: validation dataset contains unexpected forecast origins"
    )

    assert validation_origins[0] == TFT_FOLD_CONFIG[fold_name]["validation_start"] - 1, (
        f"{fold_name}: incorrect validation forecast origin"
    )

    assert saved_preprocessor is not None

    print("Training samples:", len(saved_training_dataset))
    print("Training origins:", len(tft_training_origins))
    print("Validation samples:", len(saved_validation_dataset))
    print("Validation forecast origin:", validation_origins[0])
    print("LSTM and TFT training origins match.")
    print("Preprocessor loaded successfully.")
    print(f"{fold_name} passed.")

config_path = TFT_DATASET_DIR / "tft_dataset_config.json"

assert config_path.exists(), "Missing TFT configuration file"

with open(config_path, "r", encoding="utf-8") as file:
    saved_tft_config = json.load(file)

print("\nTFT configuration loaded successfully.")
print("All five saved TFT folds passed verification.")


Verifying development_fold_1
Training samples: 49200
Training origins: 246
Validation samples: 200
Validation forecast origin: 1801
LSTM and TFT training origins match.
Preprocessor loaded successfully.
development_fold_1 passed.

Verifying development_fold_2
Training samples: 50200
Training origins: 251
Validation samples: 200
Validation forecast origin: 1829
LSTM and TFT training origins match.
Preprocessor loaded successfully.
development_fold_2 passed.

Verifying development_fold_3
Training samples: 51200
Training origins: 256
Validation samples: 200
Validation forecast origin: 1857
LSTM and TFT training origins match.
Preprocessor loaded successfully.
development_fold_3 passed.

Verifying internal_test
Training samples: 52200
Training origins: 261
Validation samples: 200
Validation forecast origin: 1885
LSTM and TFT training origins match.
Preprocessor loaded successfully.
internal_test passed.

Verifying official_evaluation
Training samples: 53200
Training origins: 266
Validatio

In [227]:
lgb_terms = [
    "LGB",
    "LIGHTGBM",
    "QUANTILE",
    "MODEL_FEATURES",
    "TARGET_COLUMN"
]

for term in lgb_terms:
    print(f"\n{term} variables:")

    for name in sorted(globals()):
        if term in name.upper():
            value = globals()[name]

            if isinstance(value, (str, int, float, bool, list, tuple, dict)):
                print(f"{name} = {value}")
            else:
                print(f"{name}: {type(value).__name__}")


LGB variables:
LGB_CATEGORICAL_FEATURES = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'target_weekday', 'target_event_name_1', 'target_event_type_1']
LGB_NUMERICAL_FEATURES = ['sales_at_origin', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28', 'sales_rolling_mean_7', 'sales_rolling_std_7', 'sales_rolling_mean_28', 'sales_rolling_std_28', 'sales_rolling_max_28', 'origin_sell_price', 'origin_price_missing', 'origin_price_change', 'origin_price_change_pct', 'origin_price_vs_8w_avg', 'origin_price_change_flag', 'target_month', 'target_year', 'target_is_weekend', 'target_day_of_month', 'target_week_of_year', 'target_is_event_day', 'target_is_snap_day', 'horizon']
LGB_PARAMS = {0.1: {'objective': 'quantile', 'alpha': 0.1, 'metric': 'quantile', 'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': -1, 'min_child_samples': 100, 'feature_fraction': 0.9, 'verbosity': -1, 'seed': 42, 'n_jobs': -1}, 0.5: {'objective': 'quantile', 'alpha': 0.5, 'metric': 'quantile', 'lear